# 02 · Tumbabiro — PI-LSTM V2.4 partition-aware: entrenamiento y validación

**Objetivo.** Entrenar y evaluar la configuración canónica V2.3, exportando el modelo, los scalers y los metadatos necesarios para una inferencia 2050 congruente.

Configuración canónica reconstruida del notebook V2.3 final:

- train: 1998–2019
- validation: 2020–2022
- test: 2023–2024
- $L=72$, $H=24$, `stride=2`
- 34 features históricas
- 17 features futuras
- encoder/decoder LSTM 96/96 + atención
- $k_{DNI,\max}=1.20$
- $k_{DHI,\max}=1.40$
- loss partition-aware.

**Artefactos contractuales**

- `PI_LSTM_V2_3_partition_push_stride2.keras`
- `scaler_hist_v23.joblib`
- `scaler_fut_v23.joblib`
- `model_metadata_v23.json`
- `test_metrics_v23.csv`

El Notebook 3 **debe cargar estos artefactos** y no reconstruir scalers ni listas de features por su cuenta.

In [1]:
# ============================================================
# 1. PI-LSTM V2.4 — CONFIGURACIÓN GENERAL
# ============================================================

# %pip install -q tensorflow scikit-learn joblib

import os
import gc
import json
import random
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    Concatenate,
    LayerNormalization,
    AdditiveAttention,
    TimeDistributed,
    Rescaling,
    Layer
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)


# ============================================================
# REPRODUCIBILIDAD
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# CONFIGURACIÓN CANÓNICA PI-LSTM V2.4
# ============================================================

@dataclass(frozen=True)
class CFG:

    # --------------------------------------------------------
    # Archivos
    # --------------------------------------------------------

    data_path: str = "consol_tumbabiro_2026_total.csv"

    model_path: str = (
        "PI_LSTM_V2_4_logDHI_partition_stride2.keras"
    )

    scaler_hist_path: str = (
        "scaler_hist_v24.joblib"
    )

    scaler_fut_path: str = (
        "scaler_fut_v24.joblib"
    )

    metadata_path: str = (
        "model_metadata_v24.json"
    )

    metrics_path: str = (
        "test_metrics_v24.csv"
    )


    # --------------------------------------------------------
    # División temporal
    # --------------------------------------------------------

    train_end: str = "2019-12-31 23:30:00"
    val_end: str = "2022-12-31 23:30:00"


    # --------------------------------------------------------
    # Ventanas temporales
    # --------------------------------------------------------

    L: int = 72
    H: int = 24
    stride: int = 2

    batch_size: int = 32

    epochs: int = 25


    # --------------------------------------------------------
    # Optimización
    # --------------------------------------------------------

    lr: float = 3e-4

    patience_es: int = 8

    patience_rlr: int = 3

    reduce_lr_factor: float = 0.5

    min_lr: float = 1e-5


    # --------------------------------------------------------
    # Arquitectura
    # --------------------------------------------------------

    latent_enc: int = 96
    latent_dec: int = 96

    dropout: float = 0.10


    # --------------------------------------------------------
    # Umbrales físicos
    # --------------------------------------------------------

    thr_cs: float = 20.0

    thr_day_cosz: float = 0.05


    # --------------------------------------------------------
    # Límites de las salidas físicas
    # --------------------------------------------------------

    # Rama directa:
    # la auditoría histórica mostró max(kDNI) = 1.0
    kdni_max: float = 1.20

    # Rama difusa:
    # auditoría histórica sin clipping:
    # max(kDHI) = 7.918
    #
    # Se adopta 8.0 como límite físico-numérico superior.
    kdhi_max: float = 8.00


    # --------------------------------------------------------
    # Pesos de la función de pérdida
    # --------------------------------------------------------

    lambda_kdni: float = 0.40

    # En V2.4 esta ponderación se aplica a qDHI = log1p(kDHI)
    lambda_qdhi: float = 1.45

    lambda_ghi: float = 0.65

    lambda_ratio: float = 1.90

    lambda_energy: float = 0.27


# ============================================================
# INSTANCIAR CONFIGURACIÓN
# ============================================================

cfg = CFG()


# ============================================================
# PARÁMETROS DERIVADOS DE LA TRANSFORMACIÓN DIFUSA
# ============================================================

# qDHI = log(1 + kDHI)
#
# Si kDHI_max = 8:
#
# qDHI_max = log(9) ≈ 2.1972246

QDHI_MAX = float(np.log1p(cfg.kdhi_max))


# ============================================================
# RESUMEN DE CONFIGURACIÓN
# ============================================================

print("TensorFlow:", tf.__version__)

print("\n=== PI-LSTM V2.4 ===")

print("SEED          :", SEED)

print("L             :", cfg.L)
print("H             :", cfg.H)
print("stride        :", cfg.stride)

print("batch_size    :", cfg.batch_size)
print("epochs_max    :", cfg.epochs)

print("learning rate :", cfg.lr)

print("kDNI max      :", cfg.kdni_max)
print("kDHI max      :", cfg.kdhi_max)

print("qDHI max      :", QDHI_MAX)

print("\nArtefactos:")
print("  modelo      :", cfg.model_path)
print("  scaler hist :", cfg.scaler_hist_path)
print("  scaler fut  :", cfg.scaler_fut_path)
print("  metadata    :", cfg.metadata_path)
print("  métricas    :", cfg.metrics_path)

cfg

# ============================================================
# VERIFICACIÓN DEL ENTORNO DE CÓMPUTO
# ============================================================

print("\n=== DISPOSITIVOS DISPONIBLES ===")

print(
    "GPU detectadas:",
    tf.config.list_physical_devices("GPU")
)

TensorFlow: 2.20.0

=== PI-LSTM V2.4 ===
SEED          : 42
L             : 72
H             : 24
stride        : 2
batch_size    : 32
epochs_max    : 25
learning rate : 0.0003
kDNI max      : 1.2
kDHI max      : 8.0
qDHI max      : 2.1972245773362196

Artefactos:
  modelo      : PI_LSTM_V2_4_logDHI_partition_stride2.keras
  scaler hist : scaler_hist_v24.joblib
  scaler fut  : scaler_fut_v24.joblib
  metadata    : model_metadata_v24.json
  métricas    : test_metrics_v24.csv

=== DISPOSITIVOS DISPONIBLES ===
GPU detectadas: []


In [2]:
# ============================================================
# 2. PI-LSTM V2.4 — COLUMNAS CANÓNICAS Y CARGA DEL HISTÓRICO
# ============================================================

# ------------------------------------------------------------
# Columnas temporales
# ------------------------------------------------------------

COL_DATE = "Date"
COL_HOUR = "Hour"
COL_MIN  = "Minute"


# ------------------------------------------------------------
# Variables meteorológicas
# ------------------------------------------------------------

COL_TEMP = "Temperature"
COL_DEW  = "Dew Point"
COL_RH   = "Relative Humidity"
COL_PRES = "Pressure"
COL_AOD  = "Aerosol Optical Depth"
COL_PW   = "Precipitable Water"
COL_WSPD = "Wind Speed"
COL_WDIR = "Wind Direction"


# ------------------------------------------------------------
# Variables radiativas
# ------------------------------------------------------------

COL_GHI = "GHI"
COL_DNI = "DNI"
COL_DHI = "DHI"

COL_THZ = "Solar Zenith Angle"

COL_CSGHI = "Clearsky GHI"
COL_CSDNI = "Clearsky DNI"
COL_CSDHI = "Clearsky DHI"


# ============================================================
# CONTRATO MÍNIMO DEL ARCHIVO HISTÓRICO
# ============================================================

REQUIRED_RAW_COLS = [
    COL_DATE,
    COL_HOUR,
    COL_MIN,

    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,
    COL_WDIR,

    COL_GHI,
    COL_DNI,
    COL_DHI,
    COL_THZ,

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# ============================================================
# VERIFICACIÓN DE RUTA
# ============================================================

data_path = Path(cfg.data_path)

print("Directorio de trabajo:")
print(" ", Path.cwd())

print("\nArchivo histórico esperado:")
print(" ", data_path.resolve())


if not data_path.exists():

    raise FileNotFoundError(
        "\nNo se encontró el archivo histórico requerido:\n"
        f"  {data_path.resolve()}\n\n"
        "Verifique que 'consol_tumbabiro_2026_total.csv' "
        "se encuentre en el directorio de trabajo actual "
        "o modifique cfg.data_path con la ruta correcta."
    )


# ============================================================
# CARGA DEL CSV
# ============================================================

df_raw = pd.read_csv(
    data_path,
    low_memory=False
)


# ============================================================
# AUDITORÍA BÁSICA DE ESTRUCTURA
# ============================================================

missing_raw_cols = [
    c
    for c in REQUIRED_RAW_COLS
    if c not in df_raw.columns
]

if missing_raw_cols:

    raise ValueError(
        "Faltan columnas obligatorias en el histórico:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_raw_cols
        )
    )


# ============================================================
# RESUMEN
# ============================================================

file_size_mb = (
    data_path.stat().st_size
    / (1024 ** 2)
)

print("\n✓ Archivo histórico encontrado")
print(
    f"✓ Tamaño archivo : {file_size_mb:.2f} MB"
)

print(
    f"✓ Dimensión raw  : {df_raw.shape}"
)

print(
    f"✓ Número columnas: {df_raw.shape[1]}"
)

print(
    f"✓ Columnas contractuales presentes: "
    f"{len(REQUIRED_RAW_COLS)} / {len(REQUIRED_RAW_COLS)}"
)

print("\nPrimeras columnas:")
print(
    df_raw.columns.tolist()
)

Directorio de trabajo:
  /content

Archivo histórico esperado:
  /content/consol_tumbabiro_2026_total.csv

✓ Archivo histórico encontrado
✓ Tamaño archivo : 60.04 MB
✓ Dimensión raw  : (473376, 28)
✓ Número columnas: 28
✓ Columnas contractuales presentes: 18 / 18

Primeras columnas:
['Date', 'Hour', 'Minute', 'Temperature', 'Alpha', 'Aerosol Optical Depth', 'Asymmetry', 'Clearsky DHI', 'Clearsky DNI', 'Clearsky GHI', 'Cloud Fill Flag', 'Cloud Type', 'Dew Point', 'DHI', 'DNI', 'Fill Flag', 'GHI', 'Ozone', 'Relative Humidity', 'Solar Zenith Angle', 'SSA', 'Surface Albedo', 'Pressure', 'Precipitable Water', 'Wind Direction', 'Wind Speed', 'Global Horizontal UV Irradiance (280-400nm)', 'Global Horizontal UV Irradiance (295-385nm)']


In [3]:
# ============================================================
# 3. PI-LSTM V2.4 — TIMESTAMP E INGENIERÍA FÍSICA BÁSICA
# ============================================================


# ============================================================
# 1. CONSTRUCCIÓN ROBUSTA DEL TIMESTAMP
# ============================================================

# ============================================================
# 1. CONSTRUCCIÓN ROBUSTA DEL TIMESTAMP
# ============================================================

def build_timestamp(df):
    """
    Construye timestamp a partir de Date, Hour y Minute.

    El histórico consolidado utiliza Date en formato:
        DD/MM/YYYY

    Ejemplo:
        13/01/1998 = 13 de enero de 1998

    Se utiliza un formato explícito para evitar interpretaciones
    ambiguas de pandas.
    """

    out = df.copy()

    # --------------------------------------------------------
    # Fecha: formato explícito DD/MM/YYYY
    # --------------------------------------------------------

    date_raw = (
        out[COL_DATE]
        .astype(str)
        .str.strip()
    )

    date_parsed = pd.to_datetime(
        date_raw,
        format="%d/%m/%Y",
        errors="coerce"
    )

    # --------------------------------------------------------
    # Hora y minuto
    # --------------------------------------------------------

    hour = pd.to_numeric(
        out[COL_HOUR],
        errors="coerce"
    )

    minute = pd.to_numeric(
        out[COL_MIN],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Validaciones de hora y minuto
    # --------------------------------------------------------

    invalid_hour = (
        hour.isna()
        | (hour < 0)
        | (hour > 23)
    )

    invalid_minute = (
        minute.isna()
        | ~minute.isin([0, 30])
    )

    if invalid_hour.any():

        print(
            "\nPrimeros valores inválidos de Hour:"
        )

        print(
            out.loc[
                invalid_hour,
                [COL_DATE, COL_HOUR, COL_MIN]
            ]
            .head(20)
            .to_string(index=False)
        )

        raise ValueError(
            "La columna Hour contiene valores inválidos."
        )

    if invalid_minute.any():

        print(
            "\nPrimeros valores inválidos de Minute:"
        )

        print(
            out.loc[
                invalid_minute,
                [COL_DATE, COL_HOUR, COL_MIN]
            ]
            .head(20)
            .to_string(index=False)
        )

        raise ValueError(
            "La columna Minute contiene valores distintos "
            "de 0 o 30."
        )

    # --------------------------------------------------------
    # Timestamp final
    # --------------------------------------------------------

    out["timestamp"] = (
        date_parsed
        + pd.to_timedelta(
            hour,
            unit="h"
        )
        + pd.to_timedelta(
            minute,
            unit="m"
        )
    )

    return out


# ============================================================
# 2. FEATURES TEMPORALES Y GEOMETRÍA
# ============================================================

def add_time_features(df):

    out = df.copy()

    ts = out["timestamp"]

    # --------------------------------------------------------
    # Ciclo diario
    # --------------------------------------------------------

    hour_float = (
        ts.dt.hour
        + ts.dt.minute / 60.0
    )

    out["sin_hour"] = np.sin(
        2.0 * np.pi * hour_float / 24.0
    ).astype(np.float32)

    out["cos_hour"] = np.cos(
        2.0 * np.pi * hour_float / 24.0
    ).astype(np.float32)


    # --------------------------------------------------------
    # Ciclo anual
    # Se conserva la formulación V2.3: 365.25 días
    # --------------------------------------------------------

    doy = ts.dt.dayofyear.astype(float)

    out["sin_doy"] = np.sin(
        2.0 * np.pi * doy / 365.25
    ).astype(np.float32)

    out["cos_doy"] = np.cos(
        2.0 * np.pi * doy / 365.25
    ).astype(np.float32)


    # --------------------------------------------------------
    # Coseno cenital positivo
    # --------------------------------------------------------

    theta = np.deg2rad(
        pd.to_numeric(
            out[COL_THZ],
            errors="coerce"
        ).to_numpy(dtype=np.float32)
    )

    out["cosz_plus"] = np.maximum(
        np.cos(theta),
        0.0
    ).astype(np.float32)

    return out


# ============================================================
# 3. ÍNDICES CLEAR-SKY — V2.4
# ============================================================

def add_kspace_features(df, thr_cs):

    out = df.copy()

    # --------------------------------------------------------
    # Convertir a arrays numéricos
    # --------------------------------------------------------

    csghi = pd.to_numeric(
        out[COL_CSGHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    csdni = pd.to_numeric(
        out[COL_CSDNI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    csdhi = pd.to_numeric(
        out[COL_CSDHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    ghi = pd.to_numeric(
        out[COL_GHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    dni = pd.to_numeric(
        out[COL_DNI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    dhi = pd.to_numeric(
        out[COL_DHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)


    # --------------------------------------------------------
    # kGHI
    # Se conserva como feature auxiliar.
    # --------------------------------------------------------

    out["kGHI"] = np.where(
        csghi > thr_cs,
        ghi / csghi,
        0.0
    ).astype(np.float32)

    out["kGHI"] = np.clip(
        out["kGHI"],
        0.0,
        2.0
    ).astype(np.float32)


    # --------------------------------------------------------
    # kDNI
    # Auditoría histórica:
    # max observado = 1.0
    # límite de salida = 1.20
    # --------------------------------------------------------

    out["kDNI"] = np.where(
        csdni > thr_cs,
        dni / csdni,
        0.0
    ).astype(np.float32)

    # Protección numérica histórica.
    # No afecta los datos auditados.
    out["kDNI"] = np.clip(
        out["kDNI"],
        0.0,
        cfg.kdni_max
    ).astype(np.float32)


    # --------------------------------------------------------
    # kDHI RAW
    #
    # V2.4 elimina el antiguo clipping a 2.0.
    # La auditoría histórica mostró:
    #
    # max = 7.918
    # P95 = 4.778
    # P99 = 5.918
    #
    # kdhi_max = 8.0 actúa únicamente como protección
    # físico-numérica y no censura el histórico auditado.
    # --------------------------------------------------------

    kdhi_raw = np.where(
        csdhi > thr_cs,
        dhi / csdhi,
        0.0
    ).astype(np.float32)

    out["kDHI"] = np.clip(
        kdhi_raw,
        0.0,
        cfg.kdhi_max
    ).astype(np.float32)


    # --------------------------------------------------------
    # Transformación logarítmica difusa
    #
    # qDHI = log(1 + kDHI)
    # --------------------------------------------------------

    out["qDHI"] = np.log1p(
        out["kDHI"].to_numpy(dtype=np.float32)
    ).astype(np.float32)


    return out


# ============================================================
# 4. VARIABLES DE PARTICIÓN, DINÁMICA Y MEMORIA
# ============================================================

def add_partition_features(df, thr_cs):

    out = df.copy()

    eps = 1e-6


    # --------------------------------------------------------
    # Rampas radiativas
    # --------------------------------------------------------

    out["dGHI"] = (
        out[COL_GHI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )

    out["dDNI"] = (
        out[COL_DNI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )

    out["dDHI"] = (
        out[COL_DHI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Fracción difusa
    # --------------------------------------------------------

    out["diffuse_ratio"] = np.where(
        out[COL_GHI] > thr_cs,
        out[COL_DHI]
        / (out[COL_GHI] + eps),
        0.0
    ).astype(np.float32)

    out["diffuse_ratio"] = (
        out["diffuse_ratio"]
        .clip(0.0, 1.2)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Fracción directa horizontal
    # --------------------------------------------------------

    out["beam_ratio"] = np.where(
        out[COL_GHI] > thr_cs,
        (
            out[COL_DNI]
            * out["cosz_plus"]
        )
        / (out[COL_GHI] + eps),
        0.0
    ).astype(np.float32)

    out["beam_ratio"] = (
        out["beam_ratio"]
        .clip(0.0, 1.5)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Variables con lag 1
    #
    # Se conserva kDHI como feature física histórica.
    # Se añade qDHI para disponer también de memoria en
    # el espacio transformado.
    # --------------------------------------------------------

    lag_bases = [
        COL_DNI,
        COL_DHI,
        COL_GHI,

        "kGHI",
        "kDNI",
        "kDHI",
        "qDHI",

        "diffuse_ratio",
        "beam_ratio",
    ]

    for base in lag_bases:

        out[f"{base}_lag1"] = (
            out[base]
            .shift(1)
        )


    # --------------------------------------------------------
    # Promedios móviles de tres pasos previos
    # 3 × 30 min = 90 min
    # --------------------------------------------------------

    roll_bases = [
        COL_DNI,
        COL_DHI,

        "kGHI",
        "kDNI",
        "kDHI",
        "qDHI",

        "diffuse_ratio",
        "beam_ratio",
    ]

    for base in roll_bases:

        out[f"{base}_roll3"] = (
            out[base]
            .shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )


    # --------------------------------------------------------
    # Dirección del viento como variable circular
    # --------------------------------------------------------

    wdir = np.deg2rad(
        pd.to_numeric(
            out[COL_WDIR],
            errors="coerce"
        ).fillna(0.0)
    )

    out["sin_wdir"] = np.sin(
        wdir
    ).astype(np.float32)

    out["cos_wdir"] = np.cos(
        wdir
    ).astype(np.float32)


    # --------------------------------------------------------
    # En esta etapa únicamente los NaN producidos por lag/roll
    # iniciales se sustituyen por cero, manteniendo la lógica
    # original de V2.3.
    # --------------------------------------------------------

    return out.fillna(0.0)


# ============================================================
# 5. CONSTRUIR DATAFRAME TEMPORAL
# ============================================================

df = build_timestamp(df_raw)


# ------------------------------------------------------------
# Auditoría del timestamp antes de eliminar registros
# ------------------------------------------------------------

n_bad_timestamp = int(
    df["timestamp"].isna().sum()
)

print(
    "Timestamps inválidos:",
    n_bad_timestamp
)

if n_bad_timestamp > 0:

    print(
        "\nPrimeros registros con timestamp inválido:"
    )

    print(
        df.loc[
            df["timestamp"].isna(),
            [COL_DATE, COL_HOUR, COL_MIN]
        ]
        .head(20)
        .to_string(index=False)
    )

    raise ValueError(
        "Existen timestamps inválidos. "
        "Debe revisarse el formato de Date antes de continuar."
    )


# ------------------------------------------------------------
# Orden y unicidad
# ------------------------------------------------------------

df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)


n_duplicates = int(
    df["timestamp"].duplicated().sum()
)

print(
    "Timestamps duplicados:",
    n_duplicates
)

if n_duplicates > 0:

    duplicated_rows = df.loc[
        df["timestamp"].duplicated(
            keep=False
        ),
        [
            "timestamp",
            COL_DATE,
            COL_HOUR,
            COL_MIN
        ]
    ]

    print(
        duplicated_rows
        .head(30)
        .to_string(index=False)
    )

    raise ValueError(
        "Se encontraron timestamps duplicados. "
        "No se eliminarán automáticamente."
    )


# ============================================================
# 6. AUDITORÍA ESTRICTA DE CONTINUIDAD A 30 MIN
# ============================================================

dt_min = (
    df["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)

bad_dt = (
    dt_min.notna()
    & ~np.isclose(
        dt_min,
        30.0
    )
)

n_bad_dt = int(
    bad_dt.sum()
)

print(
    "Intervalos distintos de 30 min:",
    n_bad_dt
)


if n_bad_dt > 0:

    bad_idx = np.where(
        bad_dt.to_numpy()
    )[0]

    gaps = []

    for i in bad_idx[:50]:

        gaps.append({
            "index": int(i),

            "timestamp_anterior":
                df.loc[
                    i - 1,
                    "timestamp"
                ],

            "timestamp_actual":
                df.loc[
                    i,
                    "timestamp"
                ],

            "delta_min":
                float(
                    dt_min.iloc[i]
                ),
        })

    gaps_df = pd.DataFrame(gaps)

    print(
        "\n=== PRIMEROS INTERVALOS ANÓMALOS ==="
    )

    print(
        gaps_df.to_string(
            index=False
        )
    )

    print(
        "\nDistribución de intervalos:"
    )

    print(
        dt_min
        .dropna()
        .value_counts()
        .sort_index()
        .head(30)
    )

    raise ValueError(
        "El histórico no es continuo a 30 min. "
        "Revise los intervalos mostrados antes de continuar."
    )


print(
    "✓ Histórico continuo a 30 min."
)


# ============================================================
# 7. CONSTRUIR FEATURES
# ============================================================

df = add_time_features(df)

df = add_kspace_features(
    df,
    cfg.thr_cs
)

df = add_partition_features(
    df,
    cfg.thr_cs
)


# ============================================================
# 8. VALIDACIONES NUMÉRICAS BÁSICAS
# ============================================================

required_engineered = [
    "timestamp",
    "cosz_plus",

    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",

    "kGHI",
    "kDNI",
    "kDHI",
    "qDHI",

    "diffuse_ratio",
    "beam_ratio",

    "sin_wdir",
    "cos_wdir",
]


missing_engineered = [
    c
    for c in required_engineered
    if c not in df.columns
]

if missing_engineered:

    raise ValueError(
        "Faltan variables de ingeniería:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_engineered
        )
    )


# ------------------------------------------------------------
# Comprobaciones del dominio
# ------------------------------------------------------------

if (
    df["kDNI"].min() < 0.0
    or
    df["kDNI"].max()
    > cfg.kdni_max + 1e-6
):
    raise ValueError(
        "kDNI fuera del dominio configurado."
    )


if (
    df["kDHI"].min() < 0.0
    or
    df["kDHI"].max()
    > cfg.kdhi_max + 1e-6
):
    raise ValueError(
        "kDHI fuera del dominio configurado."
    )


if (
    df["qDHI"].min() < 0.0
    or
    df["qDHI"].max()
    > QDHI_MAX + 1e-6
):
    raise ValueError(
        "qDHI fuera del dominio configurado."
    )


# ============================================================
# 9. RESUMEN
# ============================================================

print("\n=== FEATURE DATAFRAME V2.4 ===")

print(
    "Dimensión:",
    df.shape
)

print(
    "Rango temporal:",
    df["timestamp"].min(),
    "->",
    df["timestamp"].max()
)

print(
    "kDNI min/max:",
    float(df["kDNI"].min()),
    float(df["kDNI"].max())
)

print(
    "kDHI min/max:",
    float(df["kDHI"].min()),
    float(df["kDHI"].max())
)

print(
    "qDHI min/max:",
    float(df["qDHI"].min()),
    float(df["qDHI"].max())
)

print(
    "QDHI_MAX teórico:",
    QDHI_MAX
)

print(
    "\n✓ Celda 3 completada correctamente."
)

Timestamps inválidos: 0
Timestamps duplicados: 0
Intervalos distintos de 30 min: 0
✓ Histórico continuo a 30 min.


/tmp/ipykernel_4514/327510248.py:246: RuntimeWarning: invalid value encountered in divide
  ghi / csghi,
/tmp/ipykernel_4514/327510248.py:266: RuntimeWarning: invalid value encountered in divide
  dni / csdni,
/tmp/ipykernel_4514/327510248.py:295: RuntimeWarning: invalid value encountered in divide
  dhi / csdhi,



=== FEATURE DATAFRAME V2.4 ===
Dimensión: (473376, 62)
Rango temporal: 1998-01-01 00:00:00 -> 2024-12-31 23:30:00
kDNI min/max: 0.0 1.0
kDHI min/max: 0.0 7.918032646179199
qDHI min/max: 0.0 2.188075304031372
QDHI_MAX teórico: 2.1972245773362196

✓ Celda 3 completada correctamente.


In [4]:
# ============================================================
# 4. PI-LSTM V2.4 — CONTRATO DE FEATURES Y TARGETS
# ============================================================


# ============================================================
# 1. VARIABLES HISTÓRICAS
#
# Se conserva el contrato dimensional:
#
#     HIST_COLS = 34
#
# Respecto de V2.3:
#
#   kDHI_lag1   -> qDHI_lag1
#   kDHI_roll3  -> qDHI_roll3
#
# kDHI instantáneo se conserva como predictor físico.
# ============================================================

HIST_COLS = [

    # --------------------------------------------------------
    # Persistencia y dinámica radiativa
    # --------------------------------------------------------

    "DNI_lag1",
    "DNI_roll3",

    "DHI_lag1",
    "DHI_roll3",

    "GHI_lag1",

    "dGHI",
    "dDNI",
    "dDHI",


    # --------------------------------------------------------
    # Índices clear-sky instantáneos
    # --------------------------------------------------------

    "kGHI",
    "kDNI",
    "kDHI",


    # --------------------------------------------------------
    # Memoria de índices clear-sky
    # --------------------------------------------------------

    "kGHI_lag1",
    "kGHI_roll3",

    "kDNI_lag1",
    "kDNI_roll3",

    # V2.4:
    # memoria difusa en espacio logarítmico
    "qDHI_lag1",
    "qDHI_roll3",


    # --------------------------------------------------------
    # Memoria de partición radiativa
    # --------------------------------------------------------

    "diffuse_ratio_lag1",
    "diffuse_ratio_roll3",

    "beam_ratio_lag1",
    "beam_ratio_roll3",


    # --------------------------------------------------------
    # Meteorología
    # --------------------------------------------------------

    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,

    "sin_wdir",
    "cos_wdir",


    # --------------------------------------------------------
    # Geometría y clear-sky
    # --------------------------------------------------------

    "cosz_plus",

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# ============================================================
# 2. VARIABLES FUTURAS ESCALADAS
# ============================================================

FUT_COLS_SCALED = [

    # Geometría
    "cosz_plus",

    # Tiempo cíclico
    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",

    # Meteorología
    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,

    # Dirección del viento circular
    "sin_wdir",
    "cos_wdir",
]


# ============================================================
# 3. VARIABLES FUTURAS FÍSICAS SIN ESCALAR
# ============================================================

FUT_COLS_RAW = [

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# Contrato futuro completo

FUT_COLS = (
    FUT_COLS_SCALED
    + FUT_COLS_RAW
)


# ============================================================
# 4. TARGETS DEL MODELO
#
# V2.4:
#
#   salida 1 -> kDNI
#   salida 2 -> qDHI = log(1 + kDHI)
#
# Además se conservan variables físicas auxiliares en Y_EXT
# para construir la loss físicamente informada.
# ============================================================

TARGET_COLS = [
    "kDNI",
    "qDHI",
]


# ============================================================
# 5. TARGET EXTENDIDO PARA LA LOSS
#
# Orden contractual:
#
# 0 -> kDNI_true
# 1 -> qDHI_true
# 2 -> GHI_true
# 3 -> DNI_cs
# 4 -> DHI_cs
# 5 -> GHI_cs
# 6 -> cosz_plus
# 7 -> DHI_true
#
# DHI_true se incorpora explícitamente en V2.4 para facilitar
# diagnósticos y futuras extensiones de la loss.
# ============================================================

Y_EXT_COLS = [

    "kDNI",
    "qDHI",

    COL_GHI,

    COL_CSDNI,
    COL_CSDHI,
    COL_CSGHI,

    "cosz_plus",

    COL_DHI,
]


# ============================================================
# 6. VALIDACIÓN DEL CONTRATO
# ============================================================

print("=== CONTRATO DE VARIABLES PI-LSTM V2.4 ===")

print(
    "HIST_COLS       :",
    len(HIST_COLS)
)

print(
    "FUT scaled      :",
    len(FUT_COLS_SCALED)
)

print(
    "FUT raw         :",
    len(FUT_COLS_RAW)
)

print(
    "FUT total       :",
    len(FUT_COLS)
)

print(
    "TARGET_COLS     :",
    len(TARGET_COLS)
)

print(
    "Y_EXT_COLS      :",
    len(Y_EXT_COLS)
)


# ------------------------------------------------------------
# Contratos dimensionales
# ------------------------------------------------------------

assert len(HIST_COLS) == 34, (
    f"Se esperaban 34 variables históricas "
    f"y se obtuvieron {len(HIST_COLS)}."
)

assert len(FUT_COLS_SCALED) == 14, (
    f"Se esperaban 14 variables futuras escaladas "
    f"y se obtuvieron {len(FUT_COLS_SCALED)}."
)

assert len(FUT_COLS_RAW) == 3

assert len(FUT_COLS) == 17

assert len(TARGET_COLS) == 2

assert len(Y_EXT_COLS) == 8


# ============================================================
# 7. VERIFICAR EXISTENCIA DE TODAS LAS COLUMNAS
# ============================================================

all_required_model_cols = list(dict.fromkeys(
    HIST_COLS
    + FUT_COLS
    + TARGET_COLS
    + Y_EXT_COLS
))


missing_model_cols = [
    c
    for c in all_required_model_cols
    if c not in df.columns
]


if missing_model_cols:

    raise ValueError(
        "Faltan columnas requeridas por el contrato V2.4:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_model_cols
        )
    )


# ============================================================
# 8. VERIFICACIÓN DE FINITUD
# ============================================================

nonfinite_report = {}

for col in all_required_model_cols:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    n_bad = int(
        np.sum(
            ~np.isfinite(values)
        )
    )

    if n_bad > 0:
        nonfinite_report[col] = n_bad


if nonfinite_report:

    raise ValueError(
        "Se encontraron valores no finitos en variables "
        "del contrato:\n"
        + json.dumps(
            nonfinite_report,
            indent=2
        )
    )


# ============================================================
# 9. VERIFICACIONES ESPECÍFICAS DE V2.4
# ============================================================

assert np.all(
    df["qDHI"].to_numpy(dtype=float)
    >= 0.0
)

assert (
    float(df["qDHI"].max())
    <= QDHI_MAX + 1e-6
)

# Verificar consistencia exacta:
#
# qDHI = log1p(kDHI)

qdhi_reconstructed = np.log1p(
    df["kDHI"].to_numpy(dtype=float)
)

qdhi_error = np.max(
    np.abs(
        qdhi_reconstructed
        - df["qDHI"].to_numpy(dtype=float)
    )
)

print(
    "\nMáximo error qDHI - log1p(kDHI):",
    qdhi_error
)

assert qdhi_error < 1e-5


# ============================================================
# 10. RESUMEN DEL CONTRATO
# ============================================================

print("\n✓ Contrato V2.4 válido")

print(
    "✓ Input histórico :",
    f"{cfg.L} × {len(HIST_COLS)}"
)

print(
    "✓ Input futuro    :",
    f"{cfg.H} × {len(FUT_COLS)}"
)

print(
    "✓ Target neuronal :",
    f"{cfg.H} × {len(TARGET_COLS)}"
)

print(
    "✓ Target extendido:",
    f"{cfg.H} × {len(Y_EXT_COLS)}"
)

print("\nTargets neuronales:")
print(TARGET_COLS)

print("\nMemoria difusa V2.4:")
print(
    [
        "kDHI",
        "qDHI_lag1",
        "qDHI_roll3"
    ]
)

=== CONTRATO DE VARIABLES PI-LSTM V2.4 ===
HIST_COLS       : 34
FUT scaled      : 14
FUT raw         : 3
FUT total       : 17
TARGET_COLS     : 2
Y_EXT_COLS      : 8

Máximo error qDHI - log1p(kDHI): 1.2221614253959956e-07

✓ Contrato V2.4 válido
✓ Input histórico : 72 × 34
✓ Input futuro    : 24 × 17
✓ Target neuronal : 24 × 2
✓ Target extendido: 24 × 8

Targets neuronales:
['kDNI', 'qDHI']

Memoria difusa V2.4:
['kDHI', 'qDHI_lag1', 'qDHI_roll3']


In [5]:
# ============================================================
# 5. PI-LSTM V2.4 — SPLIT TEMPORAL Y ESCALAMIENTO
# ============================================================


# ============================================================
# 1. DEFINIR LÍMITES TEMPORALES
# ============================================================

train_end = pd.Timestamp(cfg.train_end)
val_end = pd.Timestamp(cfg.val_end)


# ============================================================
# 2. PARTICIONAMIENTO CRONOLÓGICO
# ============================================================

train_df = (
    df[
        df["timestamp"] <= train_end
    ]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    df[
        (df["timestamp"] > train_end)
        &
        (df["timestamp"] <= val_end)
    ]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    df[
        df["timestamp"] > val_end
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 3. VALIDAR QUE LOS SUBCONJUNTOS NO ESTÉN VACÍOS
# ============================================================

if len(train_df) == 0:
    raise ValueError("train_df está vacío.")

if len(val_df) == 0:
    raise ValueError("val_df está vacío.")

if len(test_df) == 0:
    raise ValueError("test_df está vacío.")


# ============================================================
# 4. AUDITORÍA TEMPORAL DE LOS SPLITS
# ============================================================

print("=== SPLIT TEMPORAL PI-LSTM V2.4 ===")

for name, part in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:

    print(f"\n{name}")

    print(
        "  n       :",
        f"{len(part):,}"
    )

    print(
        "  inicio  :",
        part["timestamp"].min()
    )

    print(
        "  fin     :",
        part["timestamp"].max()
    )


# ============================================================
# 5. CONTRATOS TEMPORALES
# ============================================================

assert (
    train_df["timestamp"].max()
    == train_end
)

assert (
    val_df["timestamp"].min()
    == train_end + pd.Timedelta(minutes=30)
)

assert (
    val_df["timestamp"].max()
    == val_end
)

assert (
    test_df["timestamp"].min()
    == val_end + pd.Timedelta(minutes=30)
)


# ============================================================
# 6. VERIFICAR CONTINUIDAD INTERNA
# ============================================================

def check_continuity(part, name):

    dt = (
        part["timestamp"]
        .diff()
        .dt.total_seconds()
        .div(60.0)
        .dropna()
        .to_numpy()
    )

    if not np.allclose(
        dt,
        30.0
    ):
        raise ValueError(
            f"{name} no es continuo a 30 min."
        )

    print(
        f"✓ {name} continuo a 30 min."
    )


check_continuity(
    train_df,
    "TRAIN"
)

check_continuity(
    val_df,
    "VALIDATION"
)

check_continuity(
    test_df,
    "TEST"
)


# ============================================================
# 7. CREAR ESCALADORES
# ============================================================

scaler_hist = StandardScaler()

scaler_fut = StandardScaler()


# ============================================================
# 8. AJUSTAR EXCLUSIVAMENTE CON TRAIN
#
# Esta es una restricción metodológica crítica:
#
#     scaler.fit(train)
#
# Nunca:
#
#     scaler.fit(val)
#     scaler.fit(test)
#
# ============================================================

scaler_hist.fit(
    train_df[HIST_COLS]
)

scaler_fut.fit(
    train_df[FUT_COLS_SCALED]
)


# ============================================================
# 9. FUNCIÓN DE TRANSFORMACIÓN
# ============================================================

def transform_partition(
    part,
    scaler_hist,
    scaler_fut
):
    """
    Transforma una partición sin reajustar los escaladores.

    HIST_COLS:
        34 variables escaladas.

    FUT_COLS_SCALED:
        14 variables escaladas.

    FUT_COLS_RAW:
        3 variables clear-sky mantenidas en unidades físicas.

    Y_EXT_COLS:
        targets y variables auxiliares para la loss,
        sin escalamiento.
    """

    x_hist = scaler_hist.transform(
        part[HIST_COLS]
    ).astype(np.float32)

    x_fut_scaled = scaler_fut.transform(
        part[FUT_COLS_SCALED]
    ).astype(np.float32)

    x_fut_raw = (
        part[FUT_COLS_RAW]
        .to_numpy(dtype=np.float32)
    )

    x_fut = np.concatenate(
        [
            x_fut_scaled,
            x_fut_raw
        ],
        axis=1
    ).astype(np.float32)

    y_ext = (
        part[Y_EXT_COLS]
        .to_numpy(dtype=np.float32)
    )

    return (
        x_hist,
        x_fut,
        y_ext
    )


# ============================================================
# 10. TRANSFORMAR TRAIN, VALIDATION Y TEST
# ============================================================

XH_train, XF_train, Y_train = (
    transform_partition(
        train_df,
        scaler_hist,
        scaler_fut
    )
)

XH_val, XF_val, Y_val = (
    transform_partition(
        val_df,
        scaler_hist,
        scaler_fut
    )
)

XH_test, XF_test, Y_test = (
    transform_partition(
        test_df,
        scaler_hist,
        scaler_fut
    )
)


# ============================================================
# 11. AUDITORÍA DE SHAPES
# ============================================================

print("\n=== MATRICES TRANSFORMADAS ===")

print(
    "XH_train:",
    XH_train.shape
)

print(
    "XF_train:",
    XF_train.shape
)

print(
    "Y_train :",
    Y_train.shape
)

print(
    "\nXH_val  :",
    XH_val.shape
)

print(
    "XF_val  :",
    XF_val.shape
)

print(
    "Y_val   :",
    Y_val.shape
)

print(
    "\nXH_test :",
    XH_test.shape
)

print(
    "XF_test :",
    XF_test.shape
)

print(
    "Y_test  :",
    Y_test.shape
)


# ============================================================
# 12. CONTRATOS DIMENSIONALES
# ============================================================

assert XH_train.shape[1] == 34
assert XH_val.shape[1] == 34
assert XH_test.shape[1] == 34

assert XF_train.shape[1] == 17
assert XF_val.shape[1] == 17
assert XF_test.shape[1] == 17

assert Y_train.shape[1] == 8
assert Y_val.shape[1] == 8
assert Y_test.shape[1] == 8


# ============================================================
# 13. VALIDAR FINITUD
# ============================================================

arrays_to_check = {
    "XH_train": XH_train,
    "XF_train": XF_train,
    "Y_train": Y_train,

    "XH_val": XH_val,
    "XF_val": XF_val,
    "Y_val": Y_val,

    "XH_test": XH_test,
    "XF_test": XF_test,
    "Y_test": Y_test,
}


for name, arr in arrays_to_check.items():

    if not np.all(
        np.isfinite(arr)
    ):
        raise ValueError(
            f"{name} contiene valores no finitos."
        )


print(
    "\n✓ Todas las matrices contienen valores finitos."
)


# ============================================================
# 14. AUDITORÍA DEL ESCALAMIENTO
#
# En TRAIN:
# media ≈ 0
# desviación ≈ 1
#
# No esperamos exactamente lo mismo en val/test.
# ============================================================

hist_train_mean = np.mean(
    XH_train,
    axis=0
)

hist_train_std = np.std(
    XH_train,
    axis=0
)

fut_train_mean = np.mean(
    XF_train[
        :,
        :len(FUT_COLS_SCALED)
    ],
    axis=0
)

fut_train_std = np.std(
    XF_train[
        :,
        :len(FUT_COLS_SCALED)
    ],
    axis=0
)


print(
    "\n=== AUDITORÍA STANDARD SCALER ==="
)

print(
    "Máx |media| HIST train:",
    float(
        np.max(
            np.abs(hist_train_mean)
        )
    )
)

print(
    "Máx |std - 1| HIST train:",
    float(
        np.max(
            np.abs(
                hist_train_std - 1.0
            )
        )
    )
)

print(
    "Máx |media| FUT train:",
    float(
        np.max(
            np.abs(fut_train_mean)
        )
    )
)

print(
    "Máx |std - 1| FUT train:",
    float(
        np.max(
            np.abs(
                fut_train_std - 1.0
            )
        )
    )
)


# ============================================================
# 15. VERIFICAR QUE LAS VARIABLES CLEAR-SKY FUTURAS
#     PERMANECEN EN UNIDADES FÍSICAS
# ============================================================

raw_start = len(
    FUT_COLS_SCALED
)

for j, col in enumerate(
    FUT_COLS_RAW
):

    transformed_col = (
        XF_train[
            :,
            raw_start + j
        ]
    )

    original_col = (
        train_df[col]
        .to_numpy(
            dtype=np.float32
        )
    )

    max_diff = float(
        np.max(
            np.abs(
                transformed_col
                - original_col
            )
        )
    )

    print(
        f"Máx diferencia raw {col}:",
        max_diff
    )

    assert max_diff < 1e-6


print(
    "\n✓ Clear-sky futuro permanece sin escalar."
)


# ============================================================
# 16. RESUMEN FINAL
# ============================================================

print(
    "\n✓ Split temporal correcto."
)

print(
    "✓ Scalers ajustados exclusivamente con TRAIN."
)

print(
    "✓ Contrato 34 / 17 / 8 validado."
)

print(
    "✓ Celda 5 completada correctamente."
)

=== SPLIT TEMPORAL PI-LSTM V2.4 ===

TRAIN
  n       : 385,680
  inicio  : 1998-01-01 00:00:00
  fin     : 2019-12-31 23:30:00

VALIDATION
  n       : 52,608
  inicio  : 2020-01-01 00:00:00
  fin     : 2022-12-31 23:30:00

TEST
  n       : 35,088
  inicio  : 2023-01-01 00:00:00
  fin     : 2024-12-31 23:30:00
✓ TRAIN continuo a 30 min.
✓ VALIDATION continuo a 30 min.
✓ TEST continuo a 30 min.

=== MATRICES TRANSFORMADAS ===
XH_train: (385680, 34)
XF_train: (385680, 17)
Y_train : (385680, 8)

XH_val  : (52608, 34)
XF_val  : (52608, 17)
Y_val   : (52608, 8)

XH_test : (35088, 34)
XF_test : (35088, 17)
Y_test  : (35088, 8)

✓ Todas las matrices contienen valores finitos.

=== AUDITORÍA STANDARD SCALER ===
Máx |media| HIST train: 2.585711555980197e-08
Máx |std - 1| HIST train: 1.1920928955078125e-07
Máx |media| FUT train: 1.567697438531468e-08
Máx |std - 1| FUT train: 1.1920928955078125e-07
Máx diferencia raw Clearsky GHI: 0.0
Máx diferencia raw Clearsky DNI: 0.0
Máx diferencia raw Clearsk

In [6]:
# ============================================================
# 6. PI-LSTM V2.4 V1 — GENERACIÓN DE VENTANAS
#    Y DATASETS CON SAMPLE_WEIGHT
# ============================================================


# ============================================================
# 1. CONTAR VENTANAS POR PARTICIÓN
# ============================================================

def count_windows(
    n_rows,
    L,
    H,
    stride
):
    """
    Número de ventanas válidas dentro de una partición temporal.

    Cada ventana utiliza:

        L pasos históricos
        H pasos futuros

    y avanza según stride.
    """

    return (
        (n_rows - L - H)
        // stride
    ) + 1


nwin_train = count_windows(
    len(train_df),
    cfg.L,
    cfg.H,
    cfg.stride
)

nwin_val = count_windows(
    len(val_df),
    cfg.L,
    cfg.H,
    cfg.stride
)

nwin_test = count_windows(
    len(test_df),
    cfg.L,
    cfg.H,
    cfg.stride
)


print(
    "=== NÚMERO DE VENTANAS ==="
)

print(
    "TRAIN:",
    f"{nwin_train:,}"
)

print(
    "VAL  :",
    f"{nwin_val:,}"
)

print(
    "TEST :",
    f"{nwin_test:,}"
)


assert nwin_train > 0
assert nwin_val > 0
assert nwin_test > 0


# ============================================================
# 2. GENERADOR DE VENTANAS
#
# SALIDA:
#
#   (
#       (x_hist, x_fut),
#       y_ext,
#       sample_weight
#   )
#
# sample_weight tiene longitud H y, por ahora, contiene unos.
#
# Esto reproduce el contrato de entrenamiento de V2.3:
#
#       x, y, sample_weight
#
# El peso físico temporal adicional se construirá dentro
# de PartitionAwareLossV24V1.
# ============================================================

def window_generator(
    X_hist,
    X_fut,
    Y_ext,
    L,
    H,
    stride
):

    n = len(
        X_hist
    )

    last_start = (
        n
        - L
        - H
    )


    for start in range(
        0,
        last_start + 1,
        stride
    ):

        hist_end = (
            start
            + L
        )

        fut_end = (
            hist_end
            + H
        )


        x_hist = (
            X_hist[
                start:hist_end
            ]
            .astype(
                np.float32,
                copy=False
            )
        )


        x_fut = (
            X_fut[
                hist_end:fut_end
            ]
            .astype(
                np.float32,
                copy=False
            )
        )


        y_ext = (
            Y_ext[
                hist_end:fut_end
            ]
            .astype(
                np.float32,
                copy=False
            )
        )


        # ----------------------------------------------------
        # sample_weight explícito
        #
        # Dimensión:
        #
        #     (H,)
        #
        # Igual que en V2.3.
        #
        # La modulación física:
        #
        #     0.5 + sqrt(cosz)
        #
        # NO se aplica aquí.
        #
        # Se aplicará dentro de la loss para reproducir
        # fielmente el comportamiento de V2.3.
        # ----------------------------------------------------

        sample_weight = np.ones(
            shape=(H,),
            dtype=np.float32
        )


        yield (
            (
                x_hist,
                x_fut
            ),
            y_ext,
            sample_weight
        )


# ============================================================
# 3. OUTPUT SIGNATURE
# ============================================================

output_signature = (

    (
        tf.TensorSpec(
            shape=(
                cfg.L,
                len(HIST_COLS)
            ),
            dtype=tf.float32
        ),

        tf.TensorSpec(
            shape=(
                cfg.H,
                len(FUT_COLS)
            ),
            dtype=tf.float32
        ),
    ),

    tf.TensorSpec(
        shape=(
            cfg.H,
            len(Y_EXT_COLS)
        ),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(
            cfg.H,
        ),
        dtype=tf.float32
    ),
)


# ============================================================
# 4. FUNCIÓN PARA CONSTRUIR DATASET
#
# IMPORTANTE:
#
# Primero se forman los batches y DESPUÉS se aplica repeat().
#
# Esto evita que el último batch incompleto de una pasada
# se complete con observaciones de la siguiente pasada.
# ============================================================

def make_dataset(
    X_hist,
    X_fut,
    Y_ext,
    n_windows,
    training=False,
    repeat=False
):

    def generator():

        yield from window_generator(
            X_hist,
            X_fut,
            Y_ext,
            cfg.L,
            cfg.H,
            cfg.stride
        )


    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=output_signature
    )


    # --------------------------------------------------------
    # SHUFFLE SOLO EN TRAIN
    # --------------------------------------------------------

    if training:

        shuffle_buffer = min(
            n_windows,
            4096
        )


        ds = ds.shuffle(
            buffer_size=shuffle_buffer,
            seed=SEED,
            reshuffle_each_iteration=True
        )


    # --------------------------------------------------------
    # FORMAR BATCHES
    #
    # drop_remainder=False:
    #
    # se conserva el último batch incompleto.
    # --------------------------------------------------------

    ds = ds.batch(
        cfg.batch_size,
        drop_remainder=False
    )


    # --------------------------------------------------------
    # REPETIR DESPUÉS DE BATCH
    #
    # Necesario para fit() cuando especificamos:
    #
    #     steps_per_epoch
    #     validation_steps
    #
    # TRAIN y VAL:
    #     repeat=True
    #
    # TEST:
    #     repeat=False
    # --------------------------------------------------------

    if repeat:

        ds = ds.repeat()


    # --------------------------------------------------------
    # PREFETCH
    # --------------------------------------------------------

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )


    return ds


# ============================================================
# 5. CONSTRUIR DATASETS
# ============================================================

ds_train = make_dataset(

    XH_train,
    XF_train,
    Y_train,

    n_windows=nwin_train,

    training=True,

    repeat=True
)


ds_val = make_dataset(

    XH_val,
    XF_val,
    Y_val,

    n_windows=nwin_val,

    training=False,

    repeat=True
)


ds_test = make_dataset(

    XH_test,
    XF_test,
    Y_test,

    n_windows=nwin_test,

    training=False,

    repeat=False
)


# ============================================================
# 6. STEPS EXACTOS POR ÉPOCA
# ============================================================

steps_train = int(
    np.ceil(
        nwin_train
        /
        cfg.batch_size
    )
)


steps_val = int(
    np.ceil(
        nwin_val
        /
        cfg.batch_size
    )
)


steps_test = int(
    np.ceil(
        nwin_test
        /
        cfg.batch_size
    )
)


print(
    "\n=== STEPS POR PARTICIÓN ==="
)

print(
    "TRAIN steps:",
    steps_train
)

print(
    "VAL steps  :",
    steps_val
)

print(
    "TEST steps :",
    steps_test
)


# ============================================================
# 7. INSPECCIÓN DE UN BATCH
# ============================================================

sample_batch = next(
    iter(
        ds_train
    )
)


(
    (
        batch_xh,
        batch_xf
    ),
    batch_y,
    batch_sw
) = sample_batch


print(
    "\n=== SHAPES DEL PRIMER BATCH ==="
)

print(
    "x_hist:",
    batch_xh.shape
)

print(
    "x_fut :",
    batch_xf.shape
)

print(
    "y_ext :",
    batch_y.shape
)

print(
    "sample_weight:",
    batch_sw.shape
)


# ============================================================
# 8. VALIDACIONES DE SHAPE
# ============================================================

assert batch_xh.shape[1:] == (
    cfg.L,
    len(HIST_COLS)
)


assert batch_xf.shape[1:] == (
    cfg.H,
    len(FUT_COLS)
)


assert batch_y.shape[1:] == (
    cfg.H,
    len(Y_EXT_COLS)
)


assert batch_sw.shape[1:] == (
    cfg.H,
)


# ============================================================
# 9. VALIDAR SAMPLE_WEIGHT
# ============================================================

sw_min = float(
    tf.reduce_min(
        batch_sw
    ).numpy()
)

sw_max = float(
    tf.reduce_max(
        batch_sw
    ).numpy()
)

sw_mean = float(
    tf.reduce_mean(
        batch_sw
    ).numpy()
)


print(
    "\n=== SAMPLE WEIGHT ==="
)

print(
    "min :",
    sw_min
)

print(
    "max :",
    sw_max
)

print(
    "mean:",
    sw_mean
)


assert np.isclose(
    sw_min,
    1.0
)

assert np.isclose(
    sw_max,
    1.0
)

assert np.isclose(
    sw_mean,
    1.0
)


# ============================================================
# 10. VALIDAR FINITUD
# ============================================================

for name, tensor in {

    "batch_xh":
        batch_xh,

    "batch_xf":
        batch_xf,

    "batch_y":
        batch_y,

    "batch_sw":
        batch_sw,

}.items():

    if not bool(
        tf.reduce_all(
            tf.math.is_finite(
                tensor
            )
        ).numpy()
    ):

        raise ValueError(
            f"{name} contiene NaN o Inf."
        )


# ============================================================
# 11. VALIDAR PRIMERA VENTANA TEMPORAL
# ============================================================

first_train_start = 0

first_train_hist_start = (
    train_df.iloc[
        first_train_start
    ]["timestamp"]
)

first_train_hist_end = (
    train_df.iloc[
        first_train_start
        + cfg.L
        - 1
    ]["timestamp"]
)

first_train_future_start = (
    train_df.iloc[
        first_train_start
        + cfg.L
    ]["timestamp"]
)

first_train_future_end = (
    train_df.iloc[
        first_train_start
        + cfg.L
        + cfg.H
        - 1
    ]["timestamp"]
)


print(
    "\n=== PRIMERA VENTANA TRAIN ==="
)

print(
    "Historia inicio:",
    first_train_hist_start
)

print(
    "Historia fin   :",
    first_train_hist_end
)

print(
    "Futuro inicio  :",
    first_train_future_start
)

print(
    "Futuro fin     :",
    first_train_future_end
)


# ============================================================
# 12. VALIDAR STRIDE
# ============================================================

second_window_start = (
    cfg.stride
)


first_future_start = (
    train_df.iloc[
        cfg.L
    ]["timestamp"]
)

second_future_start = (
    train_df.iloc[
        second_window_start
        + cfg.L
    ]["timestamp"]
)


stride_minutes = (

    second_future_start
    -
    first_future_start

).total_seconds() / 60.0


print(
    "\nStride temporal entre ventanas:",
    stride_minutes,
    "min"
)


assert np.isclose(
    stride_minutes,
    30.0 * cfg.stride
)


# ============================================================
# 13. RESUMEN
# ============================================================

print(
    "\n✓ Ventanas construidas sin cruzar particiones."
)

print(
    "✓ sample_weight restaurado como tercer elemento."
)

print(
    "✓ sample_weight inicial = 1 para todos los horizontes."
)

print(
    "✓ repeat() aplicado después de batch()."
)

print(
    "✓ TRAIN y VALIDATION son repetibles para fit()."
)

print(
    "✓ TEST permanece finito."
)

print(
    "✓ Celda 6 V1 completada correctamente."
)

=== NÚMERO DE VENTANAS ===
TRAIN: 192,793
VAL  : 26,257
TEST : 17,497

=== STEPS POR PARTICIÓN ===
TRAIN steps: 6025
VAL steps  : 821
TEST steps : 547

=== SHAPES DEL PRIMER BATCH ===
x_hist: (32, 72, 34)
x_fut : (32, 24, 17)
y_ext : (32, 24, 8)
sample_weight: (32, 24)

=== SAMPLE WEIGHT ===
min : 1.0
max : 1.0
mean: 1.0

=== PRIMERA VENTANA TRAIN ===
Historia inicio: 1998-01-01 00:00:00
Historia fin   : 1998-01-02 11:30:00
Futuro inicio  : 1998-01-02 12:00:00
Futuro fin     : 1998-01-02 23:30:00

Stride temporal entre ventanas: 60.0 min

✓ Ventanas construidas sin cruzar particiones.
✓ sample_weight restaurado como tercer elemento.
✓ sample_weight inicial = 1 para todos los horizontes.
✓ repeat() aplicado después de batch().
✓ TRAIN y VALIDATION son repetibles para fit().
✓ TEST permanece finito.
✓ Celda 6 V1 completada correctamente.


In [7]:
# ============================================================
# 7. PI-LSTM V2.4 — ARQUITECTURA ENCODER–DECODER
#    Y CABEZAS DE SALIDA
# ============================================================


# ============================================================
# 1. CAPA SERIALIZABLE qDHI -> kDHI
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="PILSTMV24"
)
class Expm1Layer(Layer):
    """
    Convierte:

        qDHI = log(1 + kDHI)

    nuevamente a:

        kDHI = exp(qDHI) - 1

    Se implementa como Layer registrada para permitir
    serialización/deserialización segura del modelo .keras.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        return tf.math.expm1(inputs)

    def get_config(self):
        config = super().get_config()
        return config


# ============================================================
# 2. CONSTRUCCIÓN DEL BACKBONE
# ============================================================

def build_pilstm_v24_backbone(
    L,
    H,
    n_hist,
    n_fut,
    latent_enc,
    latent_dec,
    dropout,
    kdni_max,
    kdhi_max
):

    # --------------------------------------------------------
    # Entradas
    # --------------------------------------------------------

    x_hist = Input(
        shape=(L, n_hist),
        name="x_hist"
    )

    x_fut = Input(
        shape=(H, n_fut),
        name="x_fut"
    )


    # ========================================================
    # 3. ENCODER
    # ========================================================

    encoder_lstm = LSTM(
        latent_enc,
        return_sequences=True,
        return_state=True,
        name="encoder_lstm"
    )

    (
        enc_seq,
        enc_h,
        enc_c
    ) = encoder_lstm(x_hist)


    enc_seq = LayerNormalization(
        name="encoder_layernorm"
    )(enc_seq)


    # ========================================================
    # 4. DECODER
    # ========================================================

    decoder_lstm = LSTM(
        latent_dec,
        return_sequences=True,
        return_state=False,
        name="decoder_lstm"
    )

    dec_seq = decoder_lstm(
        x_fut,
        initial_state=[
            enc_h,
            enc_c
        ]
    )

    dec_seq = LayerNormalization(
        name="decoder_layernorm"
    )(dec_seq)


    # ========================================================
    # 5. ATENCIÓN ADITIVA
    # ========================================================

    attention = AdditiveAttention(
        name="additive_attention"
    )

    att_context = attention([
        dec_seq,
        enc_seq
    ])


    # ========================================================
    # 6. FUSIÓN
    #
    # decoder state
    # + attention context
    # + future exogenous predictors
    # ========================================================

    fusion = Concatenate(
        axis=-1,
        name="fusion"
    )([
        dec_seq,
        att_context,
        x_fut
    ])


    # ========================================================
    # 7. BLOQUE DENSO
    # ========================================================

    z = TimeDistributed(
        Dense(
            64,
            activation="swish"
        ),
        name="td_dense64"
    )(fusion)

    z = Dropout(
        dropout,
        name="dropout"
    )(z)

    z = TimeDistributed(
        Dense(
            32,
            activation="swish"
        ),
        name="td_dense32"
    )(z)


    # ========================================================
    # 8. RAMA DIRECTA — kDNI
    #
    # kDNI_hat = kdni_max * sigmoid(z)
    # ========================================================

    kdni_sig = TimeDistributed(
        Dense(
            1,
            activation="sigmoid"
        ),
        name="kDNI_sig"
    )(z)

    kdni_hat = Rescaling(
        scale=kdni_max,
        name="kDNI_hat"
    )(kdni_sig)


    # ========================================================
    # 9. RAMA DIFUSA — qDHI
    #
    # qDHI_hat =
    #     log(1 + kdhi_max) * sigmoid(z)
    #
    # posteriormente:
    #
    # kDHI_hat =
    #     exp(qDHI_hat) - 1
    # ========================================================

    qdhi_sig = TimeDistributed(
        Dense(
            1,
            activation="sigmoid"
        ),
        name="qDHI_sig"
    )(z)

    qdhi_max = float(
        np.log1p(kdhi_max)
    )

    qdhi_hat = Rescaling(
        scale=qdhi_max,
        name="qDHI_hat"
    )(qdhi_sig)

    kdhi_hat = Expm1Layer(
        name="kDHI_hat"
    )(qdhi_hat)


    # ========================================================
    # 10. SALIDA NEURONAL DE ENTRENAMIENTO
    #
    # La salida contractual del backbone será:
    #
    #   [kDNI_hat, qDHI_hat]
    #
    # kDHI_hat se conserva como tensor intermedio y será
    # reconstruido dentro de la loss a partir de qDHI_hat.
    # ========================================================

    model_output = Concatenate(
        axis=-1,
        name="model_output"
    )([
        kdni_hat,
        qdhi_hat
    ])


    # ========================================================
    # 11. MODELO
    # ========================================================

    model = Model(
        inputs=[
            x_hist,
            x_fut
        ],
        outputs=model_output,
        name="PI_LSTM_V2_4"
    )


    return model


# ============================================================
# 12. CONSTRUIR BACKBONE
# ============================================================

backbone = build_pilstm_v24_backbone(
    L=cfg.L,
    H=cfg.H,

    n_hist=len(HIST_COLS),
    n_fut=len(FUT_COLS),

    latent_enc=cfg.latent_enc,
    latent_dec=cfg.latent_dec,

    dropout=cfg.dropout,

    kdni_max=cfg.kdni_max,
    kdhi_max=cfg.kdhi_max
)


# ============================================================
# 13. RESUMEN DE ARQUITECTURA
# ============================================================

backbone.summary()


# ============================================================
# 14. AUDITORÍA DE SHAPE DE SALIDA
# ============================================================

test_pred = backbone(
    [
        batch_xh,
        batch_xf
    ],
    training=False
)


print(
    "\n=== AUDITORÍA BACKBONE V2.4 ==="
)

print(
    "Input histórico:",
    batch_xh.shape
)

print(
    "Input futuro   :",
    batch_xf.shape
)

print(
    "Output         :",
    test_pred.shape
)


assert test_pred.shape[1:] == (
    cfg.H,
    2
)


# ============================================================
# 15. AUDITORÍA DE DOMINIOS DE SALIDA
# ============================================================

kdni_pred_test = (
    test_pred[
        ...,
        0
    ]
    .numpy()
)

qdhi_pred_test = (
    test_pred[
        ...,
        1
    ]
    .numpy()
)

kdhi_pred_test = np.expm1(
    qdhi_pred_test
)


print(
    "\nkDNI pred min/max:",
    float(
        np.min(kdni_pred_test)
    ),
    float(
        np.max(kdni_pred_test)
    )
)

print(
    "qDHI pred min/max:",
    float(
        np.min(qdhi_pred_test)
    ),
    float(
        np.max(qdhi_pred_test)
    )
)

print(
    "kDHI reconstruido min/max:",
    float(
        np.min(kdhi_pred_test)
    ),
    float(
        np.max(kdhi_pred_test)
    )
)


# ============================================================
# 16. CONTRATOS DE DOMINIO
# ============================================================

assert np.all(
    kdni_pred_test >= 0.0
)

assert np.all(
    kdni_pred_test
    <= cfg.kdni_max + 1e-6
)

assert np.all(
    qdhi_pred_test >= 0.0
)

assert np.all(
    qdhi_pred_test
    <= QDHI_MAX + 1e-6
)

assert np.all(
    kdhi_pred_test >= 0.0
)

assert np.all(
    kdhi_pred_test
    <= cfg.kdhi_max + 1e-5
)


# ============================================================
# 17. VERIFICAR SERIALIZACIÓN
# ============================================================

test_model_path = (
    "_tmp_pilstm_v24_serialization_test.keras"
)

backbone.save(
    test_model_path
)

loaded_test_model = (
    tf.keras.models.load_model(
        test_model_path,
        compile=False
    )
)

loaded_pred = loaded_test_model(
    [
        batch_xh,
        batch_xf
    ],
    training=False
).numpy()

original_pred = (
    test_pred.numpy()
)

serialization_diff = float(
    np.max(
        np.abs(
            loaded_pred
            - original_pred
        )
    )
)


print(
    "\nMáx diferencia después de guardar/cargar:",
    serialization_diff
)


assert serialization_diff < 1e-6


# ------------------------------------------------------------
# Eliminar archivo temporal
# ------------------------------------------------------------

if Path(
    test_model_path
).exists():

    Path(
        test_model_path
    ).unlink()


# ============================================================
# 18. RESUMEN FINAL
# ============================================================

print(
    "\n✓ Arquitectura PI-LSTM V2.4 construida."
)

print(
    "✓ Salida neuronal:",
    "[kDNI_hat, qDHI_hat]"
)

print(
    "✓ kDNI ∈ [0,",
    cfg.kdni_max,
    "]"
)

print(
    "✓ qDHI ∈ [0,",
    QDHI_MAX,
    "]"
)

print(
    "✓ kDHI reconstruido ∈ [0,",
    cfg.kdhi_max,
    "]"
)

print(
    "✓ Modelo serializable sin Lambda."
)

print(
    "✓ Celda 7 completada correctamente."
)

Model: "PI_LSTM_V2_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x_hist (InputLayer) │ (None, 72, 34)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ x_fut (InputLayer)  │ (None, 24, 17)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 72, 96),  │     50,304 │ x_hist[0][0]      │
│                     │ (None, 96),       │            │                   │
│                     │ (None, 96)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ (None, 24, 96)    │     43,776 │ x_fut[0][0],      │
│                     │                   │            │ encoder_lstm[0][… │
│                     │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layernorm   │ (None, 24, 96)    │        192 │ decoder_lstm[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_layernorm   │ (None, 72, 96)    │        192 │ encoder_lstm[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_attention  │ (None, 24, 96)    │         96 │ decoder_layernor… │
│ (AdditiveAttention) │                   │            │ encoder_layernor… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion              │ (None, 24, 209)   │          0 │ decoder_layernor… │
│ (Concatenate)       │                   │            │ additive_attenti… │
│                     │                   │            │ x_fut[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_dense64          │ (None, 24, 64)    │     13,440 │ fusion[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 24, 64)    │          0 │ td_dense64[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ td_dense32          │ (None, 24, 32)    │      2,080 │ dropout[0][0]     │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kDNI_sig            │ (None, 24, 1)     │         33 │ td_dense32[0][0]  │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ qDHI_sig            │ (None, 24, 1)     │         33 │ td_dense32[0][0]  │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kDNI_hat            │ (None, 24, 1)     │          0 │ kDNI_sig[0][0]    │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ qDHI_hat            │ (None, 24, 1)     │          0 │ qDHI_sig[0][0]    │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ model_output        │ (None, 24, 2)     │          0 │ kDNI_hat[0][0],   │
│ (Concatenate)       │                   │            │ qDHI_hat[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 110,146 (430.26 KB)

 Trainable params: 110,146 (430.26 KB)

 Non-trainable params: 0 (0.00 B)


=== AUDITORÍA BACKBONE V2.4 ===
Input histórico: (32, 72, 34)
Input futuro   : (32, 24, 17)
Output         : (32, 24, 2)

kDNI pred min/max: 1.4490324657829911e-25 0.7381315231323242
qDHI pred min/max: 0.0 1.2297993898391724
kDHI reconstruido min/max: 0.0 2.4205431938171387

Máx diferencia después de guardar/cargar: 0.0

✓ Arquitectura PI-LSTM V2.4 construida.
✓ Salida neuronal: [kDNI_hat, qDHI_hat]
✓ kDNI ∈ [0, 1.2 ]
✓ qDHI ∈ [0, 2.1972245773362196 ]
✓ kDHI reconstruido ∈ [0, 8.0 ]
✓ Modelo serializable sin Lambda.
✓ Celda 7 completada correctamente.


In [13]:
print("lambda_kdni :", cfg.lambda_kdni)
print("lambda_qdhi :", cfg.lambda_qdhi)
print("lambda_ghi  :", cfg.lambda_ghi)
print("lambda_ratio:", cfg.lambda_ratio)
print("lambda_energy:", cfg.lambda_energy)

lambda_kdni : 0.4
lambda_qdhi : 1.45
lambda_ghi  : 0.65
lambda_ratio: 1.9
lambda_energy: 0.27


In [14]:
# ============================================================
# 8. PI-LSTM V2.4 V1 — PARTITION-AWARE LOSS
#    FIEL A V2.3 + TARGET DIFUSO qDHI
# ============================================================
#
# PRINCIPIO DE LA ABLACIÓN
# ------------------------
#
# Se conserva la formulación física de V2.3:
#
#   - Huber delta = 0.10
#   - reduction = NONE
#   - sample_weight explícito
#   - ponderación solar:
#
#         w = sample_weight * (0.5 + w_day)
#
#   - reconstrucción física DNI/DHI/GHI
#   - término GHI normalizado
#   - término de razón difusa
#   - término agregado del horizonte
#   - mismos pesos lambda
#
# ÚNICO CAMBIO SUPERVISADO:
#
#       V2.3 : kDHI
#       V2.4 : qDHI = log(1 + kDHI)
#
# Para los términos físicos:
#
#       kDHI = exp(qDHI) - 1
#
# ============================================================


# ============================================================
# 1. ÍNDICES CANÓNICOS
# ============================================================

IDX_FUT = {
    c: i
    for i, c in enumerate(FUT_COLS)
}


IDX_YEXT = {
    c: i
    for i, c in enumerate(Y_EXT_COLS)
}


print(
    "=== ÍNDICES DE LA LOSS V2.4 V1 ==="
)

print(
    "FUT_COLS:",
    len(FUT_COLS)
)

print(
    "Y_EXT_COLS:",
    len(Y_EXT_COLS)
)

print(
    "IDX_YEXT:",
    IDX_YEXT
)


# ============================================================
# 2. VALIDAR CONTRATO DE Y_EXT
#
# V2.4 V1 debe tener:
#
#   0  kDNI
#   1  qDHI
#   2  GHI
#   3  Clearsky DNI
#   4  Clearsky DHI
#   5  Clearsky GHI
#   6  cosz_plus
#   7  DHI
# ============================================================

EXPECTED_Y_EXT_V24 = [

    "kDNI",
    "qDHI",

    COL_GHI,

    COL_CSDNI,
    COL_CSDHI,
    COL_CSGHI,

    "cosz_plus",

    COL_DHI,
]


assert list(
    Y_EXT_COLS
) == EXPECTED_Y_EXT_V24, (

    "Y_EXT_COLS no coincide con el contrato "
    "esperado para V2.4 V1.\n"
    f"Actual:   {Y_EXT_COLS}\n"
    f"Esperado: {EXPECTED_Y_EXT_V24}"
)


# ============================================================
# 3. PARTITION-AWARE LOSS V2.4 V1
#
# Se mantiene como tf.keras.layers.Layer,
# igual que en V2.3.
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="PILSTM"
)
class PartitionAwareLossV24V1(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        scaler_fut_mean,
        scaler_fut_scale,
        fut_cols_scaled,
        idx_fut,
        idx_yext,
        cfg,
        **kwargs
    ):

        super().__init__(
            name="partition_aware_loss_v24_v1",
            **kwargs
        )


        # ----------------------------------------------------
        # Información necesaria para desestandarizar x_fut
        # ----------------------------------------------------

        self.scaler_fut_mean = tf.constant(
            scaler_fut_mean,
            dtype=tf.float32
        )

        self.scaler_fut_scale = tf.constant(
            scaler_fut_scale,
            dtype=tf.float32
        )

        self.fut_cols_scaled = list(
            fut_cols_scaled
        )

        self.idx_fut = dict(
            idx_fut
        )

        self.idx_yext = dict(
            idx_yext
        )

        self.cfg = cfg


        # ----------------------------------------------------
        # HUBER CANÓNICO V2.3
        #
        # Guardamos delta explícitamente porque algunas
        # versiones de Keras 3 no lo exponen como atributo
        # ni dentro de get_config().
        # ----------------------------------------------------

        self.huber_delta = 0.10


        self.huber = tf.keras.losses.Huber(

            delta=self.huber_delta,

            reduction=tf.keras.losses.Reduction.NONE
        )


    # ========================================================
    # 4. DESESTANDARIZAR FEATURE FUTURA
    # ========================================================

    def _destandardize(
        self,
        x_fut,
        feat
    ):

        idx_full = self.idx_fut[
            feat
        ]

        idx_scaled = (
            self.fut_cols_scaled.index(
                feat
            )
        )


        return (

            x_fut[
                ...,
                idx_full
            ]

            *

            self.scaler_fut_scale[
                idx_scaled
            ]

            +

            self.scaler_fut_mean[
                idx_scaled
            ]
        )


    # ========================================================
    # 5. COMPONENTES DE LA LOSS
    #
    # Esta función devuelve también componentes individuales
    # para poder auditarlos durante entrenamiento.
    # ========================================================

    def components(
        self,
        x_batch,
        y_true,
        y_pred,
        sample_weight
    ):

        eps = tf.constant(
            1e-6,
            dtype=tf.float32
        )


        # ----------------------------------------------------
        # x_batch viene de Celda 6 como:
        #
        #     (x_hist, x_fut)
        # ----------------------------------------------------

        x_hist, x_fut = x_batch


        x_fut = tf.cast(
            x_fut,
            tf.float32
        )

        y_true = tf.cast(
            y_true,
            tf.float32
        )

        y_pred = tf.cast(
            y_pred,
            tf.float32
        )

        sample_weight = tf.cast(
            sample_weight,
            tf.float32
        )


        # ====================================================
        # 6. TARGETS REALES
        # ====================================================

        kDNI_true = y_true[
            ...,
            self.idx_yext[
                "kDNI"
            ]
        ]


        qDHI_true = y_true[
            ...,
            self.idx_yext[
                "qDHI"
            ]
        ]


        GHI_true = y_true[
            ...,
            self.idx_yext[
                COL_GHI
            ]
        ]


        DHI_true = y_true[
            ...,
            self.idx_yext[
                COL_DHI
            ]
        ]


        # ====================================================
        # 7. PREDICCIONES DE LA RED
        # ====================================================

        kDNI_pred = tf.clip_by_value(

            y_pred[
                ...,
                0
            ],

            0.0,

            self.cfg.kdni_max
        )


        qDHI_pred = tf.clip_by_value(

            y_pred[
                ...,
                1
            ],

            0.0,

            QDHI_MAX
        )


        # ====================================================
        # 8. RECONSTRUIR kDHI
        #
        # qDHI = log(1+kDHI)
        #
        # kDHI = exp(qDHI)-1
        # ====================================================

        kDHI_true = tf.math.expm1(
            qDHI_true
        )


        kDHI_pred = tf.math.expm1(
            qDHI_pred
        )


        # Safety cap V2.4:
        #
        # el límite 8.0 no censura el histórico observado,
        # cuyo máximo auditado es < 8.

        kDHI_true = tf.clip_by_value(

            kDHI_true,

            0.0,

            self.cfg.kdhi_max
        )


        kDHI_pred = tf.clip_by_value(

            kDHI_pred,

            0.0,

            self.cfg.kdhi_max
        )


        # ====================================================
        # 9. VARIABLES FÍSICAS FUTURAS
        #
        # Igual que V2.3:
        #
        # cosz se recupera desde x_fut desestandarizado.
        #
        # Las clear-sky permanecen RAW dentro de FUT_COLS.
        # ====================================================

        cosz = tf.clip_by_value(

            self._destandardize(
                x_fut,
                "cosz_plus"
            ),

            0.0,

            1.0
        )


        csghi = tf.cast(

            x_fut[
                ...,
                self.idx_fut[
                    COL_CSGHI
                ]
            ],

            tf.float32
        )


        csdni = tf.cast(

            x_fut[
                ...,
                self.idx_fut[
                    COL_CSDNI
                ]
            ],

            tf.float32
        )


        csdhi = tf.cast(

            x_fut[
                ...,
                self.idx_fut[
                    COL_CSDHI
                ]
            ],

            tf.float32
        )


        # ====================================================
        # 10. RECONSTRUCCIÓN FÍSICA
        # ====================================================

        DNI_pred = (

            kDNI_pred
            *
            csdni
        )


        DHI_pred = (

            kDHI_pred
            *
            csdhi
        )


        GHI_pred = (

            DNI_pred
            *
            cosz

            +

            DHI_pred
        )


        # ====================================================
        # 11. PESO SOLAR / TEMPORAL
        #
        # EXACTAMENTE LA FORMA V2.3:
        #
        # w_day =
        #   I(cosz > threshold)
        #   * sqrt(max(cosz,0))
        #
        # w =
        #   sample_weight
        #   * (0.5 + w_day)
        # ====================================================

        w_day = (

            tf.cast(

                cosz
                >
                self.cfg.thr_day_cosz,

                tf.float32
            )

            *

            tf.sqrt(

                tf.maximum(
                    cosz,
                    0.0
                )
            )
        )


        w = (

            sample_weight

            *

            (
                0.5
                +
                w_day
            )
        )


        # ====================================================
        # 12. TÉRMINO kDNI
        #
        # Igual que V2.3.
        # ====================================================

        l_kdni = self.huber(

            w
            *
            kDNI_true,

            w
            *
            kDNI_pred
        )


        # ====================================================
        # 13. TÉRMINO qDHI
        #
        # ÚNICO CAMBIO SUPERVISADO RESPECTO DE V2.3.
        #
        # V2.3:
        #   Huber(w*kDHI_true, w*kDHI_pred)
        #
        # V2.4 V1:
        #   Huber(w*qDHI_true, w*qDHI_pred)
        # ====================================================

        l_qdhi = self.huber(

            w
            *
            qDHI_true,

            w
            *
            qDHI_pred
        )


        # ====================================================
        # 14. TÉRMINO GHI NORMALIZADO
        #
        # Igual que V2.3.
        # ====================================================

        csghi_safe = tf.maximum(
            csghi,
            50.0
        )


        GHI_true_n = (

            GHI_true
            /
            csghi_safe
        )


        GHI_pred_n = (

            GHI_pred
            /
            csghi_safe
        )


        l_ghi = self.huber(

            w
            *
            GHI_true_n,

            w
            *
            GHI_pred_n
        )


        # ====================================================
        # 15. TÉRMINO DE PARTICIÓN DIFUSA
        #
        # Igual que V2.3.
        # ====================================================

        ratio_true = tf.clip_by_value(

            DHI_true
            /
            (
                GHI_true
                +
                eps
            ),

            0.0,

            1.2
        )


        ratio_pred = tf.clip_by_value(

            DHI_pred
            /
            (
                GHI_pred
                +
                eps
            ),

            0.0,

            1.2
        )


        ratio_boost = (

            1.0

            +

            0.70

            *

            tf.cast(

                ratio_true
                >
                0.45,

                tf.float32
            )
        )


        l_ratio = self.huber(

            w
            *
            ratio_boost
            *
            ratio_true,

            w
            *
            ratio_boost
            *
            ratio_pred
        )


        # ====================================================
        # 16. CONSISTENCIA AGREGADA DEL HORIZONTE
        #
        # Igual que V2.3.
        #
        # IMPORTANTE:
        # no representa energía física Wh/m².
        #
        # Es consistencia agregada de GHI normalizado
        # durante el horizonte H.
        # ====================================================

        denom = (

            tf.reduce_sum(
                w,
                axis=1
            )

            +

            eps
        )


        e_true = (

            tf.reduce_sum(

                w
                *
                GHI_true_n,

                axis=1
            )

            /

            denom
        )


        e_pred = (

            tf.reduce_sum(

                w
                *
                GHI_pred_n,

                axis=1
            )

            /

            denom
        )


        l_aggregate = tf.abs(

            e_true
            -
            e_pred
        )


        # ====================================================
        # 17. REDUCIR COMPONENTES
        # ====================================================

        loss_kdni = tf.reduce_mean(
            l_kdni
        )


        loss_qdhi = tf.reduce_mean(
            l_qdhi
        )


        loss_ghi = tf.reduce_mean(
            l_ghi
        )


        loss_ratio = tf.reduce_mean(
            l_ratio
        )


        loss_aggregate = tf.reduce_mean(
            l_aggregate
        )


        # ====================================================
        # 18. LOSS TOTAL
        #
        # Se mantienen exactamente los lambda de V2.3.
        #
        # lambda_kdhi pasa a ponderar qDHI.
        # ====================================================

        total = (

            self.cfg.lambda_kdni
            *
            loss_kdni

            +

            self.cfg.lambda_qdhi
            *
            loss_qdhi

            +

            self.cfg.lambda_ghi
            *
            loss_ghi

            +

            self.cfg.lambda_ratio
            *
            loss_ratio

            +

            self.cfg.lambda_energy
            *
            loss_aggregate
        )


        # ====================================================
        # 19. MÉTRICAS AUXILIARES
        # ====================================================

        diagnostics = {

            "total":
                total,

            "kdni":
                loss_kdni,

            "qdhi":
                loss_qdhi,

            "ghi":
                loss_ghi,

            "ratio":
                loss_ratio,

            "aggregate":
                loss_aggregate,

            "ratio_true_mean":
                tf.reduce_mean(
                    ratio_true
                ),

            "ratio_pred_mean":
                tf.reduce_mean(
                    ratio_pred
                ),

            "kdhi_true_mean":
                tf.reduce_mean(
                    kDHI_true
                ),

            "kdhi_pred_mean":
                tf.reduce_mean(
                    kDHI_pred
                ),

            "qdhi_true_mean":
                tf.reduce_mean(
                    qDHI_true
                ),

            "qdhi_pred_mean":
                tf.reduce_mean(
                    qDHI_pred
                ),

            "weight_mean":
                tf.reduce_mean(
                    w
                ),

            "weight_min":
                tf.reduce_min(
                    w
                ),

            "weight_max":
                tf.reduce_max(
                    w
                ),
        }


        return diagnostics


    # ========================================================
    # 20. CALL
    # ========================================================

    def call(
        self,
        x_batch,
        y_true,
        y_pred,
        sample_weight
    ):

        diagnostics = self.components(

            x_batch,
            y_true,
            y_pred,
            sample_weight
        )


        return diagnostics[
            "total"
        ]


# ============================================================
# 21. INSTANCIAR LOSS
# ============================================================

loss_layer_v24_v1 = PartitionAwareLossV24V1(

    scaler_fut_mean=(
        scaler_fut.mean_
        .astype(
            np.float32
        )
    ),

    scaler_fut_scale=(
        scaler_fut.scale_
        .astype(
            np.float32
        )
    ),

    fut_cols_scaled=FUT_COLS_SCALED,

    idx_fut=IDX_FUT,

    idx_yext=IDX_YEXT,

    cfg=cfg
)


# ============================================================
# 22. CONFIGURACIÓN DECLARADA
#
# NO intentamos leer huber.delta:
# Keras 3 no lo expone de manera estable.
# ============================================================

print(
    "\n=== PARTITION-AWARE LOSS V2.4 V1 ==="
)

print(
    "Clase:",
    type(
        loss_layer_v24_v1
    ).__name__
)

print(
    "Huber delta declarado:",
    loss_layer_v24_v1.huber_delta
)

print(
    "Huber reduction:",
    loss_layer_v24_v1.huber.reduction
)


print(
    "\nPesos lambda:"
)

print(
    "lambda_kdni :",
    cfg.lambda_kdni
)

print(
    "lambda_qdhi :",
    cfg.lambda_qdhi
)

print(
    "lambda_ghi  :",
    cfg.lambda_ghi
)

print(
    "lambda_ratio:",
    cfg.lambda_ratio
)

print(
    "lambda_agg  :",
    cfg.lambda_energy
)


# ============================================================
# 23. TEST NUMÉRICO INDEPENDIENTE DEL HUBER
#
# Este test comprueba que el objeto Huber construido con
# delta=0.10 produce exactamente la expresión matemática
# correspondiente, sin depender de atributos internos.
# ============================================================

huber_test_true = tf.constant(

    [
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
    ],

    dtype=tf.float32
)


huber_test_pred = tf.constant(

    [
        [0.05, 0.10, 0.20, -0.30],
        [0.02, -0.15, 0.40, -0.08],
    ],

    dtype=tf.float32
)


huber_tf_result = (
    loss_layer_v24_v1.huber(

        huber_test_true,

        huber_test_pred
    )
)


huber_error = (

    huber_test_true
    -
    huber_test_pred
)


huber_abs_error = tf.abs(
    huber_error
)


delta_check = tf.constant(

    loss_layer_v24_v1.huber_delta,

    dtype=tf.float32
)


huber_manual_elementwise = tf.where(

    huber_abs_error
    <=
    delta_check,

    0.5
    *
    tf.square(
        huber_error
    ),

    delta_check
    *
    (
        huber_abs_error
        -
        0.5
        *
        delta_check
    )
)


# tf.keras.losses.Huber reduce la última dimensión
# aun cuando reduction=NONE.

huber_manual_result = tf.reduce_mean(

    huber_manual_elementwise,

    axis=-1
)


huber_numeric_difference = float(

    tf.reduce_max(

        tf.abs(

            huber_tf_result
            -
            huber_manual_result
        )
    ).numpy()
)


print(
    "\n=== AUDITORÍA NUMÉRICA HUBER ==="
)

print(
    "Resultado TF:",
    huber_tf_result.numpy()
)

print(
    "Resultado manual:",
    huber_manual_result.numpy()
)

print(
    "Máxima diferencia:",
    huber_numeric_difference
)


assert np.isclose(

    loss_layer_v24_v1.huber_delta,

    0.10
)


assert huber_numeric_difference < 1e-7


# ============================================================
# 24. TEST SOBRE UN BATCH REAL
#
# batch_xh, batch_xf, batch_y, batch_sw
# provienen de la Celda 6 V1.
# ============================================================

pred_batch_v1 = backbone(

    [
        batch_xh,
        batch_xf
    ],

    training=False
)


components_v1 = (

    loss_layer_v24_v1.components(

        (
            batch_xh,
            batch_xf
        ),

        batch_y,

        pred_batch_v1,

        batch_sw
    )
)


print(
    "\n=== LOSS V2.4 V1 — BATCH REAL ==="
)


for key in [

    "total",

    "kdni",
    "qdhi",
    "ghi",
    "ratio",
    "aggregate",

    "ratio_true_mean",
    "ratio_pred_mean",

    "kdhi_true_mean",
    "kdhi_pred_mean",

    "qdhi_true_mean",
    "qdhi_pred_mean",

    "weight_mean",
    "weight_min",
    "weight_max",

]:

    print(

        f"{key:20s}: "

        f"{float(components_v1[key].numpy()):.10f}"
    )


# ============================================================
# 25. VALIDAR RECONSTRUCCIÓN MANUAL DE LA LOSS TOTAL
# ============================================================

manual_total_v1 = (

    cfg.lambda_kdni
    *
    components_v1[
        "kdni"
    ]

    +

    cfg.lambda_qdhi
    *
    components_v1[
        "qdhi"
    ]

    +

    cfg.lambda_ghi
    *
    components_v1[
        "ghi"
    ]

    +

    cfg.lambda_ratio
    *
    components_v1[
        "ratio"
    ]

    +

    cfg.lambda_energy
    *
    components_v1[
        "aggregate"
    ]
)


total_difference_v1 = float(

    tf.abs(

        manual_total_v1

        -

        components_v1[
            "total"
        ]
    ).numpy()
)


print(
    "\n=== VALIDACIÓN DE LOSS TOTAL ==="
)

print(
    "Total loss:",
    float(
        components_v1[
            "total"
        ].numpy()
    )
)

print(
    "Total reconstruido:",
    float(
        manual_total_v1.numpy()
    )
)

print(
    "Diferencia:",
    total_difference_v1
)


assert total_difference_v1 < 1e-7


# ============================================================
# 26. VALIDAR PESO SOLAR
#
# sample_weight = 1.
#
# Por tanto:
#
#   noche / bajo sol:
#       w = 0.5
#
#   día:
#       w = 0.5 + sqrt(cosz)
#
# Deberíamos tener aproximadamente:
#
#       0.5 <= w <= 1.5
# ============================================================

weight_min_v1 = float(

    components_v1[
        "weight_min"
    ].numpy()
)


weight_max_v1 = float(

    components_v1[
        "weight_max"
    ].numpy()
)


assert (
    weight_min_v1
    >=
    0.5 - 1e-6
)


assert (
    weight_max_v1
    <=
    1.5 + 1e-6
)


# ============================================================
# 27. VALIDAR GRADIENTES
# ============================================================

with tf.GradientTape() as tape:

    pred_grad_v1 = backbone(

        [
            batch_xh,
            batch_xf
        ],

        training=True
    )


    components_grad_v1 = (

        loss_layer_v24_v1.components(

            (
                batch_xh,
                batch_xf
            ),

            batch_y,

            pred_grad_v1,

            batch_sw
        )
    )


    total_grad_v1 = (

        components_grad_v1[
            "total"
        ]
    )


gradients_v1 = tape.gradient(

    total_grad_v1,

    backbone.trainable_variables
)


n_trainable_v1 = len(

    backbone.trainable_variables
)


n_gradients_v1 = len(
    gradients_v1
)


n_none_gradients_v1 = sum(

    g is None

    for g in gradients_v1
)


n_nonfinite_gradients_v1 = 0


for g in gradients_v1:

    if g is None:

        continue


    finite_g = bool(

        tf.reduce_all(

            tf.math.is_finite(
                g
            )
        ).numpy()
    )


    if not finite_g:

        n_nonfinite_gradients_v1 += 1


print(
    "\n=== GRADIENTES ==="
)

print(
    "Tensores entrenables:",
    n_trainable_v1
)

print(
    "Gradientes obtenidos:",
    n_gradients_v1
)

print(
    "Gradientes None:",
    n_none_gradients_v1
)

print(
    "Gradientes no finitos:",
    n_nonfinite_gradients_v1
)


assert (
    n_gradients_v1
    ==
    n_trainable_v1
)


assert (
    n_none_gradients_v1
    ==
    0
)


assert (
    n_nonfinite_gradients_v1
    ==
    0
)


# ============================================================
# 28. VALIDACIONES DE DOMINIO DE LAS PREDICCIONES
# ============================================================

pred_batch_np_v1 = (
    pred_batch_v1.numpy()
)


kdni_pred_min_v1 = float(

    np.min(
        pred_batch_np_v1[
            ...,
            0
        ]
    )
)


kdni_pred_max_v1 = float(

    np.max(
        pred_batch_np_v1[
            ...,
            0
        ]
    )
)


qdhi_pred_min_v1 = float(

    np.min(
        pred_batch_np_v1[
            ...,
            1
        ]
    )
)


qdhi_pred_max_v1 = float(

    np.max(
        pred_batch_np_v1[
            ...,
            1
        ]
    )
)


print(
    "\n=== DOMINIO DE SALIDAS ==="
)

print(
    "kDNI pred min/max:",
    kdni_pred_min_v1,
    kdni_pred_max_v1
)

print(
    "qDHI pred min/max:",
    qdhi_pred_min_v1,
    qdhi_pred_max_v1
)

print(
    "QDHI_MAX:",
    QDHI_MAX
)


assert (
    kdni_pred_min_v1
    >=
    -1e-6
)


assert (
    kdni_pred_max_v1
    <=
    cfg.kdni_max + 1e-6
)


assert (
    qdhi_pred_min_v1
    >=
    -1e-6
)


assert (
    qdhi_pred_max_v1
    <=
    QDHI_MAX + 1e-6
)


# ============================================================
# 29. RESUMEN FINAL
# ============================================================

print(
    "\n✓ PartitionAwareLoss implementada como Layer, igual que V2.3."
)

print(
    "✓ Huber delta=0.10 validado numéricamente."
)

print(
    "✓ Huber reduction=NONE utilizado."
)

print(
    "✓ sample_weight explícito recibido desde Celda 6."
)

print(
    "✓ Ponderación solar V2.3 restaurada."
)

print(
    "✓ Clear-sky y cosz obtenidos desde x_fut."
)

print(
    "✓ Término kDNI preservado."
)

print(
    "✓ Término supervisado kDHI sustituido únicamente por qDHI."
)

print(
    "✓ kDHI reconstruido con expm1(qDHI) para términos físicos."
)

print(
    "✓ Término GHI preservado."
)

print(
    "✓ Término de partición difusa preservado."
)

print(
    "✓ Consistencia agregada del horizonte preservada."
)

print(
    "✓ Loss total reconstruida exactamente."
)

print(
    "✓ Gradientes completos y finitos."
)

print(
    "✓ Celda 8 V1 completada correctamente."
)

=== ÍNDICES DE LA LOSS V2.4 V1 ===
FUT_COLS: 17
Y_EXT_COLS: 8
IDX_YEXT: {'kDNI': 0, 'qDHI': 1, 'GHI': 2, 'Clearsky DNI': 3, 'Clearsky DHI': 4, 'Clearsky GHI': 5, 'cosz_plus': 6, 'DHI': 7}

=== PARTITION-AWARE LOSS V2.4 V1 ===
Clase: PartitionAwareLossV24V1
Huber delta declarado: 0.1
Huber reduction: none

Pesos lambda:
lambda_kdni : 0.4
lambda_qdhi : 1.45
lambda_ghi  : 0.65
lambda_ratio: 1.9
lambda_agg  : 0.27

=== AUDITORÍA NUMÉRICA HUBER ===
Resultado TF: [0.0115625 0.0121   ]
Resultado manual: [0.0115625 0.0121   ]
Máxima diferencia: 9.313225746154785e-10

=== LOSS V2.4 V1 — BATCH REAL ===
total               : 0.4724702835
kdni                : 0.0445300639
qdhi                : 0.0979437158
ghi                 : 0.0551349781
ratio               : 0.0681582764
aggregate           : 0.5455607772
ratio_true_mean     : 0.3626850545
ratio_pred_mean     : 0.0005996585
kdhi_true_mean      : 1.3632732630
kdhi_pred_mean      : 0.5222486854
qdhi_true_mean      : 0.6544001102
qdhi_pred_mean 

In [15]:
# ============================================================
# 9. PI-LSTM V2.4 V1 — CUSTOM TRAINER
#    CON SAMPLE_WEIGHT EXPLÍCITO
# ============================================================
#
# CONTRATO DEL DATASET — CELDA 6 V1
# ----------------------------------
#
# Cada batch tiene la forma:
#
#   (
#       (x_hist, x_fut),
#       y_ext,
#       sample_weight
#   )
#
# donde:
#
#   x_hist        : (B, L, 34)
#   x_fut         : (B, H, 17)
#   y_ext         : (B, H, 8)
#   sample_weight : (B, H)
#
#
# CONTRATO DE LA LOSS — CELDA 8 V1
# ---------------------------------
#
# loss_layer_v24_v1.components(
#     x_batch,
#     y_true,
#     y_pred,
#     sample_weight
# )
#
#
# OBJETIVO
# --------
#
# Mantener la lógica del entrenamiento V2.3:
#
#   forward
#   -> partition-aware loss
#   -> gradient tape
#   -> Adam
#
# sin modificar la arquitectura del backbone.
# ============================================================


# ============================================================
# 1. CUSTOM TRAINER
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="PILSTM"
)
class PILSTMTrainerV24V1(
    tf.keras.Model
):

    def __init__(
        self,
        backbone,
        loss_layer,
        **kwargs
    ):

        super().__init__(
            name="pi_lstm_trainer_v24_v1",
            **kwargs
        )


        self.backbone = backbone

        self.loss_layer = loss_layer


        # ====================================================
        # TRACKERS DE LOSS
        # ====================================================

        self.loss_tracker = (
            tf.keras.metrics.Mean(
                name="loss"
            )
        )


        self.kdni_tracker = (
            tf.keras.metrics.Mean(
                name="kdni_loss"
            )
        )


        self.qdhi_tracker = (
            tf.keras.metrics.Mean(
                name="qdhi_loss"
            )
        )


        self.ghi_tracker = (
            tf.keras.metrics.Mean(
                name="ghi_loss"
            )
        )


        self.ratio_tracker = (
            tf.keras.metrics.Mean(
                name="ratio_loss"
            )
        )


        self.aggregate_tracker = (
            tf.keras.metrics.Mean(
                name="aggregate_loss"
            )
        )


        # ====================================================
        # TRACKERS FÍSICOS / DIAGNÓSTICOS
        # ====================================================

        self.ratio_true_tracker = (
            tf.keras.metrics.Mean(
                name="ratio_true_mean"
            )
        )


        self.ratio_pred_tracker = (
            tf.keras.metrics.Mean(
                name="ratio_pred_mean"
            )
        )


        self.kdhi_true_tracker = (
            tf.keras.metrics.Mean(
                name="kdhi_true_mean"
            )
        )


        self.kdhi_pred_tracker = (
            tf.keras.metrics.Mean(
                name="kdhi_pred_mean"
            )
        )


        self.qdhi_true_tracker = (
            tf.keras.metrics.Mean(
                name="qdhi_true_mean"
            )
        )


        self.qdhi_pred_tracker = (
            tf.keras.metrics.Mean(
                name="qdhi_pred_mean"
            )
        )


        self.weight_mean_tracker = (
            tf.keras.metrics.Mean(
                name="weight_mean"
            )
        )


    # ========================================================
    # 2. MÉTRICAS QUE KERAS DEBE REINICIAR CADA ÉPOCA
    # ========================================================

    @property
    def metrics(self):

        return [

            self.loss_tracker,

            self.kdni_tracker,
            self.qdhi_tracker,

            self.ghi_tracker,
            self.ratio_tracker,
            self.aggregate_tracker,

            self.ratio_true_tracker,
            self.ratio_pred_tracker,

            self.kdhi_true_tracker,
            self.kdhi_pred_tracker,

            self.qdhi_true_tracker,
            self.qdhi_pred_tracker,

            self.weight_mean_tracker,
        ]


    # ========================================================
    # 3. FORWARD
    #
    # El Trainer actúa como wrapper del backbone.
    # ========================================================

    def call(
        self,
        inputs,
        training=False
    ):

        return self.backbone(
            inputs,
            training=training
        )


    # ========================================================
    # 4. ACTUALIZAR TRACKERS
    # ========================================================

    def _update_trackers(
        self,
        comps
    ):

        self.loss_tracker.update_state(
            comps[
                "total"
            ]
        )


        self.kdni_tracker.update_state(
            comps[
                "kdni"
            ]
        )


        self.qdhi_tracker.update_state(
            comps[
                "qdhi"
            ]
        )


        self.ghi_tracker.update_state(
            comps[
                "ghi"
            ]
        )


        self.ratio_tracker.update_state(
            comps[
                "ratio"
            ]
        )


        self.aggregate_tracker.update_state(
            comps[
                "aggregate"
            ]
        )


        self.ratio_true_tracker.update_state(
            comps[
                "ratio_true_mean"
            ]
        )


        self.ratio_pred_tracker.update_state(
            comps[
                "ratio_pred_mean"
            ]
        )


        self.kdhi_true_tracker.update_state(
            comps[
                "kdhi_true_mean"
            ]
        )


        self.kdhi_pred_tracker.update_state(
            comps[
                "kdhi_pred_mean"
            ]
        )


        self.qdhi_true_tracker.update_state(
            comps[
                "qdhi_true_mean"
            ]
        )


        self.qdhi_pred_tracker.update_state(
            comps[
                "qdhi_pred_mean"
            ]
        )


        self.weight_mean_tracker.update_state(
            comps[
                "weight_mean"
            ]
        )


    # ========================================================
    # 5. RESULTADOS DE TRACKERS
    # ========================================================

    def _metric_results(
        self
    ):

        return {

            m.name:
                m.result()

            for m in self.metrics
        }


    # ========================================================
    # 6. TRAIN STEP
    # ========================================================

    def train_step(
        self,
        data
    ):

        # ----------------------------------------------------
        # CONTRATO DATASET V1
        #
        # data =
        # (
        #     (x_hist, x_fut),
        #     y_true,
        #     sample_weight
        # )
        # ----------------------------------------------------

        (
            x_batch,
            y_true,
            sample_weight
        ) = data


        # ----------------------------------------------------
        # Validación conceptual:
        #
        # x_batch es una tupla:
        #
        #     (x_hist, x_fut)
        # ----------------------------------------------------

        with tf.GradientTape() as tape:

            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            y_pred = self.backbone(

                x_batch,

                training=True
            )


            # ------------------------------------------------
            # Loss física V2.4 V1
            # ------------------------------------------------

            comps = self.loss_layer.components(

                x_batch,

                y_true,

                y_pred,

                sample_weight
            )


            total_loss = comps[
                "total"
            ]


        # ----------------------------------------------------
        # Gradientes
        # ----------------------------------------------------

        trainable_vars = (
            self.backbone.trainable_variables
        )


        gradients = tape.gradient(

            total_loss,

            trainable_vars
        )


        # ----------------------------------------------------
        # Protección contra gradientes None
        # ----------------------------------------------------

        valid_pairs = [

            (
                grad,
                var
            )

            for grad, var
            in zip(
                gradients,
                trainable_vars
            )

            if grad is not None
        ]


        if len(
            valid_pairs
        ) != len(
            trainable_vars
        ):

            raise RuntimeError(

                "Se detectaron gradientes None "
                "durante train_step."
            )


        # ----------------------------------------------------
        # Protección contra NaN / Inf
        # ----------------------------------------------------

        for grad, var in valid_pairs:

            tf.debugging.assert_all_finite(

                grad,

                message=(
                    "Gradiente no finito en "
                    f"{var.name}"
                )
            )


        # ----------------------------------------------------
        # Adam update
        # ----------------------------------------------------

        self.optimizer.apply_gradients(
            valid_pairs
        )


        # ----------------------------------------------------
        # Actualizar trackers
        # ----------------------------------------------------

        self._update_trackers(
            comps
        )


        return self._metric_results()


    # ========================================================
    # 7. TEST / VALIDATION STEP
    # ========================================================

    def test_step(
        self,
        data
    ):

        (
            x_batch,
            y_true,
            sample_weight
        ) = data


        # ----------------------------------------------------
        # Forward sin dropout
        # ----------------------------------------------------

        y_pred = self.backbone(

            x_batch,

            training=False
        )


        # ----------------------------------------------------
        # Loss física
        # ----------------------------------------------------

        comps = self.loss_layer.components(

            x_batch,

            y_true,

            y_pred,

            sample_weight
        )


        # ----------------------------------------------------
        # Actualizar trackers
        # ----------------------------------------------------

        self._update_trackers(
            comps
        )


        return self._metric_results()


# ============================================================
# 8. INSTANCIAR TRAINER
# ============================================================

trainer_v24_v1 = PILSTMTrainerV24V1(

    backbone=backbone,

    loss_layer=loss_layer_v24_v1
)


# ============================================================
# 9. OPTIMIZADOR
#
# Se conserva Adam y el learning rate de la configuración.
# ============================================================

optimizer_v24_v1 = (
    tf.keras.optimizers.Adam(

        learning_rate=cfg.lr
    )
)


# ============================================================
# 10. COMPILE
#
# La loss NO se pasa aquí:
#
# se calcula manualmente dentro de train_step/test_step.
# ============================================================

trainer_v24_v1.compile(

    optimizer=optimizer_v24_v1,

    run_eagerly=False
)


# ============================================================
# 11. CONSTRUIR EL TRAINER
#
# Ejecutamos un forward para registrar shapes.
# No modifica pesos.
# ============================================================

trainer_build_pred = trainer_v24_v1(

    (
        batch_xh,
        batch_xf
    ),

    training=False
)


print(
    "=== TRAINER V2.4 V1 ==="
)

print(
    "Clase:",
    type(
        trainer_v24_v1
    ).__name__
)

print(
    "Backbone:",
    trainer_v24_v1.backbone.name
)

print(
    "Optimizer:",
    type(
        trainer_v24_v1.optimizer
    ).__name__
)

print(
    "Learning rate:",
    float(
        tf.keras.backend.get_value(
            trainer_v24_v1.optimizer.learning_rate
        )
    )
)


print(
    "\n=== SHAPES ==="
)

print(
    "Input hist:",
    batch_xh.shape
)

print(
    "Input fut :",
    batch_xf.shape
)

print(
    "Output    :",
    trainer_build_pred.shape
)


# ============================================================
# 12. PARÁMETROS
# ============================================================

print(
    "\n=== PARÁMETROS ==="
)

print(
    "Backbone params:",
    f"{backbone.count_params():,}"
)

print(
    "Trainer params :",
    f"{trainer_v24_v1.count_params():,}"
)

print(
    "Trainable tensors:",
    len(
        backbone.trainable_variables
    )
)


assert (
    backbone.count_params()
    ==
    110146
)


assert (
    len(
        backbone.trainable_variables
    )
    ==
    19
)


# ============================================================
# 13. SMOKE TEST DE VALIDATION / TEST_STEP
#
# IMPORTANTE:
#
# No ejecutamos train_step aquí porque modificaría los pesos
# del backbone antes del entrenamiento real.
#
# test_step NO actualiza pesos.
# ============================================================

trainer_v24_v1.reset_metrics()


test_step_result_v1 = (
    trainer_v24_v1.test_step(

        (
            (
                batch_xh,
                batch_xf
            ),

            batch_y,

            batch_sw
        )
    )
)


print(
    "\n=== SMOKE TEST — TEST_STEP ==="
)


for key, value in (
    test_step_result_v1.items()
):

    print(

        f"{key:20s}: "

        f"{float(value.numpy()):.10f}"
    )


# ============================================================
# 14. VALIDAR QUE TEST_STEP NO MODIFICÓ LOS PESOS
#
# Guardamos una copia antes y después de un segundo test_step.
# ============================================================

weights_before_test = [

    w.numpy().copy()

    for w in backbone.weights
]


_ = trainer_v24_v1.test_step(

    (
        (
            batch_xh,
            batch_xf
        ),

        batch_y,

        batch_sw
    )
)


weights_after_test = [

    w.numpy().copy()

    for w in backbone.weights
]


max_weight_change_test = 0.0


for w_before, w_after in zip(

    weights_before_test,

    weights_after_test
):

    diff = float(

        np.max(

            np.abs(

                w_after
                -
                w_before
            )
        )
    )


    max_weight_change_test = max(

        max_weight_change_test,

        diff
    )


print(
    "\nMáximo cambio de pesos tras test_step:",
    max_weight_change_test
)


assert (
    max_weight_change_test
    ==
    0.0
)


# ============================================================
# 15. VALIDAR MÉTRICAS REGISTRADAS
# ============================================================

metric_names_v1 = [

    m.name

    for m in trainer_v24_v1.metrics
]


expected_metric_names_v1 = [

    "loss",

    "kdni_loss",
    "qdhi_loss",

    "ghi_loss",
    "ratio_loss",
    "aggregate_loss",

    "ratio_true_mean",
    "ratio_pred_mean",

    "kdhi_true_mean",
    "kdhi_pred_mean",

    "qdhi_true_mean",
    "qdhi_pred_mean",

    "weight_mean",
]


print(
    "\n=== MÉTRICAS REGISTRADAS ==="
)


for name in metric_names_v1:

    print(
        name
    )


assert (
    metric_names_v1
    ==
    expected_metric_names_v1
)


# ============================================================
# 16. VALIDAR ESTRUCTURA DEL DATASET
# ============================================================

dataset_sample_v1 = next(
    iter(
        ds_train
    )
)


(
    (
        ds_xh,
        ds_xf
    ),
    ds_y,
    ds_sw
) = dataset_sample_v1


print(
    "\n=== CONTRATO DATASET → TRAINER ==="
)

print(
    "x_hist        :",
    ds_xh.shape
)

print(
    "x_fut         :",
    ds_xf.shape
)

print(
    "y_ext         :",
    ds_y.shape
)

print(
    "sample_weight :",
    ds_sw.shape
)


assert (
    ds_xh.shape[1:]
    ==
    (
        cfg.L,
        len(HIST_COLS)
    )
)


assert (
    ds_xf.shape[1:]
    ==
    (
        cfg.H,
        len(FUT_COLS)
    )
)


assert (
    ds_y.shape[1:]
    ==
    (
        cfg.H,
        len(Y_EXT_COLS)
    )
)


assert (
    ds_sw.shape[1:]
    ==
    (
        cfg.H,
    )
)


# ============================================================
# 17. VALIDAR FINITUD DEL SMOKE TEST
# ============================================================

for key, value in (
    test_step_result_v1.items()
):

    if not bool(

        tf.reduce_all(

            tf.math.is_finite(
                value
            )
        ).numpy()
    ):

        raise ValueError(

            f"Métrica no finita: {key}"
        )


# ============================================================
# 18. VALIDAR OPTIMIZADOR
# ============================================================

current_lr_v1 = float(

    tf.keras.backend.get_value(

        trainer_v24_v1.optimizer.learning_rate
    )
)


assert np.isclose(

    current_lr_v1,

    cfg.lr,

    rtol=1e-6
)


# ============================================================
# 19. LIMPIAR TRACKERS
#
# Muy importante antes de iniciar fit():
#
# el smoke test anterior no debe contaminar las métricas
# de la primera época.
# ============================================================

trainer_v24_v1.reset_metrics()


for metric in trainer_v24_v1.metrics:

    assert np.isclose(

        float(
            metric.result().numpy()
        ),

        0.0
    )


# ============================================================
# 20. RESUMEN FINAL
# ============================================================

print(
    "\n✓ Trainer V2.4 V1 construido correctamente."
)

print(
    "✓ Contrato dataset con sample_weight validado."
)

print(
    "✓ PartitionAwareLoss V1 conectada correctamente."
)

print(
    "✓ Adam configurado con learning rate esperado."
)

print(
    "✓ Backbone conserva 110,146 parámetros."
)

print(
    "✓ Backbone conserva 19 tensores entrenables."
)

print(
    "✓ test_step ejecutado correctamente."
)

print(
    "✓ test_step no modifica pesos."
)

print(
    "✓ Métricas físicas registradas."
)

print(
    "✓ Trackers reiniciados antes del entrenamiento."
)

print(
    "✓ Celda 9 V1 completada correctamente."
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'pi_lstm_trainer_v24_v1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


=== TRAINER V2.4 V1 ===
Clase: PILSTMTrainerV24V1
Backbone: PI_LSTM_V2_4
Optimizer: Adam
Learning rate: 0.0003000000142492354

=== SHAPES ===
Input hist: (32, 72, 34)
Input fut : (32, 24, 17)
Output    : (32, 24, 2)

=== PARÁMETROS ===
Backbone params: 110,146
Trainer params : 110,146
Trainable tensors: 19

=== SMOKE TEST — TEST_STEP ===
loss                : 0.4724702835
kdni_loss           : 0.0445300639
qdhi_loss           : 0.0979437158
ghi_loss            : 0.0551349781
ratio_loss          : 0.0681582764
aggregate_loss      : 0.5455607772
ratio_true_mean     : 0.3626850545
ratio_pred_mean     : 0.0005996585
kdhi_true_mean      : 1.3632732630
kdhi_pred_mean      : 0.5222486854
qdhi_true_mean      : 0.6544001102
qdhi_pred_mean      : 0.3248815835
weight_mean         : 0.9802716374

Máximo cambio de pesos tras test_step: 0.0

=== MÉTRICAS REGISTRADAS ===
loss
kdni_loss
qdhi_loss
ghi_loss
ratio_loss
aggregate_loss
ratio_true_mean
ratio_pred_mean
kdhi_true_mean
kdhi_pred_mean
qdhi_true

In [ ]:
# ============================================================
# 10. PI-LSTM V2.4 V1 — ENTRENAMIENTO CANÓNICO
# ============================================================
#
# OBJETIVO
# --------
#
# Entrenar desde la inicialización actual:
#
#   PI-LSTM V2.4 V1
#
# manteniendo:
#
#   L       = 72
#   H       = 24
#   stride  = 2
#   batch   = 32
#   lr      = 3e-4
#   epochs  = 25
#
#   EarlyStopping patience = 8
#   ReduceLROnPlateau      = 3
#
# La loss corresponde a la formulación V2.3 restaurada,
# con el único cambio supervisado:
#
#       kDHI -> qDHI = log(1+kDHI)
#
# ============================================================


# ============================================================
# 1. DIRECTORIO DE SALIDA EN GOOGLE COLAB
# ============================================================

OUTPUT_DIR_V1 = Path(
    "/content/model_v24_v1_outputs"
)


OUTPUT_DIR_V1.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "=== DIRECTORIO V2.4 V1 ==="
)

print(
    OUTPUT_DIR_V1.resolve()
)


# ============================================================
# 2. RUTAS DE ARTEFACTOS
# ============================================================

MODEL_PATH_V1 = (
    OUTPUT_DIR_V1
    /
    "PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras"
)


HISTORY_PATH_V1 = (
    OUTPUT_DIR_V1
    /
    "training_history_v24_v1.csv"
)


METADATA_PATH_V1 = (
    OUTPUT_DIR_V1
    /
    "model_metadata_v24_v1.json"
)


SCALER_HIST_PATH_V1 = (
    OUTPUT_DIR_V1
    /
    "scaler_hist_v24_v1.joblib"
)


SCALER_FUT_PATH_V1 = (
    OUTPUT_DIR_V1
    /
    "scaler_fut_v24_v1.joblib"
)


BEST_EPOCH_PATH_V1 = (
    OUTPUT_DIR_V1
    /
    "best_epoch_v24_v1.json"
)


# ============================================================
# 3. PROTECCIÓN CONTRA SOBRESCRITURA ACCIDENTAL
#
# Si vuelves a ejecutar deliberadamente el entrenamiento,
# elimina manualmente la carpeta o cambia el nombre.
# ============================================================

critical_existing_files = [

    MODEL_PATH_V1,
    HISTORY_PATH_V1,
]


existing_critical = [

    str(p)

    for p in critical_existing_files

    if p.exists()
]


if existing_critical:

    raise FileExistsError(

        "Ya existen resultados de un entrenamiento V1:\n\n"

        +

        "\n".join(
            existing_critical
        )

        +

        "\n\nPara evitar sobrescribir una corrida previa, "
        "elimina o renombra esos archivos antes de entrenar."
    )


# ============================================================
# 4. COMPROBACIONES PREVIAS
# ============================================================

print(
    "\n=== CONFIGURACIÓN DE ENTRENAMIENTO ==="
)

print(
    "Train windows :",
    f"{nwin_train:,}"
)

print(
    "Val windows   :",
    f"{nwin_val:,}"
)

print(
    "Train steps   :",
    steps_train
)

print(
    "Val steps     :",
    steps_val
)

print(
    "Batch size    :",
    cfg.batch_size
)

print(
    "Epochs máximos:",
    cfg.epochs
)

print(
    "Learning rate :",
    cfg.lr
)

print(
    "ES patience   :",
    cfg.patience_es
)

print(
    "RLR patience  :",
    cfg.patience_rlr
)

print(
    "RLR factor    :",
    cfg.reduce_lr_factor
)

print(
    "Minimum LR    :",
    cfg.min_lr
)


assert (
    steps_train
    ==
    6025
)


assert (
    steps_val
    ==
    821
)


assert (
    backbone.count_params()
    ==
    110146
)


assert (
    len(
        backbone.trainable_variables
    )
    ==
    19
)


# ============================================================
# 5. COMPROBAR QUE EL OPTIMIZADOR NO HA ENTRENADO TODAVÍA
# ============================================================

optimizer_iterations_before = int(

    trainer_v24_v1.optimizer
    .iterations
    .numpy()
)


print(
    "\nOptimizer iterations antes de reinicializar:",
    optimizer_iterations_before
)


# test_step no aumenta iterations.
#
# Si aquí apareciera > 0, significaría que se ejecutó
# previamente train_step o fit().

assert (
    optimizer_iterations_before
    ==
    0
), (
    "El optimizador actual ya realizó actualizaciones. "
    "No continuar sin revisar el estado del backbone."
)


# ============================================================
# 6. OPTIMIZADOR NUEVO PARA LA CORRIDA CANÓNICA
#
# Aunque el optimizador de Celda 9 está limpio,
# creamos uno nuevo para dejar explícito que esta corrida
# comienza con estado Adam virgen.
# ============================================================

optimizer_train_v1 = (
    tf.keras.optimizers.Adam(

        learning_rate=cfg.lr
    )
)


trainer_v24_v1.compile(

    optimizer=optimizer_train_v1,

    run_eagerly=False
)


trainer_v24_v1.reset_metrics()


assert (
    int(
        trainer_v24_v1.optimizer
        .iterations
        .numpy()
    )
    ==
    0
)


# ============================================================
# 7. REPRODUCIBILIDAD DE LA CORRIDA
#
# Esto controla dropout y demás operaciones aleatorias
# durante fit().
#
# No reinicializa el backbone.
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

tf.random.set_seed(
    SEED
)


print(
    "\nSeed entrenamiento:",
    SEED
)


# ============================================================
# 8. CALLBACK PARA GUARDAR EL MEJOR BACKBONE
#
# ModelCheckpoint aplicado directamente al Trainer podría
# guardar el wrapper personalizado.
#
# Para máxima portabilidad guardamos explícitamente
# únicamente el backbone.
# ============================================================

class BestBackboneSaver(
    tf.keras.callbacks.Callback
):

    def __init__(
        self,
        filepath,
        monitor="val_loss",
        mode="min",
        verbose=1
    ):

        super().__init__()


        self.filepath = Path(
            filepath
        )

        self.monitor = monitor

        self.mode = mode

        self.verbose = verbose


        if mode == "min":

            self.best = np.inf

            self.is_better = (
                lambda current, best:
                current < best
            )

        elif mode == "max":

            self.best = -np.inf

            self.is_better = (
                lambda current, best:
                current > best
            )

        else:

            raise ValueError(
                "mode debe ser 'min' o 'max'."
            )


        self.best_epoch = None


    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        logs = logs or {}


        current = logs.get(
            self.monitor
        )


        if current is None:

            return


        current = float(
            current
        )


        if self.is_better(
            current,
            self.best
        ):

            self.best = current

            self.best_epoch = (
                epoch + 1
            )


            self.model.backbone.save(
                self.filepath,
                overwrite=True
            )


            if self.verbose:

                print(

                    "\n✓ Mejor backbone actualizado: "
                    f"epoch {self.best_epoch} | "
                    f"{self.monitor}={self.best:.10f}"
                )


# ============================================================
# 9. CALLBACKS
# ============================================================

best_backbone_callback_v1 = (
    BestBackboneSaver(

        filepath=MODEL_PATH_V1,

        monitor="val_loss",

        mode="min",

        verbose=1
    )
)


early_stopping_v1 = (
    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        mode="min",

        patience=cfg.patience_es,

        restore_best_weights=True,

        verbose=1
    )
)


reduce_lr_v1 = (
    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_loss",

        mode="min",

        factor=cfg.reduce_lr_factor,

        patience=cfg.patience_rlr,

        min_lr=cfg.min_lr,

        verbose=1
    )
)


csv_logger_v1 = (
    tf.keras.callbacks.CSVLogger(

        HISTORY_PATH_V1,

        separator=",",

        append=False
    )
)


callbacks_v1 = [

    best_backbone_callback_v1,

    reduce_lr_v1,

    early_stopping_v1,

    csv_logger_v1,
]


# ============================================================
# 10. GUARDAR SCALERS ANTES DEL ENTRENAMIENTO
#
# Ambos fueron ajustados exclusivamente con TRAIN.
# ============================================================

joblib.dump(

    scaler_hist,

    SCALER_HIST_PATH_V1
)


joblib.dump(

    scaler_fut,

    SCALER_FUT_PATH_V1
)


print(
    "\n✓ Scalers V1 guardados."
)


# ============================================================
# 11. ESTADO PRE-ENTRENAMIENTO
# ============================================================

initial_lr_v1 = float(

    tf.keras.backend.get_value(

        trainer_v24_v1.optimizer.learning_rate
    )
)


print(
    "\n=== ESTADO PRE-ENTRENAMIENTO ==="
)

print(
    "Optimizer:",
    type(
        trainer_v24_v1.optimizer
    ).__name__
)

print(
    "Iterations:",
    int(
        trainer_v24_v1.optimizer
        .iterations
        .numpy()
    )
)

print(
    "Learning rate:",
    initial_lr_v1
)

print(
    "Modelo destino:",
    MODEL_PATH_V1
)


assert np.isclose(

    initial_lr_v1,

    cfg.lr,

    rtol=1e-6
)


# ============================================================
# 12. ENTRENAMIENTO
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    " INICIO ENTRENAMIENTO PI-LSTM V2.4 V1"
)

print(
    "============================================================\n"
)


history_v1 = trainer_v24_v1.fit(

    ds_train,

    validation_data=ds_val,

    epochs=cfg.epochs,

    steps_per_epoch=steps_train,

    validation_steps=steps_val,

    callbacks=callbacks_v1,

    verbose=1
)


print(
    "\n"
    "============================================================"
)

print(
    " FIN ENTRENAMIENTO PI-LSTM V2.4 V1"
)

print(
    "============================================================"
)


# ============================================================
# 13. CONVERTIR HISTORY A DATAFRAME
# ============================================================

history_df_v1 = pd.DataFrame(
    history_v1.history
)


history_df_v1.insert(

    0,

    "epoch",

    np.arange(
        1,
        len(history_df_v1) + 1
    )
)


print(
    "\n=== HISTORIAL DE ENTRENAMIENTO ==="
)

print(
    history_df_v1.to_string(
        index=False
    )
)


# ============================================================
# 14. IDENTIFICAR MEJOR EPOCH
# ============================================================

if (
    "val_loss"
    not in history_df_v1.columns
):

    raise RuntimeError(
        "history no contiene val_loss."
    )


best_idx_v1 = int(

    history_df_v1[
        "val_loss"
    ].idxmin()
)


best_epoch_v1 = int(

    history_df_v1.loc[
        best_idx_v1,
        "epoch"
    ]
)


best_val_loss_v1 = float(

    history_df_v1.loc[
        best_idx_v1,
        "val_loss"
    ]
)


best_train_loss_v1 = float(

    history_df_v1.loc[
        best_idx_v1,
        "loss"
    ]
)


print(
    "\n=== MEJOR EPOCH ==="
)

print(
    "Epoch:",
    best_epoch_v1
)

print(
    "Train loss:",
    best_train_loss_v1
)

print(
    "Validation loss:",
    best_val_loss_v1
)


# ============================================================
# 15. CONSISTENCIA CON CALLBACK DE GUARDADO
# ============================================================

print(
    "\n=== CALLBACK BEST BACKBONE ==="
)

print(
    "Best epoch callback:",
    best_backbone_callback_v1.best_epoch
)

print(
    "Best val_loss callback:",
    best_backbone_callback_v1.best
)


assert (
    best_backbone_callback_v1.best_epoch
    ==
    best_epoch_v1
)


assert np.isclose(

    best_backbone_callback_v1.best,

    best_val_loss_v1,

    rtol=1e-6,
    atol=1e-8
)


# ============================================================
# 16. CONFIRMAR QUE EARLY STOPPING RESTAURÓ
#     LOS MEJORES PESOS EN MEMORIA
# ============================================================

print(
    "\n=== EARLY STOPPING ==="
)

print(
    "Epochs ejecutados:",
    len(
        history_df_v1
    )
)

print(
    "Epoch máximo:",
    cfg.epochs
)

print(
    "Best epoch:",
    best_epoch_v1
)


if (
    len(history_df_v1)
    <
    cfg.epochs
):

    print(
        "✓ Early stopping activado."
    )

else:

    print(
        "✓ Se completaron todos los epochs."
    )


# ============================================================
# 17. GUARDAR BACKBONE RESTAURADO
#
# EarlyStopping(restore_best_weights=True) deja el Trainer
# con los mejores pesos.
#
# Guardamos nuevamente sobre MODEL_PATH_V1 para asegurar
# que el archivo y el modelo en memoria sean idénticos.
# ============================================================

backbone.save(

    MODEL_PATH_V1,

    overwrite=True
)


print(
    "\n✓ Mejor backbone guardado definitivamente."
)


# ============================================================
# 18. GUARDAR HISTORY DEFINITIVO
#
# CSVLogger ya lo creó, pero lo escribimos también desde
# history_df para garantizar que incluya la columna epoch.
# ============================================================

history_df_v1.to_csv(

    HISTORY_PATH_V1,

    index=False
)


# ============================================================
# 19. GUARDAR INFORMACIÓN DEL MEJOR EPOCH
# ============================================================

best_epoch_info_v1 = {

    "best_epoch":
        best_epoch_v1,

    "best_train_loss":
        best_train_loss_v1,

    "best_val_loss":
        best_val_loss_v1,

    "epochs_executed":
        int(
            len(
                history_df_v1
            )
        ),

    "early_stopping_triggered":
        bool(
            len(
                history_df_v1
            )
            <
            cfg.epochs
        ),
}


with open(

    BEST_EPOCH_PATH_V1,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        best_epoch_info_v1,

        f,

        indent=2
    )


# ============================================================
# 20. METADATOS DE REPRODUCIBILIDAD
# ============================================================

metadata_v1 = {

    "experiment":
        "PI-LSTM V2.4 V1",

    "status":
        "canonical_retraining",

    "seed":
        int(
            SEED
        ),

    "data_path":
        str(
            cfg.data_path
        ),

    "model_path":
        str(
            MODEL_PATH_V1
        ),

    # --------------------------------------------------------
    # Splits
    # --------------------------------------------------------

    "train_start":
        str(
            train_df[
                "timestamp"
            ].min()
        ),

    "train_end":
        str(
            train_df[
                "timestamp"
            ].max()
        ),

    "val_start":
        str(
            val_df[
                "timestamp"
            ].min()
        ),

    "val_end":
        str(
            val_df[
                "timestamp"
            ].max()
        ),

    "test_start":
        str(
            test_df[
                "timestamp"
            ].min()
        ),

    "test_end":
        str(
            test_df[
                "timestamp"
            ].max()
        ),

    # --------------------------------------------------------
    # Dimensiones
    # --------------------------------------------------------

    "L":
        int(
            cfg.L
        ),

    "H":
        int(
            cfg.H
        ),

    "stride":
        int(
            cfg.stride
        ),

    "batch_size":
        int(
            cfg.batch_size
        ),

    "n_hist_features":
        int(
            len(
                HIST_COLS
            )
        ),

    "n_future_features":
        int(
            len(
                FUT_COLS
            )
        ),

    "n_yext":
        int(
            len(
                Y_EXT_COLS
            )
        ),

    # --------------------------------------------------------
    # Ventanas
    # --------------------------------------------------------

    "nwin_train":
        int(
            nwin_train
        ),

    "nwin_val":
        int(
            nwin_val
        ),

    "nwin_test":
        int(
            nwin_test
        ),

    "steps_train":
        int(
            steps_train
        ),

    "steps_val":
        int(
            steps_val
        ),

    "steps_test":
        int(
            steps_test
        ),

    # --------------------------------------------------------
    # Arquitectura
    # --------------------------------------------------------

    "backbone_name":
        backbone.name,

    "backbone_params":
        int(
            backbone.count_params()
        ),

    "trainable_tensors":
        int(
            len(
                backbone.trainable_variables
            )
        ),

    "latent_enc":
        int(
            cfg.latent_enc
        ),

    "latent_dec":
        int(
            cfg.latent_dec
        ),

    "dropout":
        float(
            cfg.dropout
        ),

    # --------------------------------------------------------
    # Targets
    # --------------------------------------------------------

    "target_direct":
        "kDNI",

    "target_diffuse":
        "qDHI=log1p(kDHI)",

    "kdni_max":
        float(
            cfg.kdni_max
        ),

    "kdhi_max":
        float(
            cfg.kdhi_max
        ),

    "qdhi_max":
        float(
            QDHI_MAX
        ),

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    "loss_family":
        "PartitionAwareLoss V2.3-compatible + qDHI",

    "huber_delta":
        float(
            loss_layer_v24_v1.huber_delta
        ),

    "huber_reduction":
        str(
            loss_layer_v24_v1.huber.reduction
        ),

    "lambda_kdni":
        float(
            cfg.lambda_kdni
        ),

    "lambda_qdhi":
        float(
            cfg.lambda_qdhi
        ),

    "lambda_ghi":
        float(
            cfg.lambda_ghi
        ),

    "lambda_ratio":
        float(
            cfg.lambda_ratio
        ),

    "lambda_aggregate":
        float(
            cfg.lambda_energy
        ),

    "thr_day_cosz":
        float(
            cfg.thr_day_cosz
        ),

    "thr_cs":
        float(
            cfg.thr_cs
        ),

    # --------------------------------------------------------
    # Optimización
    # --------------------------------------------------------

    "optimizer":
        "Adam",

    "initial_learning_rate":
        float(
            cfg.lr
        ),

    "epochs_max":
        int(
            cfg.epochs
        ),

    "patience_early_stopping":
        int(
            cfg.patience_es
        ),

    "patience_reduce_lr":
        int(
            cfg.patience_rlr
        ),

    "reduce_lr_factor":
        float(
            cfg.reduce_lr_factor
        ),

    "min_lr":
        float(
            cfg.min_lr
        ),

    # --------------------------------------------------------
    # Resultado entrenamiento
    # --------------------------------------------------------

    "epochs_executed":
        int(
            len(
                history_df_v1
            )
        ),

    "best_epoch":
        int(
            best_epoch_v1
        ),

    "best_train_loss":
        float(
            best_train_loss_v1
        ),

    "best_val_loss":
        float(
            best_val_loss_v1
        ),

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    "hist_cols":
        list(
            HIST_COLS
        ),

    "future_cols":
        list(
            FUT_COLS
        ),

    "y_ext_cols":
        list(
            Y_EXT_COLS
        ),
}


with open(

    METADATA_PATH_V1,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        metadata_v1,

        f,

        indent=2,

        ensure_ascii=False
    )


# ============================================================
# 21. VALIDAR ARCHIVOS GENERADOS
# ============================================================

required_files_v1 = [

    MODEL_PATH_V1,

    HISTORY_PATH_V1,

    METADATA_PATH_V1,

    SCALER_HIST_PATH_V1,

    SCALER_FUT_PATH_V1,

    BEST_EPOCH_PATH_V1,
]


print(
    "\n=== ARTEFACTOS GENERADOS ==="
)


for p in required_files_v1:

    exists = p.exists()

    size_mb = (

        p.stat().st_size
        /
        (1024 ** 2)

        if exists

        else np.nan
    )


    print(

        f"{p.name:55s} | "
        f"exists={exists} | "
        f"{size_mb:.3f} MB"
    )


    assert exists


# ============================================================
# 22. VERIFICAR MODELO GUARDADO
#
# Se carga SIN compile porque únicamente necesitamos
# comprobar arquitectura y pesos.
# ============================================================

loaded_backbone_v1 = (
    tf.keras.models.load_model(

        MODEL_PATH_V1,

        compile=False
    )
)


print(
    "\n=== VERIFICACIÓN DEL MODELO GUARDADO ==="
)

print(
    "Nombre:",
    loaded_backbone_v1.name
)

print(
    "Parámetros:",
    f"{loaded_backbone_v1.count_params():,}"
)

print(
    "Output shape:",
    loaded_backbone_v1.output_shape
)


assert (
    loaded_backbone_v1.count_params()
    ==
    backbone.count_params()
)


assert (
    loaded_backbone_v1.output_shape[-2:]
    ==
    (
        cfg.H,
        2
    )
)


# ============================================================
# 23. COMPARAR PREDICCIÓN MODELO EN MEMORIA
#     VS MODELO RECARGADO
# ============================================================

pred_memory_v1 = backbone(

    [
        batch_xh,
        batch_xf
    ],

    training=False
).numpy()


pred_loaded_v1 = loaded_backbone_v1(

    [
        batch_xh,
        batch_xf
    ],

    training=False
).numpy()


reload_max_difference_v1 = float(

    np.max(

        np.abs(

            pred_memory_v1
            -
            pred_loaded_v1
        )
    )
)


print(
    "\nMáxima diferencia memoria vs modelo recargado:",
    reload_max_difference_v1
)


assert (
    reload_max_difference_v1
    <
    1e-6
)


# ============================================================
# 24. RESUMEN FINAL
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    " RESUMEN ENTRENAMIENTO PI-LSTM V2.4 V1"
)

print(
    "============================================================"
)


print(
    "Epochs ejecutados :",
    len(
        history_df_v1
    )
)

print(
    "Mejor epoch       :",
    best_epoch_v1
)

print(
    "Best train loss   :",
    best_train_loss_v1
)

print(
    "Best val loss     :",
    best_val_loss_v1
)

print(
    "Modelo guardado   :",
    MODEL_PATH_V1
)

print(
    "Salida            :",
    OUTPUT_DIR_V1
)


print(
    "\n✓ Entrenamiento V2.4 V1 completado."
)

print(
    "✓ Mejor epoch identificado."
)

print(
    "✓ EarlyStopping / ReduceLROnPlateau configurados."
)

print(
    "✓ Backbone del mejor epoch preservado."
)

print(
    "✓ Scalers guardados."
)

print(
    "✓ History guardado."
)

print(
    "✓ Metadatos guardados."
)

print(
    "✓ Modelo recargado y verificado."
)

print(
    "✓ Celda 10 V1 completada correctamente."
)

In [ ]:
# ============================================================
# 11. PI-LSTM V2.4 — CARGA DEL MEJOR MODELO
#     Y EVALUACIÓN INDEPENDIENTE DE TEST
# ============================================================


# ============================================================
# 1. CARGAR EL MEJOR BACKBONE DESDE DISCO
# ============================================================

best_model_path = Path(
    cfg.model_path
)

if not best_model_path.exists():

    raise FileNotFoundError(
        f"No existe el modelo:\n{best_model_path}"
    )


best_backbone = tf.keras.models.load_model(
    best_model_path,
    compile=False
)


print(
    "=== MODELO V2.4 RECARGADO ==="
)

print(
    "Archivo:",
    best_model_path
)

print(
    "Parámetros:",
    best_backbone.count_params()
)

print(
    "Output shape:",
    best_backbone.output_shape
)


assert (
    best_backbone.count_params()
    == 110146
)

assert (
    best_backbone.output_shape[-1]
    == 2
)


# ============================================================
# 2. COMPROBAR DOMINIO DE UNA PREDICCIÓN
# ============================================================

pred_check = best_backbone(
    [
        batch_xh,
        batch_xf
    ],
    training=False
).numpy()


kdni_check = pred_check[
    ...,
    0
]

qdhi_check = pred_check[
    ...,
    1
]

kdhi_check = np.expm1(
    qdhi_check
)


print(
    "\n=== DOMINIO DEL MODELO RECARGADO ==="
)

print(
    "kDNI min/max:",
    float(kdni_check.min()),
    float(kdni_check.max())
)

print(
    "qDHI min/max:",
    float(qdhi_check.min()),
    float(qdhi_check.max())
)

print(
    "kDHI min/max:",
    float(kdhi_check.min()),
    float(kdhi_check.max())
)


assert kdni_check.min() >= 0.0
assert kdni_check.max() <= cfg.kdni_max + 1e-6

assert qdhi_check.min() >= 0.0
assert qdhi_check.max() <= QDHI_MAX + 1e-6

assert kdhi_check.min() >= 0.0
assert kdhi_check.max() <= cfg.kdhi_max + 1e-5


# ============================================================
# 3. EVALUAR LA LOSS SOBRE TEST COMPLETO
# ============================================================

test_loss_tracker = tf.keras.metrics.Mean()

test_kdni_tracker = tf.keras.metrics.Mean()
test_qdhi_tracker = tf.keras.metrics.Mean()
test_ghi_tracker = tf.keras.metrics.Mean()
test_ratio_tracker = tf.keras.metrics.Mean()
test_aggregate_tracker = tf.keras.metrics.Mean()


for (
    (
        xh_batch,
        xf_batch
    ),
    y_batch
) in ds_test:

    y_pred_batch = best_backbone(
        [
            xh_batch,
            xf_batch
        ],
        training=False
    )

    comps = loss_v24.components(
        y_batch,
        y_pred_batch
    )

    test_loss_tracker.update_state(
        comps["total"]
    )

    test_kdni_tracker.update_state(
        comps["kdni"]
    )

    test_qdhi_tracker.update_state(
        comps["qdhi"]
    )

    test_ghi_tracker.update_state(
        comps["ghi"]
    )

    test_ratio_tracker.update_state(
        comps["ratio"]
    )

    test_aggregate_tracker.update_state(
        comps["aggregate"]
    )


print(
    "\n=== LOSS EN TEST 2023–2024 ==="
)

print(
    "test_loss      :",
    float(
        test_loss_tracker.result().numpy()
    )
)

print(
    "test_kdni      :",
    float(
        test_kdni_tracker.result().numpy()
    )
)

print(
    "test_qdhi      :",
    float(
        test_qdhi_tracker.result().numpy()
    )
)

print(
    "test_ghi       :",
    float(
        test_ghi_tracker.result().numpy()
    )
)

print(
    "test_ratio     :",
    float(
        test_ratio_tracker.result().numpy()
    )
)

print(
    "test_aggregate :",
    float(
        test_aggregate_tracker.result().numpy()
    )
)


# ============================================================
# 4. GENERAR PREDICCIONES DE TODAS LAS VENTANAS DE TEST
#
# IMPORTANTE:
# Cada fila de estas matrices representa una instancia de
# pronóstico. Debido a H=24 y stride=2, un mismo timestamp
# físico puede aparecer en varias ventanas.
#
# Estas predicciones se utilizarán inicialmente para evaluar
# precisión de forecasting.
# ============================================================

pred_batches = []
true_batches = []


for (
    (
        xh_batch,
        xf_batch
    ),
    y_batch
) in ds_test:

    pred_batch = best_backbone(
        [
            xh_batch,
            xf_batch
        ],
        training=False
    )

    pred_batches.append(
        pred_batch.numpy()
    )

    true_batches.append(
        y_batch.numpy()
    )


Y_pred_test_windows = np.concatenate(
    pred_batches,
    axis=0
)

Y_true_test_windows = np.concatenate(
    true_batches,
    axis=0
)


print(
    "\n=== MATRICES DE PRONÓSTICO TEST ==="
)

print(
    "Y_pred_test_windows:",
    Y_pred_test_windows.shape
)

print(
    "Y_true_test_windows:",
    Y_true_test_windows.shape
)


assert Y_pred_test_windows.shape == (
    nwin_test,
    cfg.H,
    2
)

assert Y_true_test_windows.shape == (
    nwin_test,
    cfg.H,
    len(Y_EXT_COLS)
)


# ============================================================
# 5. EXTRAER TARGETS Y RECONSTRUIR VARIABLES FÍSICAS
# ============================================================

# ------------------------------------------------------------
# Predicción
# ------------------------------------------------------------

kdni_pred = Y_pred_test_windows[
    ...,
    0
]

qdhi_pred = Y_pred_test_windows[
    ...,
    1
]

kdhi_pred = np.expm1(
    qdhi_pred
)


# ------------------------------------------------------------
# Observación / auxiliares físicas
# ------------------------------------------------------------

kdni_true = Y_true_test_windows[
    ...,
    0
]

qdhi_true = Y_true_test_windows[
    ...,
    1
]

kdhi_true = np.expm1(
    qdhi_true
)

ghi_true = Y_true_test_windows[
    ...,
    2
]

csdni = Y_true_test_windows[
    ...,
    3
]

csdhi = Y_true_test_windows[
    ...,
    4
]

csghi = Y_true_test_windows[
    ...,
    5
]

cosz = Y_true_test_windows[
    ...,
    6
]

dhi_true = Y_true_test_windows[
    ...,
    7
]


# ------------------------------------------------------------
# Reconstrucción física predicha
# ------------------------------------------------------------

dni_pred = (
    kdni_pred
    * csdni
)

dhi_pred = (
    kdhi_pred
    * csdhi
)

ghi_pred = (
    dni_pred
    * cosz
    + dhi_pred
)


# ------------------------------------------------------------
# DNI observado
#
# Se utiliza el DNI real implícito en kDNI × DNI_cs.
# ------------------------------------------------------------

dni_true = (
    kdni_true
    * csdni
)


# ============================================================
# 6. CIERRE FÍSICO DE LA PREDICCIÓN
# ============================================================

closure_pred = (
    ghi_pred
    - (
        dni_pred
        * cosz
        + dhi_pred
    )
)


closure_mae = float(
    np.mean(
        np.abs(
            closure_pred
        )
    )
)

closure_max = float(
    np.max(
        np.abs(
            closure_pred
        )
    )
)


print(
    "\n=== CIERRE FÍSICO ==="
)

print(
    "Closure MAE:",
    closure_mae
)

print(
    "Closure max abs:",
    closure_max
)


assert closure_mae < 1e-5


# ============================================================
# 7. DOMINIOS FÍSICOS
# ============================================================

print(
    "\n=== RANGOS FÍSICOS PREDICHOS ==="
)

print(
    "DNI min/max:",
    float(dni_pred.min()),
    float(dni_pred.max())
)

print(
    "DHI min/max:",
    float(dhi_pred.min()),
    float(dhi_pred.max())
)

print(
    "GHI min/max:",
    float(ghi_pred.min()),
    float(ghi_pred.max())
)


for name, arr in {
    "DNI_pred": dni_pred,
    "DHI_pred": dhi_pred,
    "GHI_pred": ghi_pred,
}.items():

    if not np.all(
        np.isfinite(arr)
    ):

        raise ValueError(
            f"{name} contiene NaN o Inf."
        )


# ============================================================
# 8. RESUMEN
# ============================================================

print(
    "\n✓ Mejor modelo V2.4 recargado desde disco."
)

print(
    "✓ Test independiente 2023–2024 evaluado."
)

print(
    "✓ Predicciones físicas reconstruidas."
)

print(
    "✓ Cierre radiativo verificado."
)

print(
    "✓ Celda 11 completada correctamente."
)

=== MODELO V2.4 RECARGADO ===
Archivo: PI_LSTM_V2_4_logDHI_partition_stride2.keras
Parámetros: 110146
Output shape: (None, 24, 2)

=== DOMINIO DEL MODELO RECARGADO ===
kDNI min/max: 2.2084796214016933e-09 0.7832181453704834
qDHI min/max: 1.4985287180024898e-06 1.7309370040893555
kDHI min/max: 1.498529854870867e-06 4.645941734313965

=== LOSS EN TEST 2023–2024 ===
test_loss      : 0.2052311897277832
test_kdni      : 0.05438792333006859
test_qdhi      : 0.03424099460244179
test_ghi       : 0.02398819848895073
test_ratio     : 0.04918034002184868
test_aggregate : 0.09182074666023254

=== MATRICES DE PRONÓSTICO TEST ===
Y_pred_test_windows: (17497, 24, 2)
Y_true_test_windows: (17497, 24, 8)

=== CIERRE FÍSICO ===
Closure MAE: 0.0
Closure max abs: 0.0

=== RANGOS FÍSICOS PREDICHOS ===
DNI min/max: 0.0 1127.4078369140625
DHI min/max: 0.0 828.0681762695312
GHI min/max: 0.0 1193.296142578125

✓ Mejor modelo V2.4 recargado desde disco.
✓ Test independiente 2023–2024 evaluado.
✓ Predicciones fís

In [ ]:
# ============================================================
# 12. PI-LSTM V2.4 — MÉTRICAS FÍSICAS DE TEST
#     Y EVALUACIÓN POR HORIZONTE
# ============================================================


# ============================================================
# 1. RECONSTRUIR LOS ÍNDICES TEMPORALES DE CADA VENTANA
# ============================================================

window_starts_test = np.arange(
    0,
    len(test_df) - cfg.L - cfg.H + 1,
    cfg.stride,
    dtype=np.int64
)


assert len(window_starts_test) == nwin_test


future_offsets = np.arange(
    cfg.H,
    dtype=np.int64
)


future_idx_test = (
    window_starts_test[:, None]
    + cfg.L
    + future_offsets[None, :]
)


print(
    "=== ÍNDICES DE TEST ==="
)

print(
    "window_starts_test:",
    window_starts_test.shape
)

print(
    "future_idx_test   :",
    future_idx_test.shape
)

print(
    "Índice futuro mínimo:",
    int(future_idx_test.min())
)

print(
    "Índice futuro máximo:",
    int(future_idx_test.max())
)


assert future_idx_test.shape == (
    nwin_test,
    cfg.H
)

assert future_idx_test.min() >= 0
assert future_idx_test.max() < len(test_df)


# ============================================================
# 2. TIMESTAMPS DE CADA INSTANCIA DE PRONÓSTICO
# ============================================================

test_timestamps = (
    test_df["timestamp"]
    .to_numpy()
)


timestamp_windows = test_timestamps[
    future_idx_test
]


print(
    "\nPrimer timestamp pronosticado:",
    timestamp_windows[0, 0]
)

print(
    "Último timestamp pronosticado:",
    timestamp_windows[-1, -1]
)


# ============================================================
# 3. OBSERVACIONES FÍSICAS ORIGINALES
#
# No reconstruimos las observaciones desde k-space.
# Se recuperan directamente del histórico NSRDB.
# ============================================================

dni_obs_series = (
    test_df[COL_DNI]
    .to_numpy(dtype=np.float32)
)

dhi_obs_series = (
    test_df[COL_DHI]
    .to_numpy(dtype=np.float32)
)

ghi_obs_series = (
    test_df[COL_GHI]
    .to_numpy(dtype=np.float32)
)

cosz_series = (
    test_df["cosz_plus"]
    .to_numpy(dtype=np.float32)
)


dni_obs_windows = dni_obs_series[
    future_idx_test
]

dhi_obs_windows = dhi_obs_series[
    future_idx_test
]

ghi_obs_windows = ghi_obs_series[
    future_idx_test
]

cosz_windows = cosz_series[
    future_idx_test
]


# ============================================================
# 4. VALIDAR SHAPES
# ============================================================

for name, arr in {

    "DNI obs": dni_obs_windows,
    "DHI obs": dhi_obs_windows,
    "GHI obs": ghi_obs_windows,
    "cosz": cosz_windows,

}.items():

    assert arr.shape == (
        nwin_test,
        cfg.H
    ), (
        f"Shape incorrecto en {name}: "
        f"{arr.shape}"
    )


# ============================================================
# 5. COMPROBAR COINCIDENCIA GHI / DHI CON Y_EXT
# ============================================================

max_diff_ghi_true = float(
    np.max(
        np.abs(
            ghi_obs_windows
            - ghi_true
        )
    )
)

max_diff_dhi_true = float(
    np.max(
        np.abs(
            dhi_obs_windows
            - dhi_true
        )
    )
)


print(
    "\n=== AUDITORÍA OBSERVACIONES ==="
)

print(
    "Máx diferencia GHI original vs Y_EXT:",
    max_diff_ghi_true
)

print(
    "Máx diferencia DHI original vs Y_EXT:",
    max_diff_dhi_true
)


# ============================================================
# 6. FUNCIÓN DE MÉTRICAS
# ============================================================

def regression_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=np.float64
    ).ravel()

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64
    ).ravel()


    valid = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]


    if len(y_true) == 0:

        return {
            "N": 0,
            "RMSE": np.nan,
            "MAE": np.nan,
            "Bias": np.nan,
            "R2": np.nan,
        }


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    bias = np.mean(
        y_pred
        - y_true
    )


    # R² requiere variabilidad en y_true

    if np.std(y_true) > 0:

        r2 = r2_score(
            y_true,
            y_pred
        )

    else:

        r2 = np.nan


    return {

        "N": int(len(y_true)),

        "RMSE": float(rmse),

        "MAE": float(mae),

        "Bias": float(bias),

        "R2": float(r2),
    }


# ============================================================
# 7. MÉTRICAS GLOBALES
# ============================================================

global_results = []


for variable, y_obs, y_hat in [

    (
        "DNI",
        dni_obs_windows,
        dni_pred
    ),

    (
        "DHI",
        dhi_obs_windows,
        dhi_pred
    ),

    (
        "GHI",
        ghi_obs_windows,
        ghi_pred
    ),

]:

    metrics = regression_metrics(
        y_obs,
        y_hat
    )

    global_results.append({
        "Subset": "Global",
        "Variable": variable,
        **metrics
    })


global_metrics_df = pd.DataFrame(
    global_results
)


print(
    "\n=== MÉTRICAS GLOBALES TEST 2023–2024 ==="
)

print(
    global_metrics_df.to_string(
        index=False
    )
)


# ============================================================
# 8. MÉTRICAS DIURNAS
#
# Se utiliza exactamente el criterio físico del modelo:
#
# cosz_plus > 0.05
# ============================================================

day_mask = (
    cosz_windows
    > cfg.thr_day_cosz
)


print(
    "\nInstancias totales:",
    int(day_mask.size)
)

print(
    "Instancias diurnas:",
    int(day_mask.sum())
)

print(
    "Fracción diurna:",
    float(day_mask.mean())
)


day_results = []


for variable, y_obs, y_hat in [

    (
        "DNI",
        dni_obs_windows,
        dni_pred
    ),

    (
        "DHI",
        dhi_obs_windows,
        dhi_pred
    ),

    (
        "GHI",
        ghi_obs_windows,
        ghi_pred
    ),

]:

    metrics = regression_metrics(
        y_obs[day_mask],
        y_hat[day_mask]
    )

    day_results.append({
        "Subset": "Daytime",
        "Variable": variable,
        **metrics
    })


day_metrics_df = pd.DataFrame(
    day_results
)


print(
    "\n=== MÉTRICAS DIURNAS TEST 2023–2024 ==="
)

print(
    day_metrics_df.to_string(
        index=False
    )
)


# ============================================================
# 9. MÉTRICAS POR HORIZONTE
#
# h = 1  -> 30 min adelante
# h = 24 -> 12 h adelante
# ============================================================

horizon_results = []


for h in range(
    cfg.H
):

    lead_steps = h + 1

    lead_hours = (
        lead_steps
        * 0.5
    )


    mask_h_day = (
        cosz_windows[:, h]
        > cfg.thr_day_cosz
    )


    for variable, y_obs, y_hat in [

        (
            "DNI",
            dni_obs_windows[:, h],
            dni_pred[:, h]
        ),

        (
            "DHI",
            dhi_obs_windows[:, h],
            dhi_pred[:, h]
        ),

        (
            "GHI",
            ghi_obs_windows[:, h],
            ghi_pred[:, h]
        ),

    ]:

        # Global para ese horizonte

        m_global = regression_metrics(
            y_obs,
            y_hat
        )

        horizon_results.append({

            "Horizon_step":
                lead_steps,

            "Lead_hours":
                lead_hours,

            "Subset":
                "Global",

            "Variable":
                variable,

            **m_global
        })


        # Diurno para ese horizonte

        m_day = regression_metrics(
            y_obs[
                mask_h_day
            ],
            y_hat[
                mask_h_day
            ]
        )

        horizon_results.append({

            "Horizon_step":
                lead_steps,

            "Lead_hours":
                lead_hours,

            "Subset":
                "Daytime",

            "Variable":
                variable,

            **m_day
        })


horizon_metrics_df = pd.DataFrame(
    horizon_results
)


# ============================================================
# 10. PARTICIÓN DIFUSA
# ============================================================

eps_eval = 1e-6


diffuse_ratio_obs = np.where(
    ghi_obs_windows > cfg.thr_cs,
    dhi_obs_windows
    / np.maximum(
        ghi_obs_windows,
        eps_eval
    ),
    np.nan
)


diffuse_ratio_pred = np.where(
    ghi_pred > cfg.thr_cs,
    dhi_pred
    / np.maximum(
        ghi_pred,
        eps_eval
    ),
    np.nan
)


ratio_valid = (
    np.isfinite(
        diffuse_ratio_obs
    )
    &
    np.isfinite(
        diffuse_ratio_pred
    )
)


ratio_metrics = regression_metrics(

    diffuse_ratio_obs[
        ratio_valid
    ],

    diffuse_ratio_pred[
        ratio_valid
    ]
)


ratio_obs_mean = float(
    np.mean(
        diffuse_ratio_obs[
            ratio_valid
        ]
    )
)

ratio_pred_mean = float(
    np.mean(
        diffuse_ratio_pred[
            ratio_valid
        ]
    )
)

ratio_bias_mean = (
    ratio_pred_mean
    - ratio_obs_mean
)


print(
    "\n=== PARTICIÓN DIFUSA TEST ==="
)

print(
    "N válido:",
    int(
        ratio_valid.sum()
    )
)

print(
    "Diffuse ratio observado:",
    ratio_obs_mean
)

print(
    "Diffuse ratio predicho :",
    ratio_pred_mean
)

print(
    "Sesgo medio            :",
    ratio_bias_mean
)

print(
    "RMSE ratio             :",
    ratio_metrics["RMSE"]
)

print(
    "MAE ratio              :",
    ratio_metrics["MAE"]
)


# ============================================================
# 11. ESTADÍSTICOS kDHI EN TEST
# ============================================================

valid_kdhi = (
    csdhi > cfg.thr_cs
)


print(
    "\n=== kDHI TEST ==="
)

print(
    "kDHI true mean:",
    float(
        np.mean(
            kdhi_true[
                valid_kdhi
            ]
        )
    )
)

print(
    "kDHI pred mean:",
    float(
        np.mean(
            kdhi_pred[
                valid_kdhi
            ]
        )
    )
)

print(
    "kDHI true median:",
    float(
        np.median(
            kdhi_true[
                valid_kdhi
            ]
        )
    )
)

print(
    "kDHI pred median:",
    float(
        np.median(
            kdhi_pred[
                valid_kdhi
            ]
        )
    )
)

print(
    "kDHI true P95:",
    float(
        np.percentile(
            kdhi_true[
                valid_kdhi
            ],
            95
        )
    )
)

print(
    "kDHI pred P95:",
    float(
        np.percentile(
            kdhi_pred[
                valid_kdhi
            ],
            95
        )
    )
)


# ============================================================
# 12. RESUMEN POR HORIZONTE: GHI DIURNO
# ============================================================

ghi_day_horizon = (
    horizon_metrics_df[
        (
            horizon_metrics_df[
                "Variable"
            ] == "GHI"
        )
        &
        (
            horizon_metrics_df[
                "Subset"
            ] == "Daytime"
        )
    ]
    .copy()
)


print(
    "\n=== GHI DIURNO POR HORIZONTE ==="
)

print(
    ghi_day_horizon[
        [
            "Horizon_step",
            "Lead_hours",
            "RMSE",
            "MAE",
            "Bias",
            "R2"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 13. VALIDACIONES FINALES
# ============================================================

assert len(
    horizon_metrics_df
) == (
    cfg.H
    * 3
    * 2
)


assert np.all(
    np.isfinite(
        global_metrics_df[
            [
                "RMSE",
                "MAE",
                "Bias",
                "R2"
            ]
        ].to_numpy()
    )
)


assert np.all(
    np.isfinite(
        day_metrics_df[
            [
                "RMSE",
                "MAE",
                "Bias",
                "R2"
            ]
        ].to_numpy()
    )
)


print(
    "\n✓ Métricas físicas globales calculadas."
)

print(
    "✓ Métricas diurnas calculadas."
)

print(
    "✓ Evaluación por horizonte completada."
)

print(
    "✓ Partición difusa evaluada."
)

print(
    "✓ Celda 12 completada correctamente."
)

=== ÍNDICES DE TEST ===
window_starts_test: (17497,)
future_idx_test   : (17497, 24)
Índice futuro mínimo: 72
Índice futuro máximo: 35087

Primer timestamp pronosticado: 2023-01-02T12:00:00.000000000
Último timestamp pronosticado: 2024-12-31T23:30:00.000000000

=== AUDITORÍA OBSERVACIONES ===
Máx diferencia GHI original vs Y_EXT: 0.0
Máx diferencia DHI original vs Y_EXT: 0.0

=== MÉTRICAS GLOBALES TEST 2023–2024 ===
Subset Variable      N       RMSE        MAE       Bias       R2
Global      DNI 419928 202.738972 109.284602 -53.317307 0.512866
Global      DHI 419928  61.456993  32.102702  10.279991 0.797422
Global      GHI 419928 117.825710  61.621328 -28.454829 0.859159

Instancias totales: 419928
Instancias diurnas: 204469
Fracción diurna: 0.4869144234249681

=== MÉTRICAS DIURNAS TEST 2023–2024 ===
 Subset Variable      N       RMSE        MAE        Bias       R2
Daytime      DNI 204469 290.443545 223.588027 -108.817483 0.172367
Daytime      DHI 204469  88.060491  65.661079   21.378

In [ ]:
# ============================================================
# 13. PI-LSTM V2.4 — RECONSTRUCCIÓN CRONOLÓGICA ÚNICA
#     Y BALANCE ENERGÉTICO
# ============================================================


# ============================================================
# 1. PREPARAR REGISTROS DE TODAS LAS INSTANCIAS DE PRONÓSTICO
# ============================================================

records = []

for i in range(nwin_test):

    start_idx = int(
        window_starts_test[i]
    )

    for h in range(cfg.H):

        target_idx = int(
            future_idx_test[i, h]
        )

        records.append({

            "window_id":
                i,

            "window_start_idx":
                start_idx,

            "horizon_step":
                h + 1,

            "lead_hours":
                0.5 * (h + 1),

            "target_idx":
                target_idx,

            "timestamp":
                test_timestamps[
                    target_idx
                ],

            "DNI_pred":
                float(
                    dni_pred[i, h]
                ),

            "DHI_pred":
                float(
                    dhi_pred[i, h]
                ),

            "GHI_pred":
                float(
                    ghi_pred[i, h]
                ),
        })


forecast_long_df = pd.DataFrame(
    records
)


print(
    "=== FORECAST LONG TABLE ==="
)

print(
    "Filas:",
    f"{len(forecast_long_df):,}"
)

print(
    "Timestamps únicos:",
    f"{forecast_long_df['timestamp'].nunique():,}"
)

print(
    "Horizonte min/max:",
    forecast_long_df[
        "horizon_step"
    ].min(),
    forecast_long_df[
        "horizon_step"
    ].max()
)


assert len(
    forecast_long_df
) == (
    nwin_test
    * cfg.H
)


# ============================================================
# 2. SERIE OBSERVADA DE REFERENCIA
# ============================================================

obs_reference = (
    test_df[
        [
            "timestamp",
            COL_DNI,
            COL_DHI,
            COL_GHI,
            "cosz_plus"
        ]
    ]
    .copy()
)


obs_reference = obs_reference.rename(
    columns={

        COL_DNI:
            "DNI_obs",

        COL_DHI:
            "DHI_obs",

        COL_GHI:
            "GHI_obs",
    }
)


# ============================================================
# 3. SHORTEST LEAD
#
# Para cada timestamp se selecciona la predicción disponible
# con el menor horizonte de anticipación.
#
# Esta estrategia representa el pronóstico con la información
# histórica más reciente disponible.
# ============================================================

shortest_lead_df = (

    forecast_long_df

    .sort_values(
        [
            "timestamp",
            "horizon_step"
        ]
    )

    .drop_duplicates(
        subset="timestamp",
        keep="first"
    )

    .sort_values(
        "timestamp"
    )

    .reset_index(
        drop=True
    )
)


shortest_lead_df = shortest_lead_df.merge(

    obs_reference,

    on="timestamp",

    how="left",

    validate="one_to_one"
)


shortest_lead_df[
    "strategy"
] = "shortest_lead"


# ============================================================
# 4. MEAN OVERLAP
#
# Para cada timestamp se promedian todas las predicciones
# provenientes de los distintos horizontes disponibles.
#
# Esta estrategia funciona como una agregación de pronósticos
# superpuestos y no como una trayectoria autorregresiva.
# ============================================================

mean_overlap_df = (

    forecast_long_df

    .groupby(
        "timestamp",
        as_index=False
    )

    .agg({

        "DNI_pred":
            "mean",

        "DHI_pred":
            "mean",

        "GHI_pred":
            "mean",

        "horizon_step":
            "count",
    })

    .rename(
        columns={
            "horizon_step":
                "n_forecasts"
        }
    )

    .sort_values(
        "timestamp"
    )

    .reset_index(
        drop=True
    )
)


mean_overlap_df = mean_overlap_df.merge(

    obs_reference,

    on="timestamp",

    how="left",

    validate="one_to_one"
)


mean_overlap_df[
    "strategy"
] = "mean_overlap"


# ============================================================
# 5. NON-OVERLAPPING BLOCKS
#
# Se seleccionan ventanas separadas exactamente H pasos.
#
# IMPORTANTE:
#
# Esta estrategia NO constituye un rollout autorregresivo.
#
# Cada bloque sigue utilizando como contexto histórico las
# observaciones reales de NSRDB contenidas en XH_test.
#
# Por tanto, representa una evaluación mediante bloques
# consecutivos no superpuestos de H pasos.
#
# Como:
#
#     H = 24
#     stride = 2
#
# tenemos:
#
#     H / stride = 12
#
# y se selecciona una ventana cada 12 orígenes.
# ============================================================

if cfg.H % cfg.stride != 0:

    raise ValueError(
        "H debe ser divisible por stride "
        "para construir bloques no superpuestos."
    )


window_jump = (
    cfg.H
    // cfg.stride
)


non_overlapping_window_ids = np.arange(
    0,
    nwin_test,
    window_jump,
    dtype=int
)


non_overlapping_records = []


for i in non_overlapping_window_ids:

    for h in range(cfg.H):

        target_idx = int(
            future_idx_test[
                i,
                h
            ]
        )

        non_overlapping_records.append({

            "window_id":
                int(i),

            "horizon_step":
                int(h + 1),

            "lead_hours":
                float(
                    0.5
                    * (h + 1)
                ),

            "target_idx":
                target_idx,

            "timestamp":
                test_timestamps[
                    target_idx
                ],

            "DNI_pred":
                float(
                    dni_pred[
                        i,
                        h
                    ]
                ),

            "DHI_pred":
                float(
                    dhi_pred[
                        i,
                        h
                    ]
                ),

            "GHI_pred":
                float(
                    ghi_pred[
                        i,
                        h
                    ]
                ),
        })


non_overlapping_blocks_df = pd.DataFrame(
    non_overlapping_records
)


# ------------------------------------------------------------
# Verificar que los bloques no generen timestamps duplicados
# ------------------------------------------------------------

n_dup_non_overlapping = int(
    non_overlapping_blocks_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
)


print(
    "\nTimestamps duplicados non-overlapping blocks:",
    n_dup_non_overlapping
)


if n_dup_non_overlapping > 0:

    raise ValueError(
        "La estrategia non_overlapping_blocks "
        "contiene timestamps duplicados."
    )


non_overlapping_blocks_df = (

    non_overlapping_blocks_df

    .sort_values(
        "timestamp"
    )

    .reset_index(
        drop=True
    )
)


non_overlapping_blocks_df = (
    non_overlapping_blocks_df.merge(

        obs_reference,

        on="timestamp",

        how="left",

        validate="one_to_one"
    )
)


non_overlapping_blocks_df[
    "strategy"
] = "non_overlapping_blocks"


# ============================================================
# 6. AUDITORÍA TEMPORAL DE LAS TRES SERIES
# ============================================================

def audit_series(
    df_strategy,
    name
):

    print(
        f"\n=== {name.upper()} ==="
    )

    print(
        "N:",
        f"{len(df_strategy):,}"
    )

    print(
        "Inicio:",
        df_strategy[
            "timestamp"
        ].min()
    )

    print(
        "Fin:",
        df_strategy[
            "timestamp"
        ].max()
    )

    print(
        "Duplicados:",
        int(
            df_strategy[
                "timestamp"
            ]
            .duplicated()
            .sum()
        )
    )


    dt = (
        df_strategy[
            "timestamp"
        ]
        .diff()
        .dt.total_seconds()
        .div(60.0)
    )


    bad_dt = (
        dt.notna()
        &
        ~np.isclose(
            dt,
            30.0
        )
    )


    print(
        "Saltos != 30 min:",
        int(
            bad_dt.sum()
        )
    )


audit_series(
    shortest_lead_df,
    "shortest lead"
)

audit_series(
    mean_overlap_df,
    "mean overlap"
)

audit_series(
    non_overlapping_blocks_df,
    "non-overlapping blocks"
)


# ============================================================
# 7. MÉTRICAS FÍSICAS POR ESTRATEGIA
# ============================================================

strategy_metrics = []


for strategy_name, strategy_df in [

    (
        "shortest_lead",
        shortest_lead_df
    ),

    (
        "mean_overlap",
        mean_overlap_df
    ),

    (
        "non_overlapping_blocks",
        non_overlapping_blocks_df
    ),

]:

    for variable in [
        "DNI",
        "DHI",
        "GHI",
    ]:

        m = regression_metrics(

            strategy_df[
                f"{variable}_obs"
            ],

            strategy_df[
                f"{variable}_pred"
            ]
        )


        strategy_metrics.append({

            "Strategy":
                strategy_name,

            "Variable":
                variable,

            **m
        })


strategy_metrics_df = pd.DataFrame(
    strategy_metrics
)


print(
    "\n=== MÉTRICAS POR ESTRATEGIA ==="
)

print(
    strategy_metrics_df.to_string(
        index=False
    )
)


# ============================================================
# 8. FUNCIÓN DE ENERGÍA
#
# Resolución temporal:
#
#     30 min = 0.5 h
#
# Irradiancia en W/m²:
#
#     sum(W/m²) × 0.5 h
#
# produce energía acumulada en Wh/m².
# ============================================================

DT_HOURS = 0.5


def energy_wh_m2(
    values
):

    arr = np.asarray(
        values,
        dtype=np.float64
    )

    return float(
        np.sum(arr)
        * DT_HOURS
    )


# ============================================================
# 9. BALANCE ENERGÉTICO POR ESTRATEGIA
# ============================================================

energy_results = []


for strategy_name, strategy_df in [

    (
        "shortest_lead",
        shortest_lead_df
    ),

    (
        "mean_overlap",
        mean_overlap_df
    ),

    (
        "non_overlapping_blocks",
        non_overlapping_blocks_df
    ),

]:

    row = {
        "Strategy":
            strategy_name,
    }


    for variable in [
        "GHI",
        "DNI",
        "DHI",
    ]:

        e_obs = energy_wh_m2(
            strategy_df[
                f"{variable}_obs"
            ]
        )

        e_pred = energy_wh_m2(
            strategy_df[
                f"{variable}_pred"
            ]
        )


        bias_pct = (
            100.0
            * (
                e_pred
                - e_obs
            )
            / e_obs
        )


        row[
            f"E_{variable}_obs_Whm2"
        ] = e_obs

        row[
            f"E_{variable}_pred_Whm2"
        ] = e_pred

        row[
            f"{variable}_energy_bias_pct"
        ] = bias_pct


    # --------------------------------------------------------
    # Partición energética difusa
    #
    # Se utiliza:
    #
    #     E_DHI / E_GHI
    #
    # como indicador de la contribución difusa acumulada.
    # --------------------------------------------------------

    row[
        "diffuse_energy_ratio_obs"
    ] = (

        row[
            "E_DHI_obs_Whm2"
        ]
        /
        row[
            "E_GHI_obs_Whm2"
        ]
    )


    row[
        "diffuse_energy_ratio_pred"
    ] = (

        row[
            "E_DHI_pred_Whm2"
        ]
        /
        row[
            "E_GHI_pred_Whm2"
        ]
    )


    row[
        "diffuse_energy_ratio_bias"
    ] = (

        row[
            "diffuse_energy_ratio_pred"
        ]
        -
        row[
            "diffuse_energy_ratio_obs"
        ]
    )


    energy_results.append(
        row
    )


energy_summary_df = pd.DataFrame(
    energy_results
)


# ============================================================
# 10. MOSTRAR RESUMEN EN kWh/m²
# ============================================================

energy_display_df = (
    energy_summary_df.copy()
)


for col in [

    "E_GHI_obs_Whm2",
    "E_GHI_pred_Whm2",

    "E_DNI_obs_Whm2",
    "E_DNI_pred_Whm2",

    "E_DHI_obs_Whm2",
    "E_DHI_pred_Whm2",

]:

    energy_display_df[
        col.replace(
            "_Whm2",
            "_kWhm2"
        )
    ] = (

        energy_display_df[
            col
        ]
        / 1000.0
    )


display_cols = [

    "Strategy",

    "E_GHI_obs_kWhm2",
    "E_GHI_pred_kWhm2",
    "GHI_energy_bias_pct",

    "E_DNI_obs_kWhm2",
    "E_DNI_pred_kWhm2",
    "DNI_energy_bias_pct",

    "E_DHI_obs_kWhm2",
    "E_DHI_pred_kWhm2",
    "DHI_energy_bias_pct",

    "diffuse_energy_ratio_obs",
    "diffuse_energy_ratio_pred",
    "diffuse_energy_ratio_bias",
]


print(
    "\n=== BALANCE ENERGÉTICO ==="
)

print(
    energy_display_df[
        display_cols
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 11. CIERRE RADIATIVO EN LAS SERIES RECONSTRUIDAS
# ============================================================

closure_results = []


for strategy_name, strategy_df in [

    (
        "shortest_lead",
        shortest_lead_df
    ),

    (
        "mean_overlap",
        mean_overlap_df
    ),

    (
        "non_overlapping_blocks",
        non_overlapping_blocks_df
    ),

]:

    closure = (

        strategy_df[
            "GHI_pred"
        ].to_numpy()

        -

        (
            strategy_df[
                "DNI_pred"
            ].to_numpy()

            *

            strategy_df[
                "cosz_plus"
            ].to_numpy()

            +

            strategy_df[
                "DHI_pred"
            ].to_numpy()
        )
    )


    closure_results.append({

        "Strategy":
            strategy_name,

        "Closure_MAE":
            float(
                np.mean(
                    np.abs(
                        closure
                    )
                )
            ),

        "Closure_max_abs":
            float(
                np.max(
                    np.abs(
                        closure
                    )
                )
            ),
    })


closure_strategy_df = pd.DataFrame(
    closure_results
)


print(
    "\n=== CIERRE RADIATIVO POR ESTRATEGIA ==="
)

print(
    closure_strategy_df.to_string(
        index=False
    )
)


# ============================================================
# 12. SOLAPAMIENTO EN MEAN OVERLAP
# ============================================================

print(
    "\n=== SOLAPAMIENTO DE PRONÓSTICOS ==="
)

print(
    mean_overlap_df[
        "n_forecasts"
    ]
    .describe()
)


print(
    "\nFrecuencias:"
)

print(
    mean_overlap_df[
        "n_forecasts"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 13. VALIDACIONES FINALES
# ============================================================

assert (
    shortest_lead_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    mean_overlap_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    non_overlapping_blocks_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)


assert np.all(
    np.isfinite(
        energy_summary_df[
            [
                "GHI_energy_bias_pct",
                "DNI_energy_bias_pct",
                "DHI_energy_bias_pct",

                "diffuse_energy_ratio_obs",
                "diffuse_energy_ratio_pred",
                "diffuse_energy_ratio_bias",
            ]
        ].to_numpy()
    )
)


print(
    "\n✓ Tres estrategias cronológicas construidas."
)

print(
    "✓ Métricas físicas calculadas."
)

print(
    "✓ Balance energético calculado."
)

print(
    "✓ Partición energética difusa calculada."
)

print(
    "✓ Cierre radiativo auditado."
)

print(
    "✓ Estrategia de bloques identificada correctamente "
    "como non_overlapping_blocks."
)

print(
    "✓ Celda 13 completada correctamente."
)

=== FORECAST LONG TABLE ===
Filas: 419,928
Timestamps únicos: 35,016
Horizonte min/max: 1 24

Timestamps duplicados non-overlapping blocks: 0

=== SHORTEST LEAD ===
N: 35,016
Inicio: 2023-01-02 12:00:00
Fin: 2024-12-31 23:30:00
Duplicados: 0
Saltos != 30 min: 0

=== MEAN OVERLAP ===
N: 35,016
Inicio: 2023-01-02 12:00:00
Fin: 2024-12-31 23:30:00
Duplicados: 0
Saltos != 30 min: 0

=== NON-OVERLAPPING BLOCKS ===
N: 35,016
Inicio: 2023-01-02 12:00:00
Fin: 2024-12-31 23:30:00
Duplicados: 0
Saltos != 30 min: 0

=== MÉTRICAS POR ESTRATEGIA ===
              Strategy Variable     N       RMSE        MAE       Bias       R2
         shortest_lead      DNI 35016 151.832110  77.824414 -31.017012 0.726675
         shortest_lead      DHI 35016  50.670408  26.328453   3.073149 0.862336
         shortest_lead      GHI 35016  91.624346  45.744638 -18.576285 0.914808
          mean_overlap      DNI 35016 196.677194 107.515496 -53.289218 0.541372
          mean_overlap      DHI 35016  60.202416  31.7960

In [ ]:
# ============================================================
# 14. PI-LSTM V2.4 — DIAGNÓSTICO MENSUAL
#     Y DE PARTICIÓN RADIATIVA
# ============================================================


# ============================================================
# 1. COPIA DE TRABAJO
# ============================================================

diag_df = (
    non_overlapping_blocks_df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


# ============================================================
# 2. VARIABLES TEMPORALES AUXILIARES
# ============================================================

diag_df["year"] = (
    diag_df["timestamp"]
    .dt.year
)

diag_df["month"] = (
    diag_df["timestamp"]
    .dt.month
)

diag_df["hour"] = (
    diag_df["timestamp"]
    .dt.hour
    + diag_df["timestamp"].dt.minute / 60.0
)


# ============================================================
# 3. ERRORES INSTANTÁNEOS
# ============================================================

for variable in [
    "GHI",
    "DNI",
    "DHI",
]:

    diag_df[
        f"{variable}_error"
    ] = (
        diag_df[
            f"{variable}_pred"
        ]
        -
        diag_df[
            f"{variable}_obs"
        ]
    )


# ============================================================
# 4. FRACCIÓN DIFUSA OBSERVADA Y PREDICHA
# ============================================================

eps = 1e-6


diag_df["rd_obs"] = np.where(

    diag_df["GHI_obs"]
    > cfg.thr_cs,

    diag_df["DHI_obs"]
    /
    np.maximum(
        diag_df["GHI_obs"],
        eps
    ),

    np.nan
)


diag_df["rd_pred"] = np.where(

    diag_df["GHI_pred"]
    > cfg.thr_cs,

    diag_df["DHI_pred"]
    /
    np.maximum(
        diag_df["GHI_pred"],
        eps
    ),

    np.nan
)


diag_df["rd_error"] = (
    diag_df["rd_pred"]
    -
    diag_df["rd_obs"]
)


# ============================================================
# 5. RECUPERAR CLEAR-SKY DHI PARA CADA TIMESTAMP
#
# Se usa test_df como fuente física original.
# ============================================================

cs_reference = (
    test_df[
        [
            "timestamp",
            COL_CSDHI,
            COL_CSDNI,
            COL_CSGHI
        ]
    ]
    .copy()
)


cs_reference = cs_reference.rename(
    columns={

        COL_CSDHI:
            "DHI_cs",

        COL_CSDNI:
            "DNI_cs",

        COL_CSGHI:
            "GHI_cs",
    }
)


diag_df = diag_df.merge(

    cs_reference,

    on="timestamp",

    how="left",

    validate="one_to_one"
)


# ============================================================
# 6. kDHI OBSERVADO Y PREDICHO
# ============================================================

valid_kdhi = (
    diag_df["DHI_cs"]
    > cfg.thr_cs
)


diag_df["kDHI_obs"] = np.where(

    valid_kdhi,

    diag_df["DHI_obs"]
    /
    diag_df["DHI_cs"],

    np.nan
)


diag_df["kDHI_pred"] = np.where(

    valid_kdhi,

    diag_df["DHI_pred"]
    /
    diag_df["DHI_cs"],

    np.nan
)


diag_df["kDHI_error"] = (
    diag_df["kDHI_pred"]
    -
    diag_df["kDHI_obs"]
)


# ============================================================
# 7. DIAGNÓSTICO MENSUAL DE ENERGÍA
# ============================================================

monthly_rows = []


for (
    year,
    month
), g in diag_df.groupby(
    [
        "year",
        "month"
    ]
):

    row = {
        "year":
            int(year),

        "month":
            int(month),

        "N":
            int(len(g)),
    }


    for variable in [
        "GHI",
        "DNI",
        "DHI",
    ]:

        e_obs = energy_wh_m2(
            g[
                f"{variable}_obs"
            ]
        )

        e_pred = energy_wh_m2(
            g[
                f"{variable}_pred"
            ]
        )


        row[
            f"E_{variable}_obs_kWhm2"
        ] = (
            e_obs
            / 1000.0
        )

        row[
            f"E_{variable}_pred_kWhm2"
        ] = (
            e_pred
            / 1000.0
        )


        if e_obs > 0:

            row[
                f"{variable}_energy_bias_pct"
            ] = (
                100.0
                * (
                    e_pred
                    - e_obs
                )
                / e_obs
            )

        else:

            row[
                f"{variable}_energy_bias_pct"
            ] = np.nan


    # --------------------------------------------------------
    # Razón energética difusa mensual
    # --------------------------------------------------------

    ghi_obs_energy = energy_wh_m2(
        g["GHI_obs"]
    )

    ghi_pred_energy = energy_wh_m2(
        g["GHI_pred"]
    )

    dhi_obs_energy = energy_wh_m2(
        g["DHI_obs"]
    )

    dhi_pred_energy = energy_wh_m2(
        g["DHI_pred"]
    )


    row[
        "diffuse_energy_ratio_obs"
    ] = (
        dhi_obs_energy
        / ghi_obs_energy
        if ghi_obs_energy > 0
        else np.nan
    )


    row[
        "diffuse_energy_ratio_pred"
    ] = (
        dhi_pred_energy
        / ghi_pred_energy
        if ghi_pred_energy > 0
        else np.nan
    )


    row[
        "diffuse_energy_ratio_bias"
    ] = (
        row[
            "diffuse_energy_ratio_pred"
        ]
        -
        row[
            "diffuse_energy_ratio_obs"
        ]
    )


    monthly_rows.append(
        row
    )


monthly_diag_df = pd.DataFrame(
    monthly_rows
)


print(
    "=== DIAGNÓSTICO ENERGÉTICO MENSUAL ==="
)

print(
    monthly_diag_df[
        [
            "year",
            "month",

            "GHI_energy_bias_pct",
            "DNI_energy_bias_pct",
            "DHI_energy_bias_pct",

            "diffuse_energy_ratio_obs",
            "diffuse_energy_ratio_pred",
            "diffuse_energy_ratio_bias",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 8. RESUMEN MENSUAL PROMEDIADO 2023–2024
# ============================================================

monthly_climatology_df = (

    monthly_diag_df

    .groupby(
        "month",
        as_index=False
    )

    .agg({

        "GHI_energy_bias_pct":
            "mean",

        "DNI_energy_bias_pct":
            "mean",

        "DHI_energy_bias_pct":
            "mean",

        "diffuse_energy_ratio_obs":
            "mean",

        "diffuse_energy_ratio_pred":
            "mean",

        "diffuse_energy_ratio_bias":
            "mean",
    })
)


print(
    "\n=== PROMEDIO MENSUAL 2023–2024 ==="
)

print(
    monthly_climatology_df.to_string(
        index=False
    )
)


# ============================================================
# 9. BANDAS DE COSENO CENITAL
# ============================================================

cosz_bins = [
    -0.001,
    0.05,
    0.20,
    0.40,
    0.60,
    0.80,
    1.001,
]


cosz_labels = [
    "night/very_low",
    "0.05-0.20",
    "0.20-0.40",
    "0.40-0.60",
    "0.60-0.80",
    "0.80-1.00",
]


diag_df[
    "cosz_bin"
] = pd.cut(

    diag_df[
        "cosz_plus"
    ],

    bins=cosz_bins,

    labels=cosz_labels,

    include_lowest=True,

    right=False
)


cosz_rows = []


for band, g in diag_df.groupby(
    "cosz_bin",
    observed=True
):

    row = {
        "cosz_bin":
            str(band),

        "N":
            int(len(g)),
    }


    for variable in [
        "DNI",
        "DHI",
        "GHI",
    ]:

        m = regression_metrics(

            g[
                f"{variable}_obs"
            ],

            g[
                f"{variable}_pred"
            ]
        )

        row[
            f"{variable}_RMSE"
        ] = m["RMSE"]

        row[
            f"{variable}_MAE"
        ] = m["MAE"]

        row[
            f"{variable}_Bias"
        ] = m["Bias"]


    valid_ratio = (
        np.isfinite(
            g["rd_obs"]
        )
        &
        np.isfinite(
            g["rd_pred"]
        )
    )


    if valid_ratio.any():

        row[
            "rd_obs_mean"
        ] = float(
            g.loc[
                valid_ratio,
                "rd_obs"
            ].mean()
        )

        row[
            "rd_pred_mean"
        ] = float(
            g.loc[
                valid_ratio,
                "rd_pred"
            ].mean()
        )

        row[
            "rd_bias"
        ] = (
            row[
                "rd_pred_mean"
            ]
            -
            row[
                "rd_obs_mean"
            ]
        )

    else:

        row["rd_obs_mean"] = np.nan
        row["rd_pred_mean"] = np.nan
        row["rd_bias"] = np.nan


    cosz_rows.append(
        row
    )


cosz_diag_df = pd.DataFrame(
    cosz_rows
)


print(
    "\n=== DIAGNÓSTICO POR COSZ_PLUS ==="
)

print(
    cosz_diag_df.to_string(
        index=False
    )
)


# ============================================================
# 10. BANDAS DE FRACCIÓN DIFUSA OBSERVADA
# ============================================================

rd_bins = [
    0.0,
    0.20,
    0.40,
    0.60,
    0.80,
    1.01,
]


rd_labels = [
    "0.00-0.20",
    "0.20-0.40",
    "0.40-0.60",
    "0.60-0.80",
    "0.80-1.00",
]


diag_df[
    "rd_bin"
] = pd.cut(

    diag_df[
        "rd_obs"
    ],

    bins=rd_bins,

    labels=rd_labels,

    include_lowest=True,

    right=False
)


rd_rows = []


for band, g in diag_df.groupby(
    "rd_bin",
    observed=True
):

    row = {

        "rd_bin":
            str(band),

        "N":
            int(len(g)),

        "rd_obs_mean":
            float(
                g["rd_obs"].mean()
            ),

        "rd_pred_mean":
            float(
                g["rd_pred"].mean()
            ),
    }


    row[
        "rd_bias"
    ] = (
        row[
            "rd_pred_mean"
        ]
        -
        row[
            "rd_obs_mean"
        ]
    )


    for variable in [
        "DNI",
        "DHI",
        "GHI",
    ]:

        m = regression_metrics(

            g[
                f"{variable}_obs"
            ],

            g[
                f"{variable}_pred"
            ]
        )


        row[
            f"{variable}_Bias"
        ] = m["Bias"]

        row[
            f"{variable}_RMSE"
        ] = m["RMSE"]


    rd_rows.append(
        row
    )


rd_diag_df = pd.DataFrame(
    rd_rows
)


print(
    "\n=== DIAGNÓSTICO POR FRACCIÓN DIFUSA OBSERVADA ==="
)

print(
    rd_diag_df.to_string(
        index=False
    )
)


# ============================================================
# 11. BANDAS DE kDHI OBSERVADO
# ============================================================

kdhi_bins = [
    0.0,
    1.0,
    1.4,
    2.0,
    3.0,
    4.0,
    6.0,
    8.01,
]


kdhi_labels = [
    "0-1",
    "1-1.4",
    "1.4-2",
    "2-3",
    "3-4",
    "4-6",
    "6-8",
]


diag_df[
    "kDHI_bin"
] = pd.cut(

    diag_df[
        "kDHI_obs"
    ],

    bins=kdhi_bins,

    labels=kdhi_labels,

    include_lowest=True,

    right=False
)


kdhi_rows = []


for band, g in diag_df.groupby(
    "kDHI_bin",
    observed=True
):

    valid = (
        np.isfinite(
            g["kDHI_obs"]
        )
        &
        np.isfinite(
            g["kDHI_pred"]
        )
    )


    if not valid.any():
        continue


    ktrue = (
        g.loc[
            valid,
            "kDHI_obs"
        ]
        .to_numpy()
    )

    kpred = (
        g.loc[
            valid,
            "kDHI_pred"
        ]
        .to_numpy()
    )


    kdhi_rows.append({

        "kDHI_bin":
            str(band),

        "N":
            int(
                valid.sum()
            ),

        "kDHI_obs_mean":
            float(
                np.mean(
                    ktrue
                )
            ),

        "kDHI_pred_mean":
            float(
                np.mean(
                    kpred
                )
            ),

        "kDHI_bias":
            float(
                np.mean(
                    kpred
                    - ktrue
                )
            ),

        "kDHI_MAE":
            float(
                np.mean(
                    np.abs(
                        kpred
                        - ktrue
                    )
                )
            ),

        "kDHI_RMSE":
            float(
                np.sqrt(
                    np.mean(
                        (
                            kpred
                            - ktrue
                        ) ** 2
                    )
                )
            ),
    })


kdhi_diag_df = pd.DataFrame(
    kdhi_rows
)


print(
    "\n=== DIAGNÓSTICO POR BANDAS DE kDHI ==="
)

print(
    kdhi_diag_df.to_string(
        index=False
    )
)


# ============================================================
# 12. RESUMEN DE SIGNO DEL SESGO
#
# Nos interesa determinar si:
#
#   DNI_pred < DNI_obs
#   DHI_pred > DHI_obs
#
# ocurre de forma generalizada o solo en ciertos regímenes.
# ============================================================

print(
    "\n=== RESUMEN DEL SESGO ESTRUCTURAL ==="
)


print(
    "Bias global DNI:",
    float(
        diag_df[
            "DNI_error"
        ].mean()
    )
)

print(
    "Bias global DHI:",
    float(
        diag_df[
            "DHI_error"
        ].mean()
    )
)

print(
    "Bias global GHI:",
    float(
        diag_df[
            "GHI_error"
        ].mean()
    )
)


valid_rd_global = (
    np.isfinite(
        diag_df["rd_error"]
    )
)


print(
    "Bias global diffuse ratio:",
    float(
        diag_df.loc[
            valid_rd_global,
            "rd_error"
        ].mean()
    )
)


valid_kdhi_global = (
    np.isfinite(
        diag_df["kDHI_error"]
    )
)


print(
    "Bias global kDHI:",
    float(
        diag_df.loc[
            valid_kdhi_global,
            "kDHI_error"
        ].mean()
    )
)


# ============================================================
# 13. IDENTIFICAR MESES EXTREMOS
# ============================================================

def extreme_months(
    df_monthly,
    column
):

    valid = (
        df_monthly[
            column
        ]
        .dropna()
    )

    idx_min = valid.idxmin()
    idx_max = valid.idxmax()


    return {

        "min_year":
            int(
                df_monthly.loc[
                    idx_min,
                    "year"
                ]
            ),

        "min_month":
            int(
                df_monthly.loc[
                    idx_min,
                    "month"
                ]
            ),

        "min_value":
            float(
                df_monthly.loc[
                    idx_min,
                    column
                ]
            ),

        "max_year":
            int(
                df_monthly.loc[
                    idx_max,
                    "year"
                ]
            ),

        "max_month":
            int(
                df_monthly.loc[
                    idx_max,
                    "month"
                ]
            ),

        "max_value":
            float(
                df_monthly.loc[
                    idx_max,
                    column
                ]
            ),
    }


for variable in [
    "GHI",
    "DNI",
    "DHI",
]:

    col = (
        f"{variable}_energy_bias_pct"
    )

    ext = extreme_months(
        monthly_diag_df,
        col
    )


    print(
        f"\n{variable}:"
    )

    print(
        "  Sesgo energético mínimo:",
        ext[
            "min_year"
        ],
        ext[
            "min_month"
        ],
        f"{ext['min_value']:.3f}%"
    )

    print(
        "  Sesgo energético máximo:",
        ext[
            "max_year"
        ],
        ext[
            "max_month"
        ],
        f"{ext['max_value']:.3f}%"
    )


# ============================================================
# 14. VALIDACIONES
# ============================================================

assert len(
    diag_df
) == len(
    non_overlapping_blocks_df
)


assert (
    diag_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)


assert np.all(
    np.isfinite(
        diag_df[
            [
                "DNI_obs",
                "DNI_pred",

                "DHI_obs",
                "DHI_pred",

                "GHI_obs",
                "GHI_pred",

                "cosz_plus",
            ]
        ].to_numpy()
    )
)


# ============================================================
# 15. RESUMEN FINAL
# ============================================================

print(
    "\n✓ Diagnóstico mensual completado."
)

print(
    "✓ Sesgo energético mensual evaluado."
)

print(
    "✓ Dependencia con cosz_plus evaluada."
)

print(
    "✓ Dependencia con fracción difusa evaluada."
)

print(
    "✓ Distribución de kDHI evaluada por bandas."
)

print(
    "✓ Celda 14 completada correctamente."
)

=== DIAGNÓSTICO ENERGÉTICO MENSUAL ===
 year  month  GHI_energy_bias_pct  DNI_energy_bias_pct  DHI_energy_bias_pct  diffuse_energy_ratio_obs  diffuse_energy_ratio_pred  diffuse_energy_ratio_bias
 2023      1           -10.708695           -25.821514            10.714584                  0.455066                   0.564248                   0.109182
 2023      2           -11.744439           -28.102208             9.633839                  0.468896                   0.582478                   0.113581
 2023      3             2.681275             1.806536             3.363813                  0.617260                   0.621363                   0.004103
 2023      4            -6.072127           -22.238591            11.773478                  0.493227                   0.586937                   0.093710
 2023      5           -15.008791           -31.864920             7.343639                  0.425384                   0.537259                   0.111875
 2023      6           -1

In [ ]:
# ============================================================
# 15. PI-LSTM V2.4 — CALIBRACIÓN ISOTÓNICA
#     DE LA PARTICIÓN RADIATIVA
#
# PRINCIPIO METODOLÓGICO:
#
#   VALIDATION 2020–2022:
#       ajustar calibrador
#
#   TEST 2023–2024:
#       aplicar calibrador congelado
#
# El TEST no interviene en el ajuste.
# ============================================================


from sklearn.isotonic import IsotonicRegression


# ============================================================
# 1. RUTAS DE SALIDA
# ============================================================

PARTITION_CALIBRATOR_PATH = Path(
    "partition_isotonic_v24.joblib"
)

CALIBRATION_METRICS_PATH = Path(
    "test_calibration_metrics_v24.csv"
)

CALIBRATED_TEST_PATH = Path(
    "test_non_overlapping_calibrated_v24.csv"
)


# ============================================================
# 2. NÚMERO EXACTO DE BATCHES DE VALIDATION
#
# ds_val tiene repeat=True.
# Por tanto, usamos take() para consumir exactamente
# una pasada completa por VALIDATION.
# ============================================================

val_steps_exact = int(
    np.ceil(
        nwin_val
        / cfg.batch_size
    )
)


print(
    "=== VALIDATION PARA CALIBRACIÓN ==="
)

print(
    "Validation windows:",
    f"{nwin_val:,}"
)

print(
    "Validation batches:",
    val_steps_exact
)


# ============================================================
# 3. PREDICCIONES COMPLETAS SOBRE VALIDATION
#
# IMPORTANTE:
# No se entrena nada en esta etapa.
# ============================================================

val_pred_batches = []
val_true_batches = []


for (
    (
        xh_batch,
        xf_batch
    ),
    y_batch
) in ds_val.take(
    val_steps_exact
):

    pred_batch = best_backbone(
        [
            xh_batch,
            xf_batch
        ],
        training=False
    )

    val_pred_batches.append(
        pred_batch.numpy()
    )

    val_true_batches.append(
        y_batch.numpy()
    )


Y_pred_val_windows = np.concatenate(
    val_pred_batches,
    axis=0
)

Y_true_val_windows = np.concatenate(
    val_true_batches,
    axis=0
)


# Protección adicional por si existiera
# cualquier batch excedente.

Y_pred_val_windows = (
    Y_pred_val_windows[
        :nwin_val
    ]
)

Y_true_val_windows = (
    Y_true_val_windows[
        :nwin_val
    ]
)


print(
    "\n=== MATRICES VALIDATION ==="
)

print(
    "Y_pred_val_windows:",
    Y_pred_val_windows.shape
)

print(
    "Y_true_val_windows:",
    Y_true_val_windows.shape
)


assert Y_pred_val_windows.shape == (
    nwin_val,
    cfg.H,
    2
)

assert Y_true_val_windows.shape == (
    nwin_val,
    cfg.H,
    len(Y_EXT_COLS)
)


# ============================================================
# 4. RECONSTRUCCIÓN FÍSICA DE VALIDATION
# ============================================================

kdni_pred_val = (
    Y_pred_val_windows[
        ...,
        0
    ]
)

qdhi_pred_val = (
    Y_pred_val_windows[
        ...,
        1
    ]
)

kdhi_pred_val = np.expm1(
    qdhi_pred_val
)


# Variables físicas auxiliares desde Y_EXT

csdni_val = (
    Y_true_val_windows[
        ...,
        3
    ]
)

csdhi_val = (
    Y_true_val_windows[
        ...,
        4
    ]
)

cosz_val_windows = (
    Y_true_val_windows[
        ...,
        6
    ]
)


dni_pred_val = (
    kdni_pred_val
    * csdni_val
)

dhi_pred_val = (
    kdhi_pred_val
    * csdhi_val
)

ghi_pred_val = (
    dni_pred_val
    * cosz_val_windows
    + dhi_pred_val
)


# ============================================================
# 5. ÍNDICES TEMPORALES DE VALIDATION
# ============================================================

window_starts_val = np.arange(
    0,
    len(val_df)
    - cfg.L
    - cfg.H
    + 1,
    cfg.stride,
    dtype=np.int64
)


assert len(
    window_starts_val
) == nwin_val


future_idx_val = (

    window_starts_val[
        :,
        None
    ]

    + cfg.L

    + np.arange(
        cfg.H,
        dtype=np.int64
    )[
        None,
        :
    ]
)


assert future_idx_val.shape == (
    nwin_val,
    cfg.H
)


# ============================================================
# 6. NON-OVERLAPPING BLOCKS EN VALIDATION
#
# Igual que en TEST:
#
# H / stride = 24 / 2 = 12
#
# Se selecciona una ventana cada 12 orígenes.
#
# Esto NO es rollout autorregresivo.
# ============================================================

if cfg.H % cfg.stride != 0:

    raise ValueError(
        "H debe ser divisible por stride."
    )


window_jump_val = (
    cfg.H
    // cfg.stride
)


non_overlap_val_ids = np.arange(
    0,
    nwin_val,
    window_jump_val,
    dtype=int
)


# ============================================================
# 7. CONSTRUIR SERIE CRONOLÓGICA VALIDATION
# ============================================================

val_timestamps = (
    val_df["timestamp"]
    .to_numpy()
)


val_records = []


for i in non_overlap_val_ids:

    for h in range(cfg.H):

        target_idx = int(
            future_idx_val[
                i,
                h
            ]
        )


        val_records.append({

            "timestamp":
                val_timestamps[
                    target_idx
                ],

            "target_idx":
                target_idx,

            "horizon_step":
                h + 1,

            "cosz_plus":
                float(
                    cosz_val_windows[
                        i,
                        h
                    ]
                ),

            "DNI_pred":
                float(
                    dni_pred_val[
                        i,
                        h
                    ]
                ),

            "DHI_pred":
                float(
                    dhi_pred_val[
                        i,
                        h
                    ]
                ),

            "GHI_pred":
                float(
                    ghi_pred_val[
                        i,
                        h
                    ]
                ),
        })


val_non_overlapping_df = pd.DataFrame(
    val_records
)


val_non_overlapping_df = (

    val_non_overlapping_df

    .sort_values(
        "timestamp"
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. OBSERVACIONES REALES DE VALIDATION
# ============================================================

val_obs_reference = (

    val_df[
        [
            "timestamp",
            COL_DNI,
            COL_DHI,
            COL_GHI,
            "cosz_plus"
        ]
    ]

    .copy()

    .rename(
        columns={

            COL_DNI:
                "DNI_obs",

            COL_DHI:
                "DHI_obs",

            COL_GHI:
                "GHI_obs",

            "cosz_plus":
                "cosz_obs",
        }
    )
)


val_non_overlapping_df = (
    val_non_overlapping_df.merge(

        val_obs_reference,

        on="timestamp",

        how="left",

        validate="one_to_one"
    )
)


print(
    "\n=== NON-OVERLAPPING VALIDATION ==="
)

print(
    "N:",
    f"{len(val_non_overlapping_df):,}"
)

print(
    "Inicio:",
    val_non_overlapping_df[
        "timestamp"
    ].min()
)

print(
    "Fin:",
    val_non_overlapping_df[
        "timestamp"
    ].max()
)

print(
    "Duplicados:",
    int(
        val_non_overlapping_df[
            "timestamp"
        ]
        .duplicated()
        .sum()
    )
)


# ============================================================
# 9. FRACCIÓN DIFUSA RAW EN VALIDATION
# ============================================================

EPS_CAL = 1e-6


val_non_overlapping_df[
    "rd_obs"
] = np.where(

    val_non_overlapping_df[
        "GHI_obs"
    ] > cfg.thr_cs,

    val_non_overlapping_df[
        "DHI_obs"
    ]
    /
    np.maximum(
        val_non_overlapping_df[
            "GHI_obs"
        ],
        EPS_CAL
    ),

    np.nan
)


val_non_overlapping_df[
    "rd_pred_raw"
] = np.where(

    val_non_overlapping_df[
        "GHI_pred"
    ] > cfg.thr_cs,

    val_non_overlapping_df[
        "DHI_pred"
    ]
    /
    np.maximum(
        val_non_overlapping_df[
            "GHI_pred"
        ],
        EPS_CAL
    ),

    np.nan
)


# ============================================================
# 10. MÁSCARA DE AJUSTE DEL CALIBRADOR
#
# Solo utilizamos:
#
#   - periodo VALIDATION
#   - régimen diurno
#   - GHI observado > threshold
#   - GHI predicho > threshold
#   - ratios finitos
#
# De este modo evitamos ajustar la calibración con
# noche o irradiancias muy pequeñas.
# ============================================================

cal_fit_mask = (

    (
        val_non_overlapping_df[
            "cosz_plus"
        ]
        > cfg.thr_day_cosz
    )

    &

    (
        val_non_overlapping_df[
            "GHI_obs"
        ]
        > cfg.thr_cs
    )

    &

    (
        val_non_overlapping_df[
            "GHI_pred"
        ]
        > cfg.thr_cs
    )

    &

    np.isfinite(
        val_non_overlapping_df[
            "rd_obs"
        ]
    )

    &

    np.isfinite(
        val_non_overlapping_df[
            "rd_pred_raw"
        ]
    )
)


x_cal = (

    val_non_overlapping_df.loc[
        cal_fit_mask,
        "rd_pred_raw"
    ]

    .to_numpy(
        dtype=np.float64
    )
)


y_cal = (

    val_non_overlapping_df.loc[
        cal_fit_mask,
        "rd_obs"
    ]

    .to_numpy(
        dtype=np.float64
    )
)


# Por coherencia física:
# 0 <= rd <= 1

x_cal = np.clip(
    x_cal,
    0.0,
    1.0
)

y_cal = np.clip(
    y_cal,
    0.0,
    1.0
)


print(
    "\n=== DATOS PARA CALIBRACIÓN ==="
)

print(
    "N calibración:",
    f"{len(x_cal):,}"
)

print(
    "rd_pred raw min/max:",
    float(
        x_cal.min()
    ),
    float(
        x_cal.max()
    )
)

print(
    "rd_obs min/max:",
    float(
        y_cal.min()
    ),
    float(
        y_cal.max()
    )
)


# ============================================================
# 11. AJUSTAR REGRESIÓN ISOTÓNICA
#
# IMPORTANTE:
# El calibrador se ajusta EXCLUSIVAMENTE con VALIDATION.
# ============================================================

partition_calibrator = IsotonicRegression(

    increasing=True,

    y_min=0.0,

    y_max=1.0,

    out_of_bounds="clip"
)


partition_calibrator.fit(
    x_cal,
    y_cal
)


print(
    "\n=== CALIBRADOR ISOTÓNICO AJUSTADO ==="
)

print(
    "N knots:",
    len(
        partition_calibrator.X_thresholds_
    )
)

print(
    "X mínimo:",
    float(
        partition_calibrator.X_thresholds_[0]
    )
)

print(
    "X máximo:",
    float(
        partition_calibrator.X_thresholds_[-1]
    )
)

print(
    "Y mínimo:",
    float(
        partition_calibrator.y_thresholds_[0]
    )
)

print(
    "Y máximo:",
    float(
        partition_calibrator.y_thresholds_[-1]
    )
)


# ============================================================
# 12. GUARDAR CALIBRADOR CONGELADO
# ============================================================

joblib.dump(
    partition_calibrator,
    PARTITION_CALIBRATOR_PATH
)


print(
    "\n✓ Calibrador guardado:"
)

print(
    PARTITION_CALIBRATOR_PATH
)


# ============================================================
# 13. AUDITORÍA DE LA FUNCIÓN DE CALIBRACIÓN
# ============================================================

calibration_grid = np.array(
    [
        0.00,
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        1.00,
    ],
    dtype=np.float64
)


calibration_grid_pred = (
    partition_calibrator.predict(
        calibration_grid
    )
)


calibration_map_df = pd.DataFrame({

    "rd_raw":
        calibration_grid,

    "rd_calibrated":
        calibration_grid_pred,

    "correction":
        (
            calibration_grid_pred
            - calibration_grid
        ),
})


print(
    "\n=== MAPA DE CALIBRACIÓN ==="
)

print(
    calibration_map_df.to_string(
        index=False
    )
)


# ============================================================
# 14. DESEMPEÑO DEL CALIBRADOR EN VALIDATION
#
# Esto describe el ajuste.
# NO constituye evaluación final independiente.
# ============================================================

rd_val_cal = (
    partition_calibrator.predict(
        x_cal
    )
)


val_ratio_raw_metrics = regression_metrics(
    y_cal,
    x_cal
)

val_ratio_cal_metrics = regression_metrics(
    y_cal,
    rd_val_cal
)


print(
    "\n=== PARTICIÓN EN VALIDATION ==="
)

print(
    "RAW:"
)

print(
    "  RMSE:",
    val_ratio_raw_metrics[
        "RMSE"
    ]
)

print(
    "  MAE :",
    val_ratio_raw_metrics[
        "MAE"
    ]
)

print(
    "  Bias:",
    val_ratio_raw_metrics[
        "Bias"
    ]
)


print(
    "\nCALIBRATED:"
)

print(
    "  RMSE:",
    val_ratio_cal_metrics[
        "RMSE"
    ]
)

print(
    "  MAE :",
    val_ratio_cal_metrics[
        "MAE"
    ]
)

print(
    "  Bias:",
    val_ratio_cal_metrics[
        "Bias"
    ]
)


# ============================================================
# 15. PREPARAR TEST PARA APLICACIÓN DEL CALIBRADOR
#
# A partir de aquí:
#
# EL CALIBRADOR YA ESTÁ CONGELADO.
#
# No se hace fit() ni ajuste alguno con TEST.
# ============================================================

test_cal_df = (
    non_overlapping_blocks_df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


test_cal_df[
    "rd_pred_raw"
] = np.where(

    test_cal_df[
        "GHI_pred"
    ] > cfg.thr_cs,

    test_cal_df[
        "DHI_pred"
    ]
    /
    np.maximum(
        test_cal_df[
            "GHI_pred"
        ],
        EPS_CAL
    ),

    np.nan
)


test_cal_df[
    "rd_obs"
] = np.where(

    test_cal_df[
        "GHI_obs"
    ] > cfg.thr_cs,

    test_cal_df[
        "DHI_obs"
    ]
    /
    np.maximum(
        test_cal_df[
            "GHI_obs"
        ],
        EPS_CAL
    ),

    np.nan
)


# ============================================================
# 16. MÁSCARA DE APLICACIÓN EN TEST
#
# La calibración solo actúa en el dominio en el que fue
# definida físicamente.
#
# Fuera de este dominio se conservan las predicciones raw.
# ============================================================

cal_apply_mask = (

    (
        test_cal_df[
            "cosz_plus"
        ]
        > cfg.thr_day_cosz
    )

    &

    (
        test_cal_df[
            "GHI_pred"
        ]
        > cfg.thr_cs
    )

    &

    np.isfinite(
        test_cal_df[
            "rd_pred_raw"
        ]
    )
)


print(
    "\n=== APLICACIÓN EN TEST ==="
)

print(
    "N total:",
    f"{len(test_cal_df):,}"
)

print(
    "N calibrado:",
    f"{int(cal_apply_mask.sum()):,}"
)

print(
    "Fracción calibrada:",
    float(
        cal_apply_mask.mean()
    )
)


# ============================================================
# 17. APLICAR CALIBRADOR CONGELADO AL TEST
# ============================================================

test_cal_df[
    "rd_cal"
] = (
    test_cal_df[
        "rd_pred_raw"
    ]
)


x_test_cal = (

    test_cal_df.loc[
        cal_apply_mask,
        "rd_pred_raw"
    ]

    .to_numpy(
        dtype=np.float64
    )
)


x_test_cal = np.clip(
    x_test_cal,
    0.0,
    1.0
)


rd_test_calibrated = (
    partition_calibrator.predict(
        x_test_cal
    )
)


rd_test_calibrated = np.clip(
    rd_test_calibrated,
    0.0,
    1.0
)


test_cal_df.loc[
    cal_apply_mask,
    "rd_cal"
] = (
    rd_test_calibrated
)


# ============================================================
# 18. RECONSTRUCCIÓN FÍSICA CALIBRADA
#
# GHI NO SE MODIFICA.
#
# DHI_cal = rd_cal * GHI_pred
#
# DNI_cal = (GHI_pred - DHI_cal) / cosz
#
# únicamente en el dominio calibrado.
#
# Fuera de dicho dominio:
#
# DNI_cal = DNI_pred
# DHI_cal = DHI_pred
# ============================================================

test_cal_df[
    "GHI_cal"
] = (
    test_cal_df[
        "GHI_pred"
    ]
)


test_cal_df[
    "DHI_cal"
] = (
    test_cal_df[
        "DHI_pred"
    ]
)


test_cal_df[
    "DNI_cal"
] = (
    test_cal_df[
        "DNI_pred"
    ]
)


# DHI calibrado

test_cal_df.loc[
    cal_apply_mask,
    "DHI_cal"
] = (

    test_cal_df.loc[
        cal_apply_mask,
        "rd_cal"
    ]

    *

    test_cal_df.loc[
        cal_apply_mask,
        "GHI_cal"
    ]
)


# DNI calibrado

test_cal_df.loc[
    cal_apply_mask,
    "DNI_cal"
] = (

    (
        test_cal_df.loc[
            cal_apply_mask,
            "GHI_cal"
        ]

        -

        test_cal_df.loc[
            cal_apply_mask,
            "DHI_cal"
        ]
    )

    /

    test_cal_df.loc[
        cal_apply_mask,
        "cosz_plus"
    ]
)


# Protección numérica

test_cal_df[
    "DNI_cal"
] = np.maximum(
    test_cal_df[
        "DNI_cal"
    ],
    0.0
)

test_cal_df[
    "DHI_cal"
] = np.maximum(
    test_cal_df[
        "DHI_cal"
    ],
    0.0
)

test_cal_df[
    "GHI_cal"
] = np.maximum(
    test_cal_df[
        "GHI_cal"
    ],
    0.0
)


# ============================================================
# 19. CIERRE RADIATIVO CALIBRADO
# ============================================================

closure_cal = (

    test_cal_df[
        "GHI_cal"
    ].to_numpy()

    -

    (
        test_cal_df[
            "DNI_cal"
        ].to_numpy()

        *

        test_cal_df[
            "cosz_plus"
        ].to_numpy()

        +

        test_cal_df[
            "DHI_cal"
        ].to_numpy()
    )
)


closure_cal_mae = float(
    np.mean(
        np.abs(
            closure_cal
        )
    )
)

closure_cal_max = float(
    np.max(
        np.abs(
            closure_cal
        )
    )
)


print(
    "\n=== CIERRE RADIATIVO CALIBRADO ==="
)

print(
    "Closure MAE:",
    closure_cal_mae
)

print(
    "Closure max abs:",
    closure_cal_max
)


# ============================================================
# 20. MÉTRICAS TEST: RAW VS CALIBRATED
# ============================================================

comparison_rows = []


for variable in [
    "DNI",
    "DHI",
    "GHI",
]:

    # --------------------------------------------------------
    # RAW
    # --------------------------------------------------------

    m_raw = regression_metrics(

        test_cal_df[
            f"{variable}_obs"
        ],

        test_cal_df[
            f"{variable}_pred"
        ]
    )


    comparison_rows.append({

        "Version":
            "V2.4_raw",

        "Variable":
            variable,

        **m_raw
    })


    # --------------------------------------------------------
    # CALIBRATED
    # --------------------------------------------------------

    m_cal = regression_metrics(

        test_cal_df[
            f"{variable}_obs"
        ],

        test_cal_df[
            f"{variable}_cal"
        ]
    )


    comparison_rows.append({

        "Version":
            "V2.4_calibrated",

        "Variable":
            variable,

        **m_cal
    })


test_calibration_metrics_df = pd.DataFrame(
    comparison_rows
)


print(
    "\n=== TEST RAW VS CALIBRATED ==="
)

print(
    test_calibration_metrics_df.to_string(
        index=False
    )
)


# ============================================================
# 21. MÉTRICAS DIURNAS TEST
# ============================================================

test_day_mask = (

    test_cal_df[
        "cosz_plus"
    ]
    > cfg.thr_day_cosz
)


day_comparison_rows = []


for variable in [
    "DNI",
    "DHI",
    "GHI",
]:

    m_raw = regression_metrics(

        test_cal_df.loc[
            test_day_mask,
            f"{variable}_obs"
        ],

        test_cal_df.loc[
            test_day_mask,
            f"{variable}_pred"
        ]
    )


    m_cal = regression_metrics(

        test_cal_df.loc[
            test_day_mask,
            f"{variable}_obs"
        ],

        test_cal_df.loc[
            test_day_mask,
            f"{variable}_cal"
        ]
    )


    day_comparison_rows.append({

        "Version":
            "V2.4_raw",

        "Variable":
            variable,

        **m_raw
    })


    day_comparison_rows.append({

        "Version":
            "V2.4_calibrated",

        "Variable":
            variable,

        **m_cal
    })


test_day_calibration_metrics_df = (
    pd.DataFrame(
        day_comparison_rows
    )
)


print(
    "\n=== TEST DIURNO RAW VS CALIBRATED ==="
)

print(
    test_day_calibration_metrics_df.to_string(
        index=False
    )
)


# ============================================================
# 22. PARTICIÓN DIFUSA TEST
# ============================================================

ratio_test_mask = (

    np.isfinite(
        test_cal_df[
            "rd_obs"
        ]
    )

    &

    np.isfinite(
        test_cal_df[
            "rd_pred_raw"
        ]
    )

    &

    np.isfinite(
        test_cal_df[
            "rd_cal"
        ]
    )

    &

    (
        test_cal_df[
            "GHI_obs"
        ]
        > cfg.thr_cs
    )

    &

    (
        test_cal_df[
            "GHI_pred"
        ]
        > cfg.thr_cs
    )
)


rd_obs_test = (

    test_cal_df.loc[
        ratio_test_mask,
        "rd_obs"
    ]

    .to_numpy(
        dtype=np.float64
    )
)


rd_raw_test = (

    test_cal_df.loc[
        ratio_test_mask,
        "rd_pred_raw"
    ]

    .to_numpy(
        dtype=np.float64
    )
)


rd_cal_test = (

    test_cal_df.loc[
        ratio_test_mask,
        "rd_cal"
    ]

    .to_numpy(
        dtype=np.float64
    )
)


ratio_raw_test_metrics = (
    regression_metrics(
        rd_obs_test,
        rd_raw_test
    )
)


ratio_cal_test_metrics = (
    regression_metrics(
        rd_obs_test,
        rd_cal_test
    )
)


print(
    "\n=== PARTICIÓN DIFUSA TEST ==="
)

print(
    "OBS mean:",
    float(
        np.mean(
            rd_obs_test
        )
    )
)

print(
    "\nRAW:"
)

print(
    "  mean:",
    float(
        np.mean(
            rd_raw_test
        )
    )
)

print(
    "  RMSE:",
    ratio_raw_test_metrics[
        "RMSE"
    ]
)

print(
    "  MAE :",
    ratio_raw_test_metrics[
        "MAE"
    ]
)

print(
    "  Bias:",
    ratio_raw_test_metrics[
        "Bias"
    ]
)


print(
    "\nCALIBRATED:"
)

print(
    "  mean:",
    float(
        np.mean(
            rd_cal_test
        )
    )
)

print(
    "  RMSE:",
    ratio_cal_test_metrics[
        "RMSE"
    ]
)

print(
    "  MAE :",
    ratio_cal_test_metrics[
        "MAE"
    ]
)

print(
    "  Bias:",
    ratio_cal_test_metrics[
        "Bias"
    ]
)


# ============================================================
# 23. BALANCE ENERGÉTICO TEST
# ============================================================

energy_cal_rows = []


for version, suffix in [

    (
        "V2.4_raw",
        "pred"
    ),

    (
        "V2.4_calibrated",
        "cal"
    ),

]:

    row = {
        "Version":
            version
    }


    for variable in [
        "GHI",
        "DNI",
        "DHI",
    ]:

        e_obs = energy_wh_m2(

            test_cal_df[
                f"{variable}_obs"
            ]
        )


        e_model = energy_wh_m2(

            test_cal_df[
                f"{variable}_{suffix}"
            ]
        )


        row[
            f"E_{variable}_obs_kWhm2"
        ] = (
            e_obs
            / 1000.0
        )


        row[
            f"E_{variable}_model_kWhm2"
        ] = (
            e_model
            / 1000.0
        )


        row[
            f"{variable}_energy_bias_pct"
        ] = (

            100.0
            * (
                e_model
                - e_obs
            )
            / e_obs
        )


    row[
        "diffuse_energy_ratio_obs"
    ] = (

        row[
            "E_DHI_obs_kWhm2"
        ]
        /
        row[
            "E_GHI_obs_kWhm2"
        ]
    )


    row[
        "diffuse_energy_ratio_model"
    ] = (

        row[
            "E_DHI_model_kWhm2"
        ]
        /
        row[
            "E_GHI_model_kWhm2"
        ]
    )


    row[
        "diffuse_energy_ratio_bias"
    ] = (

        row[
            "diffuse_energy_ratio_model"
        ]
        -
        row[
            "diffuse_energy_ratio_obs"
        ]
    )


    energy_cal_rows.append(
        row
    )


test_energy_calibration_df = (
    pd.DataFrame(
        energy_cal_rows
    )
)


print(
    "\n=== BALANCE ENERGÉTICO TEST ==="
)

print(
    test_energy_calibration_df.to_string(
        index=False
    )
)


# ============================================================
# 24. CAMBIO RELATIVO DE MÉTRICAS
# ============================================================

metric_improvement_rows = []


for variable in [
    "DNI",
    "DHI",
    "GHI",
]:

    raw_row = (
        test_calibration_metrics_df[
            (
                test_calibration_metrics_df[
                    "Version"
                ] == "V2.4_raw"
            )
            &
            (
                test_calibration_metrics_df[
                    "Variable"
                ] == variable
            )
        ]
        .iloc[0]
    )


    cal_row = (
        test_calibration_metrics_df[
            (
                test_calibration_metrics_df[
                    "Version"
                ]
                == "V2.4_calibrated"
            )
            &
            (
                test_calibration_metrics_df[
                    "Variable"
                ] == variable
            )
        ]
        .iloc[0]
    )


    metric_improvement_rows.append({

        "Variable":
            variable,

        "RMSE_raw":
            raw_row["RMSE"],

        "RMSE_cal":
            cal_row["RMSE"],

        "RMSE_change_pct":
            (
                100.0
                * (
                    cal_row["RMSE"]
                    - raw_row["RMSE"]
                )
                / raw_row["RMSE"]
            ),

        "MAE_raw":
            raw_row["MAE"],

        "MAE_cal":
            cal_row["MAE"],

        "MAE_change_pct":
            (
                100.0
                * (
                    cal_row["MAE"]
                    - raw_row["MAE"]
                )
                / raw_row["MAE"]
            ),

        "Bias_raw":
            raw_row["Bias"],

        "Bias_cal":
            cal_row["Bias"],

        "R2_raw":
            raw_row["R2"],

        "R2_cal":
            cal_row["R2"],
    })


metric_improvement_df = pd.DataFrame(
    metric_improvement_rows
)


print(
    "\n=== CAMBIO DE MÉTRICAS TRAS CALIBRACIÓN ==="
)

print(
    metric_improvement_df.to_string(
        index=False
    )
)


# ============================================================
# 25. GUARDAR RESULTADOS
# ============================================================

test_calibration_metrics_df.to_csv(
    CALIBRATION_METRICS_PATH,
    index=False
)


test_cal_df.to_csv(
    CALIBRATED_TEST_PATH,
    index=False
)


print(
    "\n=== ARCHIVOS GUARDADOS ==="
)

print(
    "Calibrador:",
    PARTITION_CALIBRATOR_PATH
)

print(
    "Métricas:",
    CALIBRATION_METRICS_PATH
)

print(
    "Serie test calibrada:",
    CALIBRATED_TEST_PATH
)


# ============================================================
# 26. VALIDACIONES FINALES
# ============================================================

assert (
    val_non_overlapping_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)


assert (
    test_cal_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)


assert np.all(
    np.isfinite(
        test_cal_df[
            [
                "DNI_cal",
                "DHI_cal",
                "GHI_cal",
            ]
        ].to_numpy()
    )
)


assert (
    test_cal_df[
        "DNI_cal"
    ].min()
    >= -1e-8
)

assert (
    test_cal_df[
        "DHI_cal"
    ].min()
    >= -1e-8
)

assert (
    test_cal_df[
        "GHI_cal"
    ].min()
    >= -1e-8
)


# Debido a redondeo float pueden existir
# residuos del orden de 1e-4 W/m².

assert closure_cal_mae < 1e-3


print(
    "\n✓ Calibrador ajustado exclusivamente con VALIDATION."
)

print(
    "✓ Calibrador congelado antes de utilizar TEST."
)

print(
    "✓ TEST 2023–2024 permaneció fuera del ajuste."
)

print(
    "✓ GHI se mantuvo sin modificación."
)

print(
    "✓ Partición DNI–DHI recalibrada."
)

print(
    "✓ Cierre radiativo preservado."
)

print(
    "✓ Celda 15 completada correctamente."
)

=== VALIDATION PARA CALIBRACIÓN ===
Validation windows: 26,257
Validation batches: 821

=== MATRICES VALIDATION ===
Y_pred_val_windows: (26257, 24, 2)
Y_true_val_windows: (26257, 24, 8)

=== NON-OVERLAPPING VALIDATION ===
N: 52,536
Inicio: 2020-01-02 12:00:00
Fin: 2022-12-31 23:30:00
Duplicados: 0

=== DATOS PARA CALIBRACIÓN ===
N calibración: 25,024
rd_pred raw min/max: 0.06671742986107389 0.9913158916579262
rd_obs min/max: 0.07479508196721311 1.0

=== CALIBRADOR ISOTÓNICO AJUSTADO ===
N knots: 209
X mínimo: 0.06671742986107389
X máximo: 0.9913158916579262
Y mínimo: 0.09117618789080055
Y máximo: 1.0

✓ Calibrador guardado:
partition_isotonic_v24.joblib

=== MAPA DE CALIBRACIÓN ===
 rd_raw  rd_calibrated  correction
    0.0       0.091176    0.091176
    0.1       0.109033    0.009033
    0.2       0.202117    0.002117
    0.3       0.316469    0.016469
    0.4       0.400024    0.000024
    0.5       0.476366   -0.023634
    0.6       0.536523   -0.063477
    0.7       0.641585   -0.0

In [ ]:
# ============================================================
# 16. PI-LSTM V2.4 — DIAGNÓSTICO POST-CALIBRACIÓN
#     EN TEST 2023–2024
#
# OBJETIVO:
#
# Comparar V2.4 RAW vs V2.4 CALIBRATED según:
#
#   1. mes
#   2. cosz_plus
#   3. fracción difusa observada
#   4. kDHI observado
#
# IMPORTANTE:
#
# Esta celda NO ajusta ni modifica el calibrador.
# TEST se utiliza únicamente para evaluación independiente.
# ============================================================


# ============================================================
# 1. COPIA DE TRABAJO
# ============================================================

postcal_df = (
    test_cal_df
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)


print(
    "=== DIAGNÓSTICO POST-CALIBRACIÓN ==="
)

print(
    "N:",
    f"{len(postcal_df):,}"
)

print(
    "Inicio:",
    postcal_df["timestamp"].min()
)

print(
    "Fin:",
    postcal_df["timestamp"].max()
)


assert (
    postcal_df["timestamp"]
    .duplicated()
    .sum()
    == 0
)


# ============================================================
# 2. VARIABLES TEMPORALES
# ============================================================

postcal_df["year"] = (
    postcal_df["timestamp"]
    .dt.year
)

postcal_df["month"] = (
    postcal_df["timestamp"]
    .dt.month
)


# ============================================================
# 3. RECUPERAR CLEAR-SKY PARA TEST
#
# Necesario para evaluar kDHI raw y calibrated.
# ============================================================

postcal_cs_reference = (

    test_df[
        [
            "timestamp",
            COL_CSDHI,
            COL_CSDNI,
            COL_CSGHI,
        ]
    ]

    .copy()

    .rename(
        columns={
            COL_CSDHI:
                "DHI_cs",

            COL_CSDNI:
                "DNI_cs",

            COL_CSGHI:
                "GHI_cs",
        }
    )
)


# Evitar columnas duplicadas si la celda se ejecuta
# más de una vez.

for col in [
    "DHI_cs",
    "DNI_cs",
    "GHI_cs",
]:

    if col in postcal_df.columns:

        postcal_df = (
            postcal_df.drop(
                columns=col
            )
        )


postcal_df = postcal_df.merge(

    postcal_cs_reference,

    on="timestamp",

    how="left",

    validate="one_to_one"
)


# ============================================================
# 4. VARIABLES DE ERROR RAW Y CALIBRATED
# ============================================================

for variable in [
    "DNI",
    "DHI",
    "GHI",
]:

    postcal_df[
        f"{variable}_error_raw"
    ] = (

        postcal_df[
            f"{variable}_pred"
        ]

        -

        postcal_df[
            f"{variable}_obs"
        ]
    )


    postcal_df[
        f"{variable}_error_cal"
    ] = (

        postcal_df[
            f"{variable}_cal"
        ]

        -

        postcal_df[
            f"{variable}_obs"
        ]
    )


# ============================================================
# 5. FRACCIONES DIFUSAS
#
# El régimen se clasifica mediante rd_obs.
# De esta manera RAW y CALIBRATED se comparan exactamente
# sobre los mismos estados físicos observados.
# ============================================================

EPS_POSTCAL = 1e-6


postcal_df["rd_obs_eval"] = np.where(

    postcal_df["GHI_obs"]
    > cfg.thr_cs,

    postcal_df["DHI_obs"]
    /
    np.maximum(
        postcal_df["GHI_obs"],
        EPS_POSTCAL
    ),

    np.nan
)


postcal_df["rd_raw_eval"] = np.where(

    postcal_df["GHI_pred"]
    > cfg.thr_cs,

    postcal_df["DHI_pred"]
    /
    np.maximum(
        postcal_df["GHI_pred"],
        EPS_POSTCAL
    ),

    np.nan
)


postcal_df["rd_cal_eval"] = np.where(

    postcal_df["GHI_cal"]
    > cfg.thr_cs,

    postcal_df["DHI_cal"]
    /
    np.maximum(
        postcal_df["GHI_cal"],
        EPS_POSTCAL
    ),

    np.nan
)


postcal_df["rd_error_raw"] = (
    postcal_df["rd_raw_eval"]
    -
    postcal_df["rd_obs_eval"]
)


postcal_df["rd_error_cal"] = (
    postcal_df["rd_cal_eval"]
    -
    postcal_df["rd_obs_eval"]
)


# ============================================================
# 6. kDHI OBSERVADO, RAW Y CALIBRATED
# ============================================================

valid_kdhi_postcal = (

    postcal_df["DHI_cs"]
    > cfg.thr_cs
)


postcal_df["kDHI_obs_eval"] = np.where(

    valid_kdhi_postcal,

    postcal_df["DHI_obs"]
    /
    postcal_df["DHI_cs"],

    np.nan
)


postcal_df["kDHI_raw_eval"] = np.where(

    valid_kdhi_postcal,

    postcal_df["DHI_pred"]
    /
    postcal_df["DHI_cs"],

    np.nan
)


postcal_df["kDHI_cal_eval"] = np.where(

    valid_kdhi_postcal,

    postcal_df["DHI_cal"]
    /
    postcal_df["DHI_cs"],

    np.nan
)


# ============================================================
# 7. RESUMEN GLOBAL RAW VS CALIBRATED
# ============================================================

global_postcal_rows = []


for version, suffix in [

    (
        "RAW",
        "pred"
    ),

    (
        "CALIBRATED",
        "cal"
    ),

]:

    for variable in [
        "DNI",
        "DHI",
        "GHI",
    ]:

        m = regression_metrics(

            postcal_df[
                f"{variable}_obs"
            ],

            postcal_df[
                f"{variable}_{suffix}"
            ]
        )


        global_postcal_rows.append({

            "Version":
                version,

            "Variable":
                variable,

            **m
        })


global_postcal_df = pd.DataFrame(
    global_postcal_rows
)


print(
    "\n=== RESUMEN GLOBAL RAW VS CALIBRATED ==="
)

print(
    global_postcal_df.to_string(
        index=False
    )
)


# ============================================================
# 8. DIAGNÓSTICO ENERGÉTICO MENSUAL
# ============================================================

monthly_postcal_rows = []


for (
    year,
    month
), g in postcal_df.groupby(
    [
        "year",
        "month"
    ]
):

    for version, suffix in [

        (
            "RAW",
            "pred"
        ),

        (
            "CALIBRATED",
            "cal"
        ),

    ]:

        row = {

            "year":
                int(year),

            "month":
                int(month),

            "Version":
                version,

            "N":
                int(len(g)),
        }


        for variable in [
            "GHI",
            "DNI",
            "DHI",
        ]:

            e_obs = energy_wh_m2(
                g[
                    f"{variable}_obs"
                ]
            )

            e_model = energy_wh_m2(
                g[
                    f"{variable}_{suffix}"
                ]
            )


            row[
                f"{variable}_energy_bias_pct"
            ] = (

                100.0
                * (
                    e_model
                    - e_obs
                )
                / e_obs

                if e_obs > 0

                else np.nan
            )


        # ----------------------------------------------------
        # Razón energética difusa mensual
        # ----------------------------------------------------

        e_ghi_obs = energy_wh_m2(
            g["GHI_obs"]
        )

        e_dhi_obs = energy_wh_m2(
            g["DHI_obs"]
        )

        e_ghi_model = energy_wh_m2(
            g[
                f"GHI_{suffix}"
            ]
        )

        e_dhi_model = energy_wh_m2(
            g[
                f"DHI_{suffix}"
            ]
        )


        row[
            "diffuse_energy_ratio_obs"
        ] = (

            e_dhi_obs
            /
            e_ghi_obs

            if e_ghi_obs > 0

            else np.nan
        )


        row[
            "diffuse_energy_ratio_model"
        ] = (

            e_dhi_model
            /
            e_ghi_model

            if e_ghi_model > 0

            else np.nan
        )


        row[
            "diffuse_energy_ratio_bias"
        ] = (

            row[
                "diffuse_energy_ratio_model"
            ]

            -

            row[
                "diffuse_energy_ratio_obs"
            ]
        )


        monthly_postcal_rows.append(
            row
        )


monthly_postcal_df = pd.DataFrame(
    monthly_postcal_rows
)


# ============================================================
# 9. TABLA MENSUAL COMPARATIVA RAW / CAL
#
# Se pivota para facilitar interpretación.
# ============================================================

monthly_compare_df = (

    monthly_postcal_df

    .pivot(
        index=[
            "year",
            "month"
        ],

        columns="Version",

        values=[
            "GHI_energy_bias_pct",
            "DNI_energy_bias_pct",
            "DHI_energy_bias_pct",
            "diffuse_energy_ratio_bias",
        ]
    )
)


monthly_compare_df.columns = [

    f"{metric}_{version}"

    for metric, version
    in monthly_compare_df.columns
]


monthly_compare_df = (

    monthly_compare_df
    .reset_index()
)


# Cambios en magnitud absoluta del sesgo

for metric in [

    "DNI_energy_bias_pct",
    "DHI_energy_bias_pct",
    "diffuse_energy_ratio_bias",

]:

    monthly_compare_df[
        f"{metric}_abs_improvement"
    ] = (

        np.abs(
            monthly_compare_df[
                f"{metric}_RAW"
            ]
        )

        -

        np.abs(
            monthly_compare_df[
                f"{metric}_CALIBRATED"
            ]
        )
    )


print(
    "\n=== DIAGNÓSTICO MENSUAL RAW VS CALIBRATED ==="
)

print(
    monthly_compare_df[
        [
            "year",
            "month",

            "DNI_energy_bias_pct_RAW",
            "DNI_energy_bias_pct_CALIBRATED",

            "DHI_energy_bias_pct_RAW",
            "DHI_energy_bias_pct_CALIBRATED",

            "diffuse_energy_ratio_bias_RAW",
            "diffuse_energy_ratio_bias_CALIBRATED",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 10. RESUMEN DE MEJORA MENSUAL
# ============================================================

print(
    "\n=== MEJORA MENSUAL EN MAGNITUD DEL SESGO ==="
)


for metric in [

    "DNI_energy_bias_pct",
    "DHI_energy_bias_pct",
    "diffuse_energy_ratio_bias",

]:

    improvement = (

        monthly_compare_df[
            f"{metric}_abs_improvement"
        ]
    )


    print(
        f"\n{metric}:"
    )

    print(
        "  Meses mejorados:",
        int(
            (
                improvement > 0
            ).sum()
        ),
        "/",
        len(improvement)
    )

    print(
        "  Meses empeorados:",
        int(
            (
                improvement < 0
            ).sum()
        ),
        "/",
        len(improvement)
    )

    print(
        "  Mejora media absoluta:",
        float(
            improvement.mean()
        )
    )


# ============================================================
# 11. DIAGNÓSTICO POR COSZ_PLUS
# ============================================================

cosz_bins_postcal = [

    -0.001,
    0.05,
    0.20,
    0.40,
    0.60,
    0.80,
    1.001,
]


cosz_labels_postcal = [

    "night/very_low",
    "0.05-0.20",
    "0.20-0.40",
    "0.40-0.60",
    "0.60-0.80",
    "0.80-1.00",
]


postcal_df["cosz_bin"] = pd.cut(

    postcal_df[
        "cosz_plus"
    ],

    bins=cosz_bins_postcal,

    labels=cosz_labels_postcal,

    include_lowest=True,

    right=False
)


cosz_postcal_rows = []


for band, g in postcal_df.groupby(
    "cosz_bin",
    observed=True
):

    row = {

        "cosz_bin":
            str(band),

        "N":
            int(len(g)),
    }


    for variable in [
        "DNI",
        "DHI",
        "GHI",
    ]:

        # RAW

        m_raw = regression_metrics(

            g[
                f"{variable}_obs"
            ],

            g[
                f"{variable}_pred"
            ]
        )


        # CALIBRATED

        m_cal = regression_metrics(

            g[
                f"{variable}_obs"
            ],

            g[
                f"{variable}_cal"
            ]
        )


        row[
            f"{variable}_Bias_RAW"
        ] = m_raw["Bias"]

        row[
            f"{variable}_Bias_CAL"
        ] = m_cal["Bias"]


        row[
            f"{variable}_RMSE_RAW"
        ] = m_raw["RMSE"]

        row[
            f"{variable}_RMSE_CAL"
        ] = m_cal["RMSE"]


    # --------------------------------------------------------
    # Fracción difusa
    # --------------------------------------------------------

    valid_rd = (

        np.isfinite(
            g[
                "rd_obs_eval"
            ]
        )

        &

        np.isfinite(
            g[
                "rd_raw_eval"
            ]
        )

        &

        np.isfinite(
            g[
                "rd_cal_eval"
            ]
        )
    )


    if valid_rd.any():

        rd_obs_band = (

            g.loc[
                valid_rd,
                "rd_obs_eval"
            ]
            .to_numpy()
        )

        rd_raw_band = (

            g.loc[
                valid_rd,
                "rd_raw_eval"
            ]
            .to_numpy()
        )

        rd_cal_band = (

            g.loc[
                valid_rd,
                "rd_cal_eval"
            ]
            .to_numpy()
        )


        row[
            "rd_obs_mean"
        ] = float(
            np.mean(
                rd_obs_band
            )
        )


        row[
            "rd_raw_mean"
        ] = float(
            np.mean(
                rd_raw_band
            )
        )


        row[
            "rd_cal_mean"
        ] = float(
            np.mean(
                rd_cal_band
            )
        )


        row[
            "rd_bias_RAW"
        ] = float(
            np.mean(
                rd_raw_band
                -
                rd_obs_band
            )
        )


        row[
            "rd_bias_CAL"
        ] = float(
            np.mean(
                rd_cal_band
                -
                rd_obs_band
            )
        )


    else:

        row["rd_obs_mean"] = np.nan
        row["rd_raw_mean"] = np.nan
        row["rd_cal_mean"] = np.nan
        row["rd_bias_RAW"] = np.nan
        row["rd_bias_CAL"] = np.nan


    cosz_postcal_rows.append(
        row
    )


cosz_postcal_df = pd.DataFrame(
    cosz_postcal_rows
)


print(
    "\n=== DIAGNÓSTICO POST-CALIBRACIÓN POR COSZ_PLUS ==="
)

print(
    cosz_postcal_df.to_string(
        index=False
    )
)


# ============================================================
# 12. DIAGNÓSTICO POR FRACCIÓN DIFUSA OBSERVADA
#
# Este bloque es especialmente importante.
#
# Permitirá comprobar si persiste:
#
#   cielos directos   -> rd demasiado alto
#   cielos difusos    -> rd demasiado bajo
# ============================================================

rd_bins_postcal = [

    0.0,
    0.20,
    0.40,
    0.60,
    0.80,
    1.01,
]


rd_labels_postcal = [

    "0.00-0.20",
    "0.20-0.40",
    "0.40-0.60",
    "0.60-0.80",
    "0.80-1.00",
]


postcal_df["rd_bin"] = pd.cut(

    postcal_df[
        "rd_obs_eval"
    ],

    bins=rd_bins_postcal,

    labels=rd_labels_postcal,

    include_lowest=True,

    right=False
)


rd_postcal_rows = []


for band, g in postcal_df.groupby(
    "rd_bin",
    observed=True
):

    valid = (

        np.isfinite(
            g["rd_obs_eval"]
        )

        &

        np.isfinite(
            g["rd_raw_eval"]
        )

        &

        np.isfinite(
            g["rd_cal_eval"]
        )
    )


    if not valid.any():

        continue


    rd_obs_band = (

        g.loc[
            valid,
            "rd_obs_eval"
        ]
        .to_numpy()
    )

    rd_raw_band = (

        g.loc[
            valid,
            "rd_raw_eval"
        ]
        .to_numpy()
    )

    rd_cal_band = (

        g.loc[
            valid,
            "rd_cal_eval"
        ]
        .to_numpy()
    )


    row = {

        "rd_bin":
            str(band),

        "N":
            int(
                valid.sum()
            ),

        "rd_obs_mean":
            float(
                np.mean(
                    rd_obs_band
                )
            ),

        "rd_raw_mean":
            float(
                np.mean(
                    rd_raw_band
                )
            ),

        "rd_cal_mean":
            float(
                np.mean(
                    rd_cal_band
                )
            ),

        "rd_bias_RAW":
            float(
                np.mean(
                    rd_raw_band
                    -
                    rd_obs_band
                )
            ),

        "rd_bias_CAL":
            float(
                np.mean(
                    rd_cal_band
                    -
                    rd_obs_band
                )
            ),
    }


    # --------------------------------------------------------
    # DNI y DHI por régimen difuso
    # --------------------------------------------------------

    for variable in [
        "DNI",
        "DHI",
        "GHI",
    ]:

        m_raw = regression_metrics(

            g.loc[
                valid,
                f"{variable}_obs"
            ],

            g.loc[
                valid,
                f"{variable}_pred"
            ]
        )


        m_cal = regression_metrics(

            g.loc[
                valid,
                f"{variable}_obs"
            ],

            g.loc[
                valid,
                f"{variable}_cal"
            ]
        )


        row[
            f"{variable}_Bias_RAW"
        ] = m_raw["Bias"]

        row[
            f"{variable}_Bias_CAL"
        ] = m_cal["Bias"]


        row[
            f"{variable}_RMSE_RAW"
        ] = m_raw["RMSE"]

        row[
            f"{variable}_RMSE_CAL"
        ] = m_cal["RMSE"]


    rd_postcal_rows.append(
        row
    )


rd_postcal_df = pd.DataFrame(
    rd_postcal_rows
)


print(
    "\n=== DIAGNÓSTICO POR FRACCIÓN DIFUSA OBSERVADA ==="
)

print(
    rd_postcal_df.to_string(
        index=False
    )
)


# ============================================================
# 13. CAMBIO DE MAGNITUD DEL SESGO DE rd POR RÉGIMEN
#
# Valor positivo:
#     la calibración redujo |bias|
#
# Valor negativo:
#     la calibración empeoró |bias|
# ============================================================

rd_postcal_df[
    "rd_abs_bias_improvement"
] = (

    np.abs(
        rd_postcal_df[
            "rd_bias_RAW"
        ]
    )

    -

    np.abs(
        rd_postcal_df[
            "rd_bias_CAL"
        ]
    )
)


print(
    "\n=== MEJORA DEL SESGO rd POR RÉGIMEN ==="
)

print(
    rd_postcal_df[
        [
            "rd_bin",
            "N",
            "rd_bias_RAW",
            "rd_bias_CAL",
            "rd_abs_bias_improvement",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 14. DIAGNÓSTICO POR BANDAS DE kDHI OBSERVADO
# ============================================================

kdhi_bins_postcal = [

    0.0,
    1.0,
    1.4,
    2.0,
    3.0,
    4.0,
    6.0,
    8.01,
]


kdhi_labels_postcal = [

    "0-1",
    "1-1.4",
    "1.4-2",
    "2-3",
    "3-4",
    "4-6",
    "6-8",
]


postcal_df["kDHI_bin"] = pd.cut(

    postcal_df[
        "kDHI_obs_eval"
    ],

    bins=kdhi_bins_postcal,

    labels=kdhi_labels_postcal,

    include_lowest=True,

    right=False
)


kdhi_postcal_rows = []


for band, g in postcal_df.groupby(
    "kDHI_bin",
    observed=True
):

    valid = (

        np.isfinite(
            g[
                "kDHI_obs_eval"
            ]
        )

        &

        np.isfinite(
            g[
                "kDHI_raw_eval"
            ]
        )

        &

        np.isfinite(
            g[
                "kDHI_cal_eval"
            ]
        )
    )


    if not valid.any():

        continue


    k_obs = (

        g.loc[
            valid,
            "kDHI_obs_eval"
        ]
        .to_numpy()
    )

    k_raw = (

        g.loc[
            valid,
            "kDHI_raw_eval"
        ]
        .to_numpy()
    )

    k_cal = (

        g.loc[
            valid,
            "kDHI_cal_eval"
        ]
        .to_numpy()
    )


    kdhi_postcal_rows.append({

        "kDHI_bin":
            str(band),

        "N":
            int(
                valid.sum()
            ),

        "kDHI_obs_mean":
            float(
                np.mean(
                    k_obs
                )
            ),

        "kDHI_raw_mean":
            float(
                np.mean(
                    k_raw
                )
            ),

        "kDHI_cal_mean":
            float(
                np.mean(
                    k_cal
                )
            ),

        "kDHI_bias_RAW":
            float(
                np.mean(
                    k_raw
                    -
                    k_obs
                )
            ),

        "kDHI_bias_CAL":
            float(
                np.mean(
                    k_cal
                    -
                    k_obs
                )
            ),

        "kDHI_RMSE_RAW":
            float(
                np.sqrt(
                    np.mean(
                        (
                            k_raw
                            -
                            k_obs
                        ) ** 2
                    )
                )
            ),

        "kDHI_RMSE_CAL":
            float(
                np.sqrt(
                    np.mean(
                        (
                            k_cal
                            -
                            k_obs
                        ) ** 2
                    )
                )
            ),
    })


kdhi_postcal_df = pd.DataFrame(
    kdhi_postcal_rows
)


kdhi_postcal_df[
    "kDHI_abs_bias_improvement"
] = (

    np.abs(
        kdhi_postcal_df[
            "kDHI_bias_RAW"
        ]
    )

    -

    np.abs(
        kdhi_postcal_df[
            "kDHI_bias_CAL"
        ]
    )
)


print(
    "\n=== DIAGNÓSTICO POST-CALIBRACIÓN POR kDHI ==="
)

print(
    kdhi_postcal_df.to_string(
        index=False
    )
)


# ============================================================
# 15. RESUMEN GLOBAL DE PARTICIÓN
# ============================================================

global_ratio_mask = (

    np.isfinite(
        postcal_df[
            "rd_obs_eval"
        ]
    )

    &

    np.isfinite(
        postcal_df[
            "rd_raw_eval"
        ]
    )

    &

    np.isfinite(
        postcal_df[
            "rd_cal_eval"
        ]
    )
)


rd_obs_global = (

    postcal_df.loc[
        global_ratio_mask,
        "rd_obs_eval"
    ]
    .to_numpy()
)


rd_raw_global = (

    postcal_df.loc[
        global_ratio_mask,
        "rd_raw_eval"
    ]
    .to_numpy()
)


rd_cal_global = (

    postcal_df.loc[
        global_ratio_mask,
        "rd_cal_eval"
    ]
    .to_numpy()
)


raw_ratio_metrics_post = (
    regression_metrics(
        rd_obs_global,
        rd_raw_global
    )
)


cal_ratio_metrics_post = (
    regression_metrics(
        rd_obs_global,
        rd_cal_global
    )
)


print(
    "\n=== RESUMEN GLOBAL DE PARTICIÓN ==="
)

print(
    "OBS mean:",
    float(
        np.mean(
            rd_obs_global
        )
    )
)

print(
    "\nRAW"
)

print(
    "mean:",
    float(
        np.mean(
            rd_raw_global
        )
    )
)

print(
    "RMSE:",
    raw_ratio_metrics_post[
        "RMSE"
    ]
)

print(
    "MAE:",
    raw_ratio_metrics_post[
        "MAE"
    ]
)

print(
    "Bias:",
    raw_ratio_metrics_post[
        "Bias"
    ]
)


print(
    "\nCALIBRATED"
)

print(
    "mean:",
    float(
        np.mean(
            rd_cal_global
        )
    )
)

print(
    "RMSE:",
    cal_ratio_metrics_post[
        "RMSE"
    ]
)

print(
    "MAE:",
    cal_ratio_metrics_post[
        "MAE"
    ]
)

print(
    "Bias:",
    cal_ratio_metrics_post[
        "Bias"
    ]
)


# ============================================================
# 16. RESUMEN AUTOMÁTICO DE REGÍMENES
# ============================================================

print(
    "\n=== RESUMEN AUTOMÁTICO DE REGÍMENES ==="
)


for _, row in rd_postcal_df.iterrows():

    raw_abs = abs(
        row[
            "rd_bias_RAW"
        ]
    )

    cal_abs = abs(
        row[
            "rd_bias_CAL"
        ]
    )


    if cal_abs < raw_abs:

        status = "MEJORA"

    elif cal_abs > raw_abs:

        status = "EMPEORA"

    else:

        status = "SIN CAMBIO"


    print(

        f"{row['rd_bin']:>10} | "
        f"raw={row['rd_bias_RAW']:+.4f} | "
        f"cal={row['rd_bias_CAL']:+.4f} | "
        f"{status}"
    )


# ============================================================
# 17. CIERRE FÍSICO POST-CALIBRACIÓN
# ============================================================

closure_postcal = (

    postcal_df[
        "GHI_cal"
    ].to_numpy()

    -

    (
        postcal_df[
            "DNI_cal"
        ].to_numpy()

        *

        postcal_df[
            "cosz_plus"
        ].to_numpy()

        +

        postcal_df[
            "DHI_cal"
        ].to_numpy()
    )
)


closure_postcal_mae = float(

    np.mean(
        np.abs(
            closure_postcal
        )
    )
)


closure_postcal_max = float(

    np.max(
        np.abs(
            closure_postcal
        )
    )
)


print(
    "\n=== CIERRE FÍSICO POST-CALIBRACIÓN ==="
)

print(
    "Closure MAE:",
    closure_postcal_mae
)

print(
    "Closure max abs:",
    closure_postcal_max
)


# ============================================================
# 18. VALIDACIONES FINALES
# ============================================================

assert len(
    postcal_df
) == len(
    test_cal_df
)


assert (
    postcal_df[
        "timestamp"
    ]
    .duplicated()
    .sum()
    == 0
)


assert np.all(
    np.isfinite(
        postcal_df[
            [
                "DNI_pred",
                "DNI_cal",
                "DNI_obs",

                "DHI_pred",
                "DHI_cal",
                "DHI_obs",

                "GHI_pred",
                "GHI_cal",
                "GHI_obs",

                "cosz_plus",
            ]
        ].to_numpy()
    )
)


# GHI debe ser exactamente el mismo antes y después
# de la calibración.

max_ghi_change = float(

    np.max(
        np.abs(
            postcal_df[
                "GHI_cal"
            ].to_numpy()

            -

            postcal_df[
                "GHI_pred"
            ].to_numpy()
        )
    )
)


print(
    "\nMáximo cambio GHI raw → calibrated:",
    max_ghi_change
)


assert max_ghi_change < 1e-10


assert closure_postcal_mae < 1e-3


# ============================================================
# 19. RESUMEN FINAL
# ============================================================

print(
    "\n✓ Diagnóstico post-calibración completado."
)

print(
    "✓ Comparación mensual RAW vs CALIBRATED completada."
)

print(
    "✓ Dependencia con cosz_plus evaluada."
)

print(
    "✓ Regímenes de fracción difusa evaluados."
)

print(
    "✓ Regímenes de kDHI evaluados."
)

print(
    "✓ GHI permaneció inalterado."
)

print(
    "✓ Cierre radiativo preservado."
)

print(
    "✓ Ningún parámetro fue ajustado con TEST."
)

print(
    "✓ Celda 16 completada correctamente."
)

=== DIAGNÓSTICO POST-CALIBRACIÓN ===
N: 35,016
Inicio: 2023-01-02 12:00:00
Fin: 2024-12-31 23:30:00

=== RESUMEN GLOBAL RAW VS CALIBRATED ===
   Version Variable     N       RMSE        MAE       Bias       R2
       RAW      DNI 35016 197.492106 106.317995 -48.565601 0.537563
       RAW      DHI 35016  59.719471  31.376998   9.405414 0.808776
       RAW      GHI 35016 112.915247  59.020368 -25.026110 0.870615
CALIBRATED      DNI 35016 192.553817 105.647425 -40.208633 0.560401
CALIBRATED      DHI 35016  58.580340  31.567746   4.277029 0.816001
CALIBRATED      GHI 35016 112.915247  59.020368 -25.026110 0.870615

=== DIAGNÓSTICO MENSUAL RAW VS CALIBRATED ===
 year  month  DNI_energy_bias_pct_RAW  DNI_energy_bias_pct_CALIBRATED  DHI_energy_bias_pct_RAW  DHI_energy_bias_pct_CALIBRATED  diffuse_energy_ratio_bias_RAW  diffuse_energy_ratio_bias_CALIBRATED
 2023      1               -25.821514                      -20.168207                10.714584                        4.777081             

## Evaluación independiente 2023–2024

La evaluación debe reportar por separado:

- exactitud predictiva de GHI, DNI y DHI;
- sesgo;
- consistencia física;
- partición difusa.

Un cierre físico pequeño no se interpreta como prueba de precisión de DNI/DHI, porque GHI se reconstruye estructuralmente.

In [ ]:
# Arrays de test
Xhist_test_all = transform_hist(test_df)
Xfut_test_all = transform_fut(test_df)
Yext_test_all = test_df[Y_EXT_COLS].to_numpy(dtype=np.float32)

def collect_test_predictions():
    preds, trues, futs = [], [], []
    n = len(test_df)
    for i in range(0, n - cfg.L - cfg.H + 1, cfg.H):
        j = i + cfg.L
        xh = Xhist_test_all[i:j][None, ...]
        xf = Xfut_test_all[j:j+cfg.H][None, ...]
        yp = trainer.backbone.predict({"x_hist": xh, "x_fut": xf}, verbose=0)[0]
        preds.append(yp)
        trues.append(Yext_test_all[j:j+cfg.H])
        futs.append(xf[0])
    return np.concatenate(preds), np.concatenate(trues), np.concatenate(futs)

pred_k, true_ext, fut_test = collect_test_predictions()

idx_cosz_scaled = FUT_COLS_SCALED.index("cosz_plus")
cosz = (
    fut_test[:, FUT_COLS.index("cosz_plus")] * scaler_fut.scale_[idx_cosz_scaled]
    + scaler_fut.mean_[idx_cosz_scaled]
)
cosz = np.clip(cosz, 0, 1)

csdni = fut_test[:, FUT_COLS.index(COL_CSDNI)]
csdhi = fut_test[:, FUT_COLS.index(COL_CSDHI)]

dni_pred = pred_k[:, 0] * csdni
dhi_pred = pred_k[:, 1] * csdhi
ghi_pred = dni_pred * cosz + dhi_pred

dni_true = true_ext[:, Y_EXT_COLS.index(COL_DNI)]
dhi_true = true_ext[:, Y_EXT_COLS.index(COL_DHI)]
ghi_true = true_ext[:, Y_EXT_COLS.index(COL_GHI)]

def metric_row(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = float(np.mean(y_pred - y_true))
    r2 = r2_score(y_true, y_pred)
    mean_true = float(np.mean(y_true))
    nrmse = rmse / mean_true if mean_true > 1e-9 else np.nan
    return {
        "variable": name,
        "RMSE": rmse,
        "MAE": mae,
        "Bias": bias,
        "R2": r2,
        "nRMSE": nrmse
    }

metrics = pd.DataFrame([
    metric_row("DNI", dni_true, dni_pred),
    metric_row("DHI", dhi_true, dhi_pred),
    metric_row("GHI", ghi_true, ghi_pred),
])

ratio_true = dhi_true / np.maximum(ghi_true, 1e-6)
ratio_pred = dhi_pred / np.maximum(ghi_pred, 1e-6)
day = (ghi_true > cfg.thr_cs) & (cosz > cfg.thr_day_cosz)

partition_summary = {
    "diffuse_ratio_true_day_mean": float(np.mean(ratio_true[day])),
    "diffuse_ratio_pred_day_mean": float(np.mean(ratio_pred[day])),
    "diffuse_ratio_bias": float(np.mean(ratio_pred[day]) - np.mean(ratio_true[day])),
}
print(metrics)
print(partition_summary)

metrics.to_csv(cfg.metrics_path, index=False)

with open(cfg.metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)
metadata["partition_test_summary"] = partition_summary
with open(cfg.metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

  variable        RMSE         MAE       Bias        R2     nRMSE
0      DNI  201.026778  107.556152  14.484655  0.520862  1.103013
1      DHI   87.481443   43.140759 -32.820946  0.589661  0.879634
2      GHI  134.104186   64.888343 -22.524658  0.817499  0.584357
{'diffuse_ratio_true_day_mean': 0.5652092695236206, 'diffuse_ratio_pred_day_mean': 0.4474118103082156, 'diffuse_ratio_bias': -0.11779745921540502}


In [ ]:
# ============================================================
# AUDITORÍA FINAL DEL METADATA
# ============================================================

with open(
    cfg.metadata_path,
    "r",
    encoding="utf-8"
) as f:

    meta_check = json.load(f)


required_metadata_keys = [

    # Modelo
    "model_path",
    "seed",

    # Secuencias
    "L",
    "H",
    "stride",

    # Entrenamiento
    "batch_size",
    "epochs_max",
    "epochs_ran",
    "initial_learning_rate",
    "patience_early_stopping",
    "patience_reduce_lr",
    "reduce_lr_factor",
    "min_learning_rate",
    "optimizer",
    "best_epoch",
    "best_val_loss",

    # Arquitectura
    "latent_enc",
    "latent_dec",
    "dropout",

    # Restricciones
    "kdni_max",
    "kdhi_max",

    # Features
    "HIST_COLS",
    "FUT_COLS",

    # Loss
    "loss_weights",

    # Evaluación
    "partition_test_summary",

    # Entorno
    "tensorflow_version",
]


missing_meta = [
    k
    for k in required_metadata_keys
    if k not in meta_check
]

if missing_meta:
    raise ValueError(
        "Faltan campos en model_metadata_v23.json: "
        + str(missing_meta)
    )


# ------------------------------------------------------------
# Contratos críticos
# ------------------------------------------------------------

assert meta_check["L"] == 72
assert meta_check["H"] == 24
assert meta_check["stride"] == 2

assert meta_check["batch_size"] == 32
assert meta_check["epochs_max"] == 25

assert np.isclose(
    meta_check["initial_learning_rate"],
    3e-4
)

assert meta_check[
    "patience_early_stopping"
] == 8

assert meta_check[
    "patience_reduce_lr"
] == 3

assert np.isclose(
    meta_check["reduce_lr_factor"],
    0.5
)

assert np.isclose(
    meta_check["min_learning_rate"],
    1e-5
)

assert meta_check["optimizer"] == "Adam"

assert len(
    meta_check["HIST_COLS"]
) == 34

assert len(
    meta_check["FUT_COLS"]
) == 17

assert np.isclose(
    meta_check["kdni_max"],
    1.20
)

assert np.isclose(
    meta_check["kdhi_max"],
    1.40
)


print("✓ Metadata completo y consistente")
print(
    "✓ Mejor época:",
    meta_check["best_epoch"]
)
print(
    "✓ Mejor val_loss:",
    meta_check["best_val_loss"]
)
print(
    "✓ Épocas ejecutadas:",
    meta_check["epochs_ran"]
)

## Contrato con el Notebook 3

El Notebook 3 debe rechazar la ejecución si:

- el `.keras` no espera `(72,34)` y `(24,17)`;
- las listas `HIST_COLS/FUT_COLS` no coinciden con `model_metadata_v23.json`;
- faltan los scalers exportados aquí;
- se intenta aplicar un clipping de $k_{DHI}$ diferente a 1.40 sin declararlo explícitamente como postproceso.

In [ ]:
# ============================================================
# DIAGNÓSTICOS AMPLIADOS DE TEST 2023–2024
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

EPS = 1e-6

# ------------------------------------------------------------
# 1. Máscaras de evaluación
# ------------------------------------------------------------

mask_all = np.ones_like(ghi_true, dtype=bool)

mask_day = (
    (cosz > cfg.thr_day_cosz)
    & (ghi_true > cfg.thr_cs)
)

print(f"N total      : {mask_all.sum():,}")
print(f"N daytime    : {mask_day.sum():,}")
print(f"% daytime    : {100 * mask_day.mean():.2f}%")

# ------------------------------------------------------------
# 2. Función general de métricas
# ------------------------------------------------------------

def compute_metrics(y_true, y_pred, mask, variable, subset):
    yt = np.asarray(y_true)[mask]
    yp = np.asarray(y_pred)[mask]

    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = mean_absolute_error(yt, yp)
    bias = float(np.mean(yp - yt))
    r2 = r2_score(yt, yp)

    mean_true = float(np.mean(yt))
    nrmse_mean = rmse / mean_true if abs(mean_true) > EPS else np.nan

    # Normalización adicional por rango observado
    data_range = float(np.max(yt) - np.min(yt))
    nrmse_range = rmse / data_range if data_range > EPS else np.nan

    return {
        "subset": subset,
        "variable": variable,
        "N": int(mask.sum()),
        "Mean_true": mean_true,
        "RMSE": rmse,
        "MAE": mae,
        "Bias": bias,
        "R2": r2,
        "nRMSE_mean": nrmse_mean,
        "nRMSE_range": nrmse_range,
    }

rows = []

for subset_name, mask in [
    ("all_time", mask_all),
    ("daytime", mask_day),
]:
    rows.extend([
        compute_metrics(dni_true, dni_pred, mask, "DNI", subset_name),
        compute_metrics(dhi_true, dhi_pred, mask, "DHI", subset_name),
        compute_metrics(ghi_true, ghi_pred, mask, "GHI", subset_name),
    ])

metrics_extended = pd.DataFrame(rows)

print("\n=== MÉTRICAS AMPLIADAS ===")
display(metrics_extended.round(4))

# ------------------------------------------------------------
# 3. Partición radiativa
# ------------------------------------------------------------

ratio_diffuse_true = np.full_like(ghi_true, np.nan, dtype=float)
ratio_diffuse_pred = np.full_like(ghi_pred, np.nan, dtype=float)

ratio_beam_true = np.full_like(ghi_true, np.nan, dtype=float)
ratio_beam_pred = np.full_like(ghi_pred, np.nan, dtype=float)

valid_ratio = mask_day.copy()

ratio_diffuse_true[valid_ratio] = (
    dhi_true[valid_ratio]
    / np.maximum(ghi_true[valid_ratio], EPS)
)

ratio_diffuse_pred[valid_ratio] = (
    dhi_pred[valid_ratio]
    / np.maximum(ghi_pred[valid_ratio], EPS)
)

ratio_beam_true[valid_ratio] = (
    dni_true[valid_ratio] * cosz[valid_ratio]
    / np.maximum(ghi_true[valid_ratio], EPS)
)

ratio_beam_pred[valid_ratio] = (
    dni_pred[valid_ratio] * cosz[valid_ratio]
    / np.maximum(ghi_pred[valid_ratio], EPS)
)

partition_metrics = pd.DataFrame({
    "metric": [
        "Diffuse ratio true mean",
        "Diffuse ratio pred mean",
        "Diffuse ratio bias",
        "Diffuse ratio MAE",
        "Beam ratio true mean",
        "Beam ratio pred mean",
        "Beam ratio bias",
        "Beam ratio MAE",
    ],
    "value": [
        np.nanmean(ratio_diffuse_true),
        np.nanmean(ratio_diffuse_pred),
        np.nanmean(ratio_diffuse_pred - ratio_diffuse_true),
        np.nanmean(np.abs(ratio_diffuse_pred - ratio_diffuse_true)),
        np.nanmean(ratio_beam_true),
        np.nanmean(ratio_beam_pred),
        np.nanmean(ratio_beam_pred - ratio_beam_true),
        np.nanmean(np.abs(ratio_beam_pred - ratio_beam_true)),
    ]
})

print("\n=== PARTICIÓN RADIATIVA ===")
display(partition_metrics.round(4))

# ------------------------------------------------------------
# 4. Consistencia física / closure
# ------------------------------------------------------------

closure_pred = ghi_pred - (dni_pred * cosz + dhi_pred)

closure_metrics = {
    "Closure mean [W/m²]": float(np.mean(closure_pred)),
    "Closure MAE [W/m²]": float(np.mean(np.abs(closure_pred))),
    "Closure RMSE [W/m²]": float(np.sqrt(np.mean(closure_pred**2))),
    "Closure max abs [W/m²]": float(np.max(np.abs(closure_pred))),
}

print("\n=== CIERRE FÍSICO ===")
for k, v in closure_metrics.items():
    print(f"{k}: {v:.8f}")

# ------------------------------------------------------------
# 5. Energía del período de test
# ------------------------------------------------------------

dt_hours = 0.5

energy_true = {
    "GHI": float(np.sum(ghi_true) * dt_hours),
    "DNI": float(np.sum(dni_true) * dt_hours),
    "DHI": float(np.sum(dhi_true) * dt_hours),
}

energy_pred = {
    "GHI": float(np.sum(ghi_pred) * dt_hours),
    "DNI": float(np.sum(dni_pred) * dt_hours),
    "DHI": float(np.sum(dhi_pred) * dt_hours),
}

energy_rows = []

for var in ["GHI", "DNI", "DHI"]:
    true_e = energy_true[var]
    pred_e = energy_pred[var]

    bias_abs = pred_e - true_e
    bias_pct = 100 * bias_abs / true_e if abs(true_e) > EPS else np.nan

    energy_rows.append({
        "variable": var,
        "Energy_true": true_e,
        "Energy_pred": pred_e,
        "Bias_energy": bias_abs,
        "Bias_energy_pct": bias_pct,
    })

energy_metrics = pd.DataFrame(energy_rows)

print("\n=== ENERGÍA TEST 2023–2024 ===")
display(energy_metrics.round(4))

# ------------------------------------------------------------
# 6. Métricas por horizonte: 30 a 720 min
# ------------------------------------------------------------

# pred_k, true_ext y fut_test fueron concatenados bloque a bloque
# Cada grupo de H=24 registros representa horizontes 1...24.
n_total = len(ghi_true)

horizon_index = np.tile(
    np.arange(1, cfg.H + 1),
    int(np.ceil(n_total / cfg.H))
)[:n_total]

horizon_rows = []

for h in range(1, cfg.H + 1):

    mh = horizon_index == h

    # Daytime específico del horizonte
    mh_day = mh & mask_day

    horizon_minutes = h * 30

    for variable, yt, yp in [
        ("GHI", ghi_true, ghi_pred),
        ("DNI", dni_true, dni_pred),
        ("DHI", dhi_true, dhi_pred),
    ]:

        # All-time
        if mh.sum() > 0:
            m = compute_metrics(
                yt, yp, mh,
                variable=variable,
                subset="all_time"
            )
            m["horizon_step"] = h
            m["horizon_minutes"] = horizon_minutes
            horizon_rows.append(m)

        # Daytime
        if mh_day.sum() > 0:
            m = compute_metrics(
                yt, yp, mh_day,
                variable=variable,
                subset="daytime"
            )
            m["horizon_step"] = h
            m["horizon_minutes"] = horizon_minutes
            horizon_rows.append(m)

horizon_metrics = pd.DataFrame(horizon_rows)

print("\n=== MÉTRICAS POR HORIZONTE ===")
display(
    horizon_metrics[
        [
            "subset",
            "variable",
            "horizon_minutes",
            "N",
            "RMSE",
            "MAE",
            "Bias",
            "R2",
            "nRMSE_mean",
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 7. Resumen especial para discusión
# ------------------------------------------------------------

discussion_summary = {
    "DNI_bias_day_Wm2": float(np.mean(dni_pred[mask_day] - dni_true[mask_day])),
    "DHI_bias_day_Wm2": float(np.mean(dhi_pred[mask_day] - dhi_true[mask_day])),
    "GHI_bias_day_Wm2": float(np.mean(ghi_pred[mask_day] - ghi_true[mask_day])),

    "diffuse_ratio_true": float(np.nanmean(ratio_diffuse_true)),
    "diffuse_ratio_pred": float(np.nanmean(ratio_diffuse_pred)),
    "diffuse_ratio_bias": float(
        np.nanmean(ratio_diffuse_pred - ratio_diffuse_true)
    ),

    "beam_ratio_true": float(np.nanmean(ratio_beam_true)),
    "beam_ratio_pred": float(np.nanmean(ratio_beam_pred)),
    "beam_ratio_bias": float(
        np.nanmean(ratio_beam_pred - ratio_beam_true)
    ),

    "GHI_energy_bias_pct": float(
        energy_metrics.loc[
            energy_metrics["variable"] == "GHI",
            "Bias_energy_pct"
        ].iloc[0]
    ),
    "DNI_energy_bias_pct": float(
        energy_metrics.loc[
            energy_metrics["variable"] == "DNI",
            "Bias_energy_pct"
        ].iloc[0]
    ),
    "DHI_energy_bias_pct": float(
        energy_metrics.loc[
            energy_metrics["variable"] == "DHI",
            "Bias_energy_pct"
        ].iloc[0]
    ),
}

print("\n=== RESUMEN PARA DISCUSIÓN ===")
for k, v in discussion_summary.items():
    print(f"{k}: {v:.6f}")

# ------------------------------------------------------------
# 8. Exportación
# ------------------------------------------------------------

metrics_extended.to_csv(
    "test_metrics_v23_extended.csv",
    index=False
)

partition_metrics.to_csv(
    "test_partition_metrics_v23.csv",
    index=False
)

energy_metrics.to_csv(
    "test_energy_metrics_v23.csv",
    index=False
)

horizon_metrics.to_csv(
    "test_horizon_metrics_v23.csv",
    index=False
)

pd.DataFrame(
    [discussion_summary]
).to_csv(
    "test_discussion_summary_v23.csv",
    index=False
)

print("\nArchivos exportados:")
print(" - test_metrics_v23_extended.csv")
print(" - test_partition_metrics_v23.csv")
print(" - test_energy_metrics_v23.csv")
print(" - test_horizon_metrics_v23.csv")
print(" - test_discussion_summary_v23.csv")

N total      : 35,016
N daytime    : 16,754
% daytime    : 47.85%

=== MÉTRICAS AMPLIADAS ===


,subset,variable,N,Mean_true,RMSE,MAE,Bias,R2,nRMSE_mean,nRMSE_range
0,all_time,DNI,35016,182.2524,201.0268,107.5562,14.4847,0.5209,1.1030,0.1965
1,all_time,DHI,35016,99.4521,87.4814,43.1408,-32.8209,0.5897,0.8796,0.1475
2,all_time,GHI,35016,229.4903,134.1042,64.8883,-22.5247,0.8175,0.5844,0.1236
3,daytime,DNI,16754,379.9215,290.3227,223.1869,30.4982,0.1678,0.7642,0.2838
4,daytime,DHI,16754,207.2592,126.3754,89.4503,-68.6302,0.0429,0.6097,0.2186
5,daytime,GHI,16754,479.0117,193.7702,134.7836,-47.1877,0.5663,0.4045,0.1821



=== PARTICIÓN RADIATIVA ===


,metric,value
0,Diffuse ratio true mean,0.5652
1,Diffuse ratio pred mean,0.4474
2,Diffuse ratio bias,-0.1178
3,Diffuse ratio MAE,0.2337
4,Beam ratio true mean,0.4349
5,Beam ratio pred mean,0.5526
6,Beam ratio bias,0.1177
7,Beam ratio MAE,0.2336



=== CIERRE FÍSICO ===
Closure mean [W/m²]: 0.00000000
Closure MAE [W/m²]: 0.00000000
Closure RMSE [W/m²]: 0.00000000
Closure max abs [W/m²]: 0.00000000

=== ENERGÍA TEST 2023–2024 ===


,variable,Energy_true,Energy_pred,Bias_energy,Bias_energy_pct
0,GHI,4017916.5,3.623555e+06,-394361.7187,-9.8151
1,DNI,3190875.5,3.444473e+06,253597.5000,7.9476
2,DHI,1741208.0,1.166579e+06,-574629.1250,-33.0018



=== MÉTRICAS POR HORIZONTE ===


,subset,variable,horizon_minutes,N,RMSE,MAE,Bias,R2,nRMSE_mean
0,all_time,GHI,30,1459,211.9361,116.3312,-105.6194,0.7412,0.5524
1,daytime,GHI,30,730,299.6202,232.5030,-211.0940,-0.6890,0.3908
2,all_time,DNI,30,1459,157.3663,78.2327,-35.4569,0.7828,0.6588
3,daytime,DNI,30,730,222.4732,156.3582,-70.8652,0.5659,0.4660
4,all_time,DHI,30,1459,136.9306,78.2276,-71.5603,0.4226,0.8859
...,...,...,...,...,...,...,...,...,...
139,daytime,GHI,720,729,270.5855,214.8724,-107.1220,-0.4691,0.3588
140,all_time,DNI,720,1459,239.7983,140.8627,15.8132,0.4933,1.0105
141,daytime,DNI,720,729,339.2423,281.9186,31.6481,-0.0073,0.7143
142,all_time,DHI,720,1459,136.2496,79.9063,-67.8725,0.4224,0.8914



=== RESUMEN PARA DISCUSIÓN ===
DNI_bias_day_Wm2: 30.498247
DHI_bias_day_Wm2: -68.630157
GHI_bias_day_Wm2: -47.187741
diffuse_ratio_true: 0.565209
diffuse_ratio_pred: 0.447412
diffuse_ratio_bias: -0.117797
beam_ratio_true: 0.434888
beam_ratio_pred: 0.552588
beam_ratio_bias: 0.117700
GHI_energy_bias_pct: -9.815080
DNI_energy_bias_pct: 7.947584
DHI_energy_bias_pct: -33.001751

Archivos exportados:
 - test_metrics_v23_extended.csv
 - test_partition_metrics_v23.csv
 - test_energy_metrics_v23.csv
 - test_horizon_metrics_v23.csv
 - test_discussion_summary_v23.csv
